# UMSF Firmware Laboratory
### A self-contained, executable companion to the ESP32/BLE source release
**Notebook release 1.2.0 · Original firmware 1.0.0 · English · CPU only**

Explore the original portable C/C++ implementation through controlled experiments, protocol inspection and reproducible figures. All firmware source, test vectors and the new experiment harness are embedded in this notebook. No external dataset, account, device, broker, credentials or Google Drive mount is needed.

**Start:** Upload this `.ipynb` to [Google Colab](https://colab.research.google.com/), select a Python 3 CPU runtime, then **Runtime → Run all**. A Linux C/C++ compiler and `make` are required; these normally exist in Colab. Local use: Python 3.10+ with GCC/G++, make, NumPy, pandas, Matplotlib and IPython.

**Self-sufficient cells.** Section 0 writes a small verified runtime (`umsf_fw_lab.py`) next to the notebook. After it has run once, *every* analysis cell can be executed on its own, in any order, and again after **Runtime → Restart session**: each cell begins with `from umsf_fw_lab import *`, and the native builds, experiment CSVs and checks it needs are produced on first use and cached. Builds are incremental, so unchanged binaries are reused.

**Evidence boundary.** Native execution verifies software behavior under synthetic inputs. It does not compile the ESP32 images or validate BLE radio performance, MQTT/mTLS, sensors, real physiology, structural safety, or gas concentrations. Heart rate is null by default; a separate synthetic mode is explicitly flagged. SpO₂ stays null. All local thresholds are uncalibrated. `possible_impact_local` is an unverified composite, never attribution of a strike.

**Learning route:** reproducibility → native tests → binary protocol → BLE transport → personnel features → stationary events → button logic → link freshness → buffering and ACKs → export. Read each explanation before changing parameters. Every chart is also exported as PNG and SVG. The original source (including its localized embedded web pages and README) is archived verbatim; the notebook, laboratory interface and new documentation are English.

## 0. Setup — run once per runtime
Three cells prepare everything; nothing is downloaded except missing Python libraries.

1. **Embedded project** (collapsed): writes `umsf_fw_payload.py` – a 88,712-byte xz-compressed tar (Base85) with all original firmware files, the experiment harness and the source inventory; SHA-256 `35ba8d748ba4bb79…`.
2. **Runtime module** (collapsed): writes `umsf_fw_lab.py` – dependency and compiler check, payload/hash verification, plotting conventions and cached native states. Expand it to read the code.
3. **Start**: imports the runtime. It verifies the payload and every original source hash and extracts the project once to `umsf_firmware_runs/project/` (later imports reuse the verified copy).

If a cell later reports `No module named 'umsf_fw_lab'`, the runtime files were removed (for example after **Disconnect and delete runtime**): run these three cells again.

In [ ]:
%%writefile umsf_fw_payload.py
# Embedded UMSF firmware project: xz-compressed tar, Base85 text. Generated by scripts/build_notebook.py.
PAYLOAD_SHA256 = '35ba8d748ba4bb799be072343cb945a67237e26f3c15f9bb6a49fc25f4a8e858'
PAYLOAD = """
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<IAD%|6Kqx8cM5<ch~E3e^rr7qf7nQ45P5`lcB`ZTk}{A44G0+?!KX8L>*gXdKzx(#h1h?ga&|7M5p?9j6O+Cd&<ZO
+c*Za(F2W>^eb3yp?|uZGN$sW@UpPH4!fNYWaIced)0ZRj>DaJD!T-
uCTPHRONJ)}h_ZK{$4!obyDKUu0*SHc7R`7tj%}20qcj4cAi=V!_5ssts4T{N9Z}o!2v2fAfri@{Wy=Fon>7X*Oco}y@^uuPfGmdWy=~CqlPS5rL>6Jk0`(l
9_V9HEic;&DEF~+kCavS`a*j}Ez|2lAU*j9F;4!QhSzPL*PY$Q5s|Uz4?UFPc%PT$BY{LF`RhQnb6RJXTWDHrBIOO9^ejtg~2`p$Je}Hw-OTXtMC=Rd~Y#k;
{f?!W*3S_2oV9I_uDCIQ~<nllfpNX{tbi(^d65^Jb!YS`wzA;@S92$F<LQ)W8n|>W*Br<(=6@P+@FYO8c5E0s|)L9&~Ph+O4VC+g3qG{4$dUQ!yUyca9g?-
4PbPUV-@Ayy+Xf_RndF9)};5QBl(SkP*nmLHxe-ZFg9oTr{q=6ggD^HtHD+Q+D{~)sv=_+Wc>XMX{kSI)Pe8x6Sas}0dFbCT2cqYrMQcvIZVP<(e4wD1Jh6N
K^#%cvo^$0Lci*K|C1HqlOyYywZUpc7JtG`fPi5Tm@mm*>AfJZxsE$~*sx>N6xR49^FfN1^Ck6E*?!Hos0#D>DqyzcSF{;oT%ME-
IO$U(9W^WR57sSk5rH6^xoGsGs_i;h41O?@^sM?<P4(GLTgZTJ`ZYn?L3m8p8HK_yd?w4X$%`djHB4<DX*7K(Nay-gPQD3KkrnF|@hCmt}kFVCcwB#UEk-
z2B@CJHSRfQ1Wstq(0T$6KA`YHM>}g%bi&0y}C&Z__UniU)E1DjNg(22z`lF8>so<`3&%yvH0x(*R?U=?CFU3nHu<DIDc7#yu%;#gyYW(;sfrlXi)GqM@|s$
GDB-
V8@ZQ2)z^9`nmp1M0mFh+;l%s62FVBf4lYN&36}DNa|`sSSNuNk^W35p*n|1<7JtlWdZMCnR`w%KmpnS!4|H6qJ&>1whkb}>~Kii&?N43Pv3;c4oSLatvFak
%%T82ywD*AV$!{-
OqDx_JX~!irn(h>L;Q<|y3kwYu(I+5*R>|bMa~vPo5+ol2x_d8_i=zkqwE8{G+T($JJg_Sn3R|PIO^8E!^WrJiACw0EoND4+t5R4!4KWcKF_1t7kD$}*d-
QpIXY^rJ$x1R)JIiw^gH5e<emovZk}AStZW}alJqR;{a}V{#mXY)pVN95kn&8Hs~x9;5VPgMr8q`!3#y*RaKGY5CaWPxrcbj?*u3G#R_*r=r%%MObj(M=n$_
nX0qD8oV@sN3<*9pdd}?;4!RhDD6>}P2tgbyLUb0aIHa%vm7V^FCGe@2#%0(7|NziKg<Yt<g10YU+?$=qzbSIQepPw_?hc#nSV84^dlMHupAbt%)qI4PB!^d
jH(aega$($N1rdyHNvFs&I%ZYuNbrW`{vjkJj;n#?i8OpN~)wCxF6tLRDzE9_y7+3)yK|P$ZU$GPr@<!?Dwy*ArutgYTX{4c+iIS5%teZK^&4%*!VtYs|7K*
}{c-
%BiDEzbNs&$o0L>C2F0e7=FMebAa?lyi!M?Kg#0^7KiD}6?2NM@eLVhwZ~!riV8hOpXH0RM20L~phjNhuNoLmUnmR+S%pGmXJ#U&_3i#u=bsn%}Y~jqBNnn<
OWqwiyr8k5Cu`YC_v6<H01DW%}1vl!Bv?Pq;8@$s2`#%wYw=93p^#l2Q-P9@l@XJh$i=1)ONMaSb7BaN1lvZ8#wn-
2}|HjLQ<tj4*4Pg9XoYMLmOrE@dxPKT1Tt!@dBto{fOIt6h*gDJ*<rKx_DLN3jV>COD2PF3WATn%H)!rIr2u@*sn~xQns`mS$8QI^Qc&?x!wh>?k2Oa3sX!6
1gI<7$;WVJT!}svh&6DjLSjn7DI6?sKQ++N>-rvhc(5`KA=y5w(#uWz~K~)V0l+1-
3@^7>cRgj?M%;5ism~QoIBpo!JsuLlMU=!sDvQy5(8?WBM9w|$CE5I#gl+VvWgc>7{g_9(u<)sAIHW~mSJlhJ+{-
anAt+oRU|Miw!v&kGIQZrPGZ?v_*zqn2YY>l<Ug4z+eQpYrs-@TV<|L0cUC{R;bDQtf@N`y&qQK}Fd`gx)-
OWSVtd{l=44CI7n_@`F~UCw4AN?6_FkK8ui}<Lr%;s)h-
fROJgpQ(eZ&w(C01!r%f!L@*hQ}SvCrV7HU;N>QlIe_w4ns3iS>8p#v}2{xNmAVL&>I{(rFH>p45W_o`pY~RUE(<NDdC>Y_(I(AP=zgTk)eh_f0x^e6V7o<}
h-tChXS%sq0@5DN~{cR^0^<p}PWZN&v8JN=Qa(<->ao!(iQPg<-
ZDF>0}rhowS$?^}nZG#%o3C@}%>BwPeO4dyMJ!0upl9jO2gF*QEfRt_8z<|MUJ=WC#_|06pvg7a-
lzZvV41~Ki(PmUt_S9%P})!h~L6wD`gxH4IJG4^1+(kPPdsshBm+^O2OoH%IXOJ6T>4=VA{l4)i`{rq~llFNh$tA&R7D5=W6Zq_vRG1h;M%P26DN@vXk`@Zw
*g&>|E+CG+Y_|uL8{lz0dFrDNm<aZR`<wbBnBOJP9%hU~!=EQUe5NaoA0HV4>u=JO2@m^!*d|HI(bFxMf>Z=M@1?9!XHXYim|9!~^kI?AsYX=EzxTIK(r2Rd
iPaRnqQEud<^us&qLUT!m1k{tlMR^$rJe#(W+(m%AUBCQkrLYwDA(D^loPsSuLFf`7$INU$za`)$L8QAM!sH@Ma*fTsu~mRx83#>jcsB_NW3Es7?OCm7F*_7
n(Xy-4Mc3$uO=e%+p*k_H&!eby=VN?zMRg9lr|(MEiwc;|A`P=-_th$oOui1K0>Dy#7=&-<DZAp}nQ~+}BcmS9DHz3V4+!QNKP7VMXTiofj~hN#B&?JJq-zB
VAF!1Y49y%<s9%K>M}T=>l<jsW*}q`gLjiECHX(U1UK_ZrMi9my=ygO3z!W8)$ZO9G{hNEck@QEo8*Y?1zVd+~;Q1+CUQ=+<{C$V40Or`N+|{%aJ`NwYj_l_
i>`sdUY)n3i=dp&7fMvi${o<-ib37JArcb@!-WgD`tK5LK_9|t@V;RsUY>45IUtoq&2<6~z(0-kKqcf_4y1HT-
9vZjgx3=&L6xi{n(^x&DF~*>#Ab#;Y^6vo8QiN!Ha~U0Lc=Z2TX(=@^@?z20?~*<!qTUMw2zM0)uW9VbOZ7+L=ODf`o^}J~f3;_RVzGg6?UQ>Y&3>iI>W)S%
af$Z~^+Y3)n+gba>0lct>R_lA(eLfRN%Kts#F>a4BSFS`4+0Ku)j;|F#~?bxH%tsy#pZSx+6ykC;+vcM1z7&xIvu-PQidQtSE^N32e4CInD;5ukKQrpx6+BR
)cM+gJZ85XwKVjWiW(gaD{fUS$HjTELGC(mYSr_!ty6j5;d<gZvUzu{E*Kq964iqyT<afeLZ6bfFgL&RlS19}3h0DQ68`ynW0WH@M;aI2mpZX~$?HI@IO9oZ
$Q|LrASA&$b`JE_)7i-
KqVi&3L>X+7D|r+d40gb^hW_Q;`LzQ_b;^>N=Sv>xu9X+di+>n&zG;J|ZqdztQ+bdXiKnh~*JF)(qIff_?5zfpP}d7^HcdWMk6R(4^2|yKXB?xxy>fsH>PJN
lNMi|mlX7D5N+2P}2$77srNZqpZuNCGjRLGRCr!0BEsq(nf)Hj(;+^T-sErl$?ha~66(-
r>XD;kLEN?g$Z+<DTr*Q~LIAnqknBx8W(D#U<#t0KLooTeZlPT|g>z@LR>lXCZ%cvTVx4)+Pje22}>^EzVC%rcR^i6ecvlr)W`zhG=lCu;CnJ&F2m0!K4blp
EzA4R`~!jN6o5x)pcoD@9(U?Psp764rna%CvTE}0`5vSA$J-
DhM^*<%|m;WhXVS)GO&+q!143qh46cVjC0yU<<<)lJ4f50uA*)<&?K{qUn!BT!IolOEpy^nwbJZT{mL_`T{tX@VM#h_Pv&tG#Yg$8;ku+7~^jKa{z1+CDXdF
8xDPD0Swap)FpcCRtt6?`XlVJ%#4w!)_ywo0y&CkNUg&hA3DL<;4l7^3TgDm}!r{4>EzcBbm6LXZ?#*jje*>=8+sFTIH_Xz1ed;ZJ>l=tZ%2O*MyIyKDsFQa
}i$h)8-3FF3`HwMt|!^%+Wd@wNJ^d)ou4}-
;rF{WfHE7&!XChhdH$j0t#=Q6WXw;g=+@m%riQ2Y#&#9mYZC!Y9N&})KY3h=W+rVZY#<qM31tRF#Y;4T?}19Y>i6h(@V^M<4s3dX;`Ru>_-
4QONavQ<R*3B`JA!aCfYf}F_dbBw$58A1Hl^7RdVTQ^q7`qb}2LH6A~ckE=0C+;Wmt)v(tK4J#pmjBtGT?Fy(hpv%%L>NY&FY1a_=(ncHJz)Fy-rx=~U5*33
{ONFI7Uu$f_uF5K)G$wS`Pf3HR8v{8K>Dn{-
~n^V|A1NeTL1WwI?heLjJA2XwY<`ek7EA4Rh1(Z~|R*}_8<(nsnM&k(`XL(fTgryv_Yc|6`<aYmoim9xWTGXOm&fy3u(|*i5AFef{D^gZIfSq-jFzmVg&wUk
czf93{lSkuD>WH|Zf#vM;4r4>LnE&em>~9BbD9T<McF<WeI<1EPX=3_+e@Yx@Z88v=-
|le68THWN7f*u?Y)k3Ilrki33eR54Psax!xN4!tx&D&}xW|hShWUxYghTkc72(ydAG%It9G7NAZcuQ`%$I97MuB0>$YB_CH)kMivhdK07PmQ-
{8x#mwofF<CGe+d<%@SMzk=*klior#`KGxGB`(7cqC@20e@K5hahWtu9TngyfY_-
M;`%sM;ViJpgVXT6xOgVBCus0VZnHG%r;K~S)?)rCL^ZRQYbtilmQP5ABh@$NILc!DfmFfBys`dut+LlUAFHVO=@2_t@4X8|Zj|=bH6F%>Iielp&l(qPXYD;
oF`Pd%JF)p}ofS1aVr1eb12)xBO*&gk$#ftRfVSDa71PQCvi~?`X56<jKTwZDP*UK<!sfwy2CDG-
#FoN%Hu1vV{@6f~2U&d<pR;g`3_`nk)rZOd7i*q~#jceHvQ+VMBm#}dUQ3%A$1OIUn!jw#Wb_PvMIM8JI%}2S5p5c8AGm_3-
9bFV?K+ql+nlKlK>$GbAtM%lF%wM<cB4#n<{&meP(Qfnp%*7A0&g)VB?2EHjnD+Tzd^d-
*>oOcvoATC9|kr$sRF;=#o>Zhj<hnr{%Oe+UmCR$oq~Dx@Bj!zHs{}%Cd%q*S}tSVzGz0Dx!~TM#P}wWpF@91SK3<LG~gL=d*$fm_9lZ06##!!^vdLsJGoeN
6j=D+LDt8jx_##I@c)zylj!f1T)b>MTy-
4!{lOl+mY(=97Ocwv()p9_6P^|AHg}_8KVC8TgtjmcZ#a&i@T}9^;o%gOESf$Z@8NqVeE$r=B`G|;aX(heu4e0nKJQnx<;I&@44fT8cl>{Rg8JLujByB+{&T
=Vb6fmh3z@Y+efVdV8E0;{C$a(2shL(H7ZW*BI5pNvRqyZhCOHkU+>n-9hgDoX;_{^=LvF!(vp`>>#WhAVH#(iG#P+3-
b4q}jy>x`R$iveOWjH8U>QWR%ZNFT#E9*)pO5JmoYdqZXn7>m?f5c4as74x-5pEG47p>4AUhpe!0H|;VOiB~X1-
7#RvA0;F;_Ct&4vaqy!Y2VZ5NQ}g0v!Ms$4deFZxcSR?v?S!y*Fr7Fu<w0Ihu<eQ!*r1I`dx1K=CKg??kF%3+o>#wK-
^{HZ2H?>8Pjy{T*6|N<orle0a=O9!`q3mXgN~6UMH<9fh%6o>TJ~GR`~&O=8O&YVSD(JUHx)=N!3Kb%o?#R#5mg0Mmgxu8ny?#-
uK{KlC|FDg!hH3Fzqj_}$MnNu-n2NP&-)BDeO5)P)H2_BpNp&1#%}4x{OKM3|+qp)9QyCa-
1!g)xx~7>_nV$s|^r|Bye(W(%jv5DSSb__v8ul}h|NpEF7t(;3vcg&x;ky99Zu_DEUBR5!o;QXght>*ySTfvHCvazu|~Y@vT8YxXyjP@8!k3Vcm23XYUwBCU
*^aEgL>6YTG<0<u5Z-m3s8aP3<7|E7(7sVjUx*2MM~xc4R0_p_PZz9ihtmuQlNY<|IHW#~yF!}HTpf$dkb$hNDn^r+dU<51}}`8JQkseh*-i|9hL<&Rz0?-
2+quyt;%eK!8aJGI8Omt@@lYX`61@nq&M2ZfJj6hMoxe~CQ?U*VwLK!qi$ll|(`Q^i%L0|y%dJj{@UI&x-NGMz;`1Q<-
_8YQ!w8nAeNE+NbjzqG+tLxyl~z;R8z=iJ`rHsYEoS;6%|@Ulg4YATVY030MC8`rlbR7Jlf#S0Q)mtEWSJcSPwP02C*es`*e{HgE3gLf_x3d9qIWEDj?hhOT
%X(%R-2Z#CX2!=|Dp_(virzQUWGAZhuZ0|$pNQHR8(EBu<rKng=@&(^dj|oMXW-kUq2ZcS_!dB3AAdC{*)GK8_bWag`>kw%4SDhOgy|w;SU<3kgYw@{7bQjM
MjWT9G^ptC=N_0^=U2z&O)kLL}L|m!>YB9^o6WytpMBwBFu7{I>;-Uxlzow&-&;H7_)*{<86T?z&M|*rJ9(qHtm40F1=B@X-_=%l0r#sS}-
m`o1KWG*sp_DNsx4@Spvw=g4R!Gne%Ejw@XC~CKz?VT1ZC>FDOb&WX=S-e%-1%$+%<L%PXrRI$A)w9<fyG{D<un37sgv_$u31Y3K#Cm!;o31p4Xks-
u%=#pa!0teH<+Pj;nhl^XbHT{|76ZCppm~SvVDIY*i`6iHOdKz<3eq#0I0(yxD91?-
gR_gHKI+te*|``d&j+k>yt?dbyCW>mJJvA*@YXT^f5K5;_l7XMLav@4ndNaw~4Q1+VJH*e#q1_cL4BiG*y(oNhB%o9Mhi(Bd((YG+_@W@$OU`?f2J18PFI;C
^66<G#}UePPehY?&xic&`S0i?`}>llr^BMIoA2I6Ip(;TyJhu279C;+rP$!#p&bQ7suH_b8SSvB=hXqLd4sDpBOby=HJp<iJJjKFv4tB2R;H-
R`|LIqbR`$<S7janotcnO{R^0Z7Y)|noUrl1FWs7?<#d&n@;rwJCs1w<!Owf4mD=_#FR@Mt255-
M?FaU?KxM?RdL8Om8tM48S3VvJdXP2Mv@PpV@XsUa1qnXE28DyZ5HelwYdRAFbHg9Ew)g^reH^j!>DbF`uJ#!mRjb@vivdO?Gi=%ab>m{MY>mAfHNH-
N#K3Yf_y6i5%bSQ4^Go8Fcybs)44FRqx{h<4yN$F$g*uZ`bGGF9=wK)WB-;Ub%VfwABG8n!>MOpurK^tsWxk;UJ?mtKyK{=FA<VqXx+%$d>wcz-
7NE)+$GM799oCIzd*1^Vp~}^kuIt|b>?e@4Ceh&;iE4A5$7PNC3VU5DqJx!K2AM5dF&cuMoF87E^H8T5+HPXDVe93Trf02zXX%}T$U0lg!V8I)kP*og^OHu^
160##fL8fPFP;UW<Yad<3JMmNG9$oQ{5osCaQstHcj5D?s7t6wd_ubt&rQHYIPjYM0ZQc^0>%IW5UnQO3&a8LXUM(ZsiV1dPMnB_lzDf64*I{e%X4|>nIE<a
jmva0ZQ1@w^J)(@RINIGvbbfl|I1RsOlZP+#1T82l3I#%;PLzoMY%H1DNwiu2pvTf_9puPq$fn<&uydzQC5o7~e)Hmgbb%$`SS=)`|@eo|<kyexeX6o`>$vh
3N<jw9KT*drhL1rVA=pY}Y=~C<nqaBeKZ`Yib&EXeDg3d1?XGKaU@)E~SfA@wGpN1cC#ugj|Ccl%3AR#BFo$gQ|K`(fmlnf6gSP^S@q)dE79!`N*d|IGR7IA
i{P|C*)g}ymcZXBwV?4*F^f0%x*^x-@OAXjsnYK!B0V*-<K%qBMG<QYah8$D!fkgrowD!=bc}98NkT^REN8pD(>ZPoWA!Gmt#0<wMS-PtQ~*QMR+WW1o<Dz6
s}4T;l)B<op$?)xZkn+fo@FpeKZ(oi*}P=P>CJeO6X|%U$${HcHS+dQVMlHMW7Rz{?q$eE&2Xw_s0EPbmO(n*01Qdll9B%`y<$RDM@#5e-iF-
K9tHk9Y~#>Bk+-Sx0C3WlA!soU-
p01!GaaB!5g$|t%qJle`$Xi7FsG=1@bbnL^+$J1tfaUytU++!SAy$rrX#zdh8^<pn}cW^_Doueu~n7q2yhFnR64;9pSroKHG#BjS@(cxY2^?Qf9VY4a3^^Vx
T;ulrC+=gM$dZ2I%oh>U=9^pixbdxxygSvteCdKxYin-AwL%*&cA!qxN)Is;MO%PGe=$mtk|ZqIw5PzEg_Fho?$SAU9upe^hZLCs3ToSjE9ean_c|`KiKr7;
dOnYfPkZOmN4+DOt0EMDkPgdG<+9pgj8KjAZ6Em4(2LPRi$;$pGC>WdQeLNt8#Z&d+3ru_YW2h>!rx;4O{uj+Cd@@<w}yBF=sTfY$JaK$;e7;tm%<p94w9CY
jda%P-J!uncpKj5_POi8r}D7f#cW*3jo*OAH0vHgssymBg;l=X-
3dMJfTQJRF$_!%&~pB0}L+?K?rxQWoVoZ~2w5Mowg3AKeWcr6AAp$NNJ7avxY?+8e$kV{3wTd~!Sdy|^+G4kqa*JZ#L3IN$xX3#_90(?r>xbp~SI89(XIfaH
lkl*blqiqZuO8E>>}i_E;I^dEB8!C>P($nKNmhN#-<A0j{p*sKrHya~!1Ui;USMn51Xb39ZY)?GtaB_g(Xtg)SL&5XRi+C=X7-
R6hp>$*RYdk0P}Tx2Aa&oVOK!U-
psju5WQCOHLni;z4a3s@3vM=cZqB9TU<8r5sOs{)wj&>(TLSoI+blZPl7Q!&Y#(QY_+_Af{3`0;#UAP)9SE27!;{<pu_9~14XBm=0T?|d%c$hx-fx#~+z=a8
L<)D~trWNpXVg~GtxI;Wd_-
}TzpdtQT#nc<A0HXhd@F`nvYn<ue&C<3K(F;n^?4L}EU@y;~@#^H#bsg`wPNbDKa=1%4ZGZ<za5S4V(l4>CB)I>V3QsH>_L9>?Fv=IE)yl;&7z&y3OH6jGNq
xRI|ro`i?!9j8KEDj2TT4(}Jyb}%7ytn}s8nP3}L$B(tpDRr$my(KZ7ono%8Bd#gAN0B4Og5K+J7IWsX`ijTS}w&DTCZ4VNweo-BT_mh;w@9;3|XxPT(uA>h
cMsHVzp)@@T9dTpbE(PW|5|V0h`cX6xT5dQu4f+l{6u>J;)>DHG%P68VjXJ0Y-
tnH2cZEG0$@){Q3sP#|f^{#(@EqTM}X@>ioew*W#=pa${<c!Fe`yt3JhDlg&t5M>s0MXy~w805Hdr&b;8A5`%(&1`DzRvapFK8GCx;dC*x!j!5?bN$hgdb!2
`6xU^$Ch0!jP@0P6$E$x$-Dsc~m6@ra5y{B7G^=ckn?#8e4I^?qKQr70}?D5Y$Y?iYRnM8A%F-&Zi^qfXc;P-!x5SkRqCZ(3>^kR#i-
eh$xyh5wjl&U8?(%hdy)*dcLvsw<<He19M#gZyy8N#IZZ7FsJU?#sWV|-
abblORs`(nz)h|4e>!Ff6rZx@He6eE8kX%KQP791)DmF9R}D2%L4u95Da2}${;=J#8o#ZFCZst!i|UQIN_>8sTCmVgQ=ZA>4zo`UpQoH7lg+Z8agpZX{Z9;)
Oka^U@CixuvuC-uCCs+#)G6|)1xe+??`{L6ej#A53+*@lekDn85~?fJ_ruKVy#(BDzJM*ng8K0>^8*3@dq^U9B8qm$5Y9g%IsL98MQw+Aq6l^V|4Ebn>hP=h
=#8ZIx(oyn^;a0|-H(Ak$n;WrgB`pZ5uNPd0EhO-a#6<n}PsD~`^r$YI4RV{qgNkRCG02q_e*?XjRvp}HDz4-|LyHjX<iRrEo<F@-
%SDoUrF4trG<S^k&H)9c(lTqn!+Y&%v=%bgsr1o4DNk{h{R0i!oX1#Fp*r{E^PSu_3n|1V~c0LdqAIcWr=;w^NUyOS(dfbFi7gRwbr-
k{&;}{KDP;6d<R1E6OB@Rz~xT%XDJSr<@g-xUdQ6e?)|A^wA%5Aq$tM+nY2+f7dbJtCk7E^Ye4}i~d-
L4~KWhD$1bw=l*l&(Ib?v2UD?Fn8N6wa0seqwtLk?tVC%X~~f?lNN7Xp7kWLQ>({G#*-apJVF?N*BnBK5eH(A|JvRjOuk#W(h3bePs=)cX@P5OwdS~bFOJ-
EHsh>^fV9oplss`>@9N53~vsktcT+-daDxw2Hc-^!H+8_g@*wBa4Lgdb<!P-^)k^rQvf|aRp6tK&0DnV2WnE3=&VM-AzLs)&hS>B1P?DA0oePKPk3OnEskHF
V|9^vb;@}uc1;y{BE45nh8x<IY4+XCEN7Q(1duyee_@9VBtTLbt2!c6>^gzakQJ)lY9t?8IfvlWAmaI?MVSK5yuGKc1L;bFrpX$+4C`QfX3R+7o<+CP*Pt0?
ZI!zn%cSKN)>;1Ilg<t@fJ&8pemEFp=xh0%ein#3=kaA>jb9ANY?{4HIQTJ$=0Lu^x1!c8b=gO!>Hcg%pGn%dg^CQ}A;cu@2Nc$t;Ks_efYe0tF6l%)I&2?A
0){3Ui_5!zDvcqa(e5W;5@+ZZ0;1+dN(9E=bT`R&`qMe7?bq-}3g{%a`%srepI{d&BGlRcQG0YQ6UJRroqb_CL0-
4nTgR#p%^D9zkyI{esmTpG?M2Dz&j!Xo9fZzTgL{h3i_=dL{<+ZwtC_4dPHQ%F7O6k2vZhKFdN&OClNL-
!MO7M&)^fRb{$oFULmgLSRNYBPF|{!@2Bfw}azUFE+Bb^krj4z9RhiD?L=NfrE6Q3kbHRM}^;O)o`m6CXT`>~UB3(<wYE4zHRx^dOL!(;dthdkN*t&_+s!p9
aV=B4>ipP`;Fa3vGepZTz+;O=^J0^&+)74xmt1K#Axz%-
HLXUcN`f%Xj2$}(Y8;3Z*lWSSWc{2qO@u3Q}Rm}N6p9)F=IXH*<z)5YCDM(q6V+DCjGcFtt?`C3+LIlPj(xdK#J6|_K%%3f3xmmk5HvfU6_N~^A1{W?o00{I
{=nD3f$CV_Bwcz;#(N{Tf%r3-k3qL1gYv-2$;^i@8X7?C9G9!Dhd4>v34TM6ZMEjD%>a&Fkh6k35oj}M%hh$*^JekR)OVYhsd4+)qIq8+wBzNbe+ZG0O-
%YSIS*<_@!@My9sUf#TL;BS0etNaI%DSDdt+vAklfQ+qP}&^DnLqJ;G?bwUf#zFBaJ~yie7FN&HDoN-
*{;;hSBH$(9kV8ob2>OCKgG%P`HCYXd0@Q%Iz6oJ6`|Ci%`)CGbBdMrV5JTW;b!vvyP*pf1T$xy2i&FYR5<vGDXxK@?`3hLKGAn#m7?AB`BHV}!SCBhgah@(
g_(+i_TOl~J0RcnH0jihjLeEy5V_}H-ktf0c>~g4YO?gBtbxeit4uWTM%^I{o;pOzMDczl<;_=bZjW6&jklhi!>IS9vKDCq)m&p<1V3jkU-
<_B=fU%if+W=<g7(e9OZmcR6a4ly+71)@*&er-ScarzJA*i@06(q-OLtC11M-|8gbI!B#S{2z9o47E&eDreLi>~z0Ly8IXV`=Mc?=2O2wH&bx-mD9=n)4{{G
^1eIsh-M(DBLqglpnHJ8w8IFKy#>2iENNSF(8Wt&buDC?b0e(A(iDss-
^SzxDNDJzrSBx+Wuorh?OY4<3u6AE}KWI6Np@{+4b056G|pAG^oQfdfJ;%LHAGtfnD;sjzq#7)44Fr(KpLbN)pUX1}}b^bbLk`*%!C{$TMNLV}v@<^{fRh!e
BT*GAC%<ZJ>xw;wG%@r1LsgBEDowzZ1gJci0SOeS@~SlLE=^RR(9T65z^ZU9A~BU@-
B7uhe|Q=eW#cxYpPze_Y>qaO8qr{S9RaU8kLE9IvJSeE(Y!OX%?jAfr0mBik=C<eE-K43{_8NEy{r>)NF94X!lHToGvwS`7OG(TD%T7u~>nZaS1x?8VSN&-
~L`L)|w$UhFIIS81zA9#lQ?MyO<w$pN|AYv*h`R*w9-DoEHHd+MvH21nuXySK)ijl6C@9|TV%Bqxq24-
6ARv41=7AxmuDYeZ#;j~w3TUPk737+UBQr;`t#95alES!X=t^Pylm9D8jXSUQ?g6LEbA@Slz@!h44@M)igBzZ{H(I-
I_vFDY^Y1P4u9rQ+yew~#BFOeBlRw#f7egBV|=DDY51jf~r`<bqa?kYCVOZ?U^1dO+ObP<tk!x8x}+god6(My6|2BL@TIwY9RZ<_8P1Y>B}SRJjIW4SBvOmo
SrPJAlFSBLp00L|~6N^>bdFZ$6ikrE$Icvd?kU#PgB)Dyjg*UP#4TRAb=XIpd84R(9-kaxBV8FJ@-?a5|J$yDHBPh<s;Tb9I}^&-
(j6~8}%L4Je?;dJteT2*V(*=qO<$dx5Cme%nH?WnLJOw1hXl}+txp@4`ZPMY^(s&q%lxR)$!Cv0xWelFe#h8P~5HPzlAJonpXY3dcnDazLqQ4sBf22CKpm>t
H2<PI-$v|f=<5^CI<1Ff5&2-c-VCHK_)<sg(T%;&F*)DnJx6R*rvb-YF+whYsTb*1Vq%bfR2DjbHCrU&}T3=A7$DfiSi&3@vufW?ub=3AlWnK$D(y!e)bNje
GV#4Ixrt#s=2qDm2e<bbb>?1G%f*F`!+@CRHiA)T{mGiw1VJou&AMvd)Mkj>PqCUO_c`TISw5#HOd=|!{KvnEc&D;mZ5!DF+Rg>ep4@~(IMN^!sfz#)OazSy
F3*cHt@WK`SK8+yTE)O>y{IbzUbqr09(kN+I$Rpt6#x>TV}g3@Cp=JScRvgVl4(<xe`mrYRe^&{Ed6Ra*b%`hLAwJPo%=V+Vt(MAf5@~O5876CG1T6j`)se6
BNcGE62xG!35RMB)zdAdpUY&q`s8}bo)7p8Q-6eA4mjz7aNV4lXGLMVyjdTHdE;K~81`-jd5x<e2pz0JrRHY;XY)qeDJqEd6@0T7n!(bAXC00qEUf*>-
sv2=_<5_Dy*N$mef*%ZqMgU#Cu#nj}8dVQ}a^vnEbGb4?P=|W>#z)*{K9&C6{p0(#J!Mpg$`O=$_p0iYwHwng&Q&`|!5wiXcDb`U_(*HI4gAhSIF<Ns<kDwt
DPK;*1y9n+LI5szYngZ@plQgxACb=~&qzL2QW{RjX!GQbhIIE2)53g6sVEUIk0_|U4&)d*eD@VmJK-D->vKde^`BAd;V-
;P1F{1?#PMFu^g?Pr5jhaE|B9PQf`fn&wRhQ!sFZ)BXeTGAk&RH)x)BK8vDPW=Y{4SzE^6YRxJCvzoXe<%t^Pxbp`s|=OdP;9sD%r%(cmq#TMKihjB|#NA_j
xMX?&<}Sm;+t{PXVq4Eh0PqUm^eaNM0+B8~=fyeUTnf&!YC10j_!`Z=ega;z+urq1e8?Ye;wmhOx?5HQGck8R~|u(2%m3o;R@&H0^F;=&VL(^%09J9hk;$lH
PuN@KujAbE=K*S(LMbF2dSV%(cufhTx+Ep2g%n0N28<Rue>d=M;jvv@Is($pNf1@~a)XYF54S?G_^}O*qNRxyqlKo(S;o`3{1kg${>&QH|q#S0uD>0t7*8pL
_YUN_8Qe@0OiZ(GK$2pa?G4_ri%?dn;8O8&U8Z7{`6f>{i~N)lg`1a`|lLsOS=hUD9Lw$Boya!*83GYPfr$Sf@^RJn;2K?E5~{tb#f*NBT%GYK-
;EAwB==(qmQsHTJY5e4$jpLT-Q!I0mTR1g6HD|JW%3oZ)e3oIH-TT-Yp7AK)#SPly)^_RJr9^vvmjZ61YafHe6Qs%(&Zy&Di84FdpVEllv6$OhbOt_fyEv*$
^$wpTg|yW7XEZv`mKB3)FA!r+DFfL!qRRx<4T+1*yV=-S*eR9u0!MYdH<X-
Y<=4wkhT2DS>4Mc)U%g0sHIwWCov&$>D~&p488>7uUI+8re7>@>UB4OeYdEzdtjm(7q6IO-
~l#VMfs7Sb{>>znSw5G!&cNTV?#AA7|GV0I&mai`0Vd<RO3<)dR)N2G*$r!tUewA!S*0eS1D30OQN<Q?$ds}dR+zT!NIGQD8F_x#<i4zLkfOj}6S$yZ}*%#f
CI!fM%E6)(9@j%q_A;Dv|o^e>KW)$D}tw<617zhFOPEo25WG+6HZ*dv>u5Ot!Ey`68Rf%sG#>jZyTo#^O|ssk@1f8w<KJ<)GZ7@*aC^!LbXV-JX&hqD*iH`$
*$+}t3_HP+6sLHmMHuv2#%+E4INsWB{oLD7bH!KagZs1yWAk!`9)Gf;NMU(rD6{uT98jP$_U1zit*0xEu>Dcu82zz2JS??ErDFJ+FA+~+@CR({Ee+SwE_e~~
GUkDpsb#K5SdJC_Lcsm8sR;NdF!9z2+~&7v%3*Uha(gvj@*=`)&NyMMGay(JRCUhCvOqbJbuwT~K+q!n`SOFhRxZ31HaJZk(e2|GFg09ey8-
=!iVQd7ydHMMxMBH0)3M@Uwi<AsjHoZv_kePHl!KKl6Oe?8$utZrb-a^>P(>>+Z$&nV&B6PqbLQIF~KQOH*bVgyC;jw|JbUFG~op=x-
hT|hIMo%hB+oF%(^=LjA#uuETV!3F2$e}IQW>B9xoM*b*#%ecM(=4z?%&u5#=^cDaaO6=Q|2A{i|(n*v9UMdq~V0<XSN9SPo9X{?yt|~g@5VNK;<%<w#1h>+
ZB_qT|5hMGWeMB^0W>#!>!p3<e8h%c->34oXJNEhSH-
&iPaP*oT{2G~m>@u@Ztxn#GSvF9Z&<k%N64|CC{$)LKCB6LG^5+jy>CDx~S;IPcCQRBSF#jq|N3?e|EPdTd-
#Br^@j>EK&G2~^pwzITh3&@TYGWa=<Y`2+rZQD1n#-
%_#cjHWTZbvTw+N|NUhCdZzY~&J*JD7OQ6!Gj`m(V9asl}q&~$Ge_%7NkpqSqSBr6s@6K7mwINqnsK&s<j&AjRZ;8am!+$i2E<=cy8nfurOJKqem#9&8;_<I
64?~;hb6sG3?qc|H+zC*|moD*UMIY3R%7%K{xpqlh77)P+T<lb|_8$&2+m6bW4?ed{=uBjwk42bsy9l7pOKxBF68pQpS^hRqR2wGSOecB$8xrq0gw3nt8w}m
!l+`fRr1|oz1DVo|^u}os{^^`|EjQv3U>A6t+K3`KCc$twvt{;~kCv7X!2YuY+{6Sd}S;D;gc1%(vZH6acA4j)tK^^U%K+`byPtySY;_)f`FX&bP-tuI2&3@
EQ(D`ZLdP*E|a4}^usO+@q#liC1z|~Xxg|D=SB1#(leMK@>gcZq~Ge0609Kh14rQSx$Nyqx|H7UWPii5qpQi(F=<d%vPb-
9izdcJ9wZ(}JeSK4_V7dDhi%kkP^Ern6DaOteKmI}SE%7d6_k%@V6O4-
%ECxSOFtb8qEx{bz978Yk*pbjmzD>yAVu{`bxtKd+h&0&d5gPk6#NWvJ3+%kSybUc}6o*(AM>z&hGNN5lNDjfmSv3feoc6NnXk{jT!jMBdqZ%zb~MFYk>gbq
2Nlp*>w3iyD2_VV|Asy_Ceo?)~00Qj7=Z!?1snI*mLsWfvawZ81K+hwOm-lrZ@Z?mF<GXZDAN^emOVE+J9Coxv6)LY?r7&geJH29Ze<G*aUXA^&ev^ta6KyA
v~lf#7T-DT8YNPL+<;?dIspKPt3jQtlD|H^s3CzX|a7db3vcz(ZJ4)iqUT)1?Xp;ivvc7IW|*&k?6FrP75hO-TCQ0Z$lB2WzPXf$IQCg?;|xRg~ggA*fIIs_
Z9j=<x&g2v*@P1XYARNM7e<^j!Q)l&u<^G$RA=n7(<*Bz4uT|21W`Di$hOKW`w;A>Kn0KrDy-
{PC9k90cEOmp8<1e|GJw}XF$=TKtF!s_}2H!C%iIPK*6d*jTjQ;)CeW0de-Xfzh=kN-
>;>PXeMtvtNSMsBn|P*{=)ZK_M~McR#)LU}@I8qWd|V1ySxCv*6u(sa_eOjnE|(1q31tFbd@=>bMnKkV&KZ<8`uSTvZUhie$}(z~d3cGwzphG1O@9BW)09q6
Dnc>+8tFXJHAgb9F*_u*Ki24M<*LDrre&Lp&mo0u%dADN~|k2S{Mwox<;&aFI1!Ak*XgD;IglE*l=9qJD+fh&!S-
gQt~I1O@QIM~Ec9@$1u!FRYg*esIZ$9W4xfFA*gfE0)cXl=cUel+HAcY8K)4F89Czu+4UA^hU;1{dn0+rR*duRZJII}%elY(S?&uOc%ziJ3;)33+ML{-
moe4WAT$HrO;QWKbkpWNBaYPDru>u&Vewg<M;INer#rMq~>7=yslh=9+TgVi~D8IdlFx#bb<~rMf642h>D)#qpbQWoTW2NavJ02ybBG`(GY}o`2MWUTr0OD#
^Bg*hx+Wb-RPB%{*Zj1E92{DLn!kr0ET9G_C(_{jSjxPr|sG7;w%)g+oBxq{Qky1LHqgLx_iwYB6z3(`K4u{7dH)ieM_{5Z@(qF}K?oq(L+Xyq20}L1cZTl=
Y&kaW`1MJPgh+Gu_*ut4XTWR9?DQ$lQGGPZ03zUhzqIYCPzGHzZHhGQX4eTi;!!P793awpf(ATI9bjL_;^&?_fxX9Xq6~ae(4_TN1pk=9AL2Op_L8=MXR%f%
FvC@9vlAI2VI;)ips`^6!FTuW5ox7uk+5rFoClc7zG7%44y;{jc^6g*?_PK-UiivGOHD`mG4yH7fzGZg}K+nTKOby6?+|l^^qjq@Se1dc5`2YkCky02VhJ=E
l|jREU{}*_WWJr&0@ARp|LKngm-!?^zI%sAiWhMioHuJ*Gt$`k6BB+_15wpRJJZf^U^-w#ii$i*R7Iv3WWsVhv|L-EK!FOX-
=u5jHeCrE=9mE`oRx7KI<6)nfLlSxowh&+u2?+$Qq;9#b%#gp@PhSXwei<YGH{HIP#)r{>lZbyUO2(UUR;CDM)xo`M%48eIaFuDs@t16Q~HLhW2ZsI6Cu=~^
{{lBtC(FG?wCpiGJGPxaR>eo;3+xXJV~j2_4McgwXXn@;lf^*+Sa$-
ADBd}<szmICVsvXK)k$Q`5b6K^v&h}a{{cPkg8<YiC&2};z|@p}WVMFxXSxk+F&igvq)WEZ_Mp9>vy6VvWN-RTLvqfCSOI9~z#D1I3!-
YT2@6*gWT)*=XIKvE_<z-Eif17y@W>X)pD#rv<pbWNeYBoafdFa6S<#=%H^XL+r}0x<0B@pjjB(hx$Xsf(7zXHtG2h}><R@d;W+G|-
WQg`)p>2?mp??E^{d23@s0SLaWnJOsb^GYd<&o_^(fqVDv%egoZcrpCS5q^v5AghFelY9luh;~nN9TeF28COEp%-Mc^*z_p#eB2LrZ;-V&|Q;bD9l_rf0-
Y}9r4A4_<10Vk_N;2ATB#?Y8iF!V*=}ES#07|lKeTmJB&4L+zYxmd>E(nN6*xa9EO0qvjf!fhx&Si+6UoyYqyMdZK2RsXz)2#Z9oFSGw>GM!1_@(rcVWT86b
x5~OA$NmTjiRumnxR}lM`U73hQqKwyJGusOKQ>B8i|?DgkNTuw<*|UK<$hw8)T8W)3P0L#hrkQR8NPIx5@PMFAN*Qj$EE8tujSenyd)Y4IlS>Nrx2xkN{T-
h*BG{n)wutQ<tNp3@~zbK%>R+n=e&;-NsG9Kz6yHIf<7v>+?JsVblTaXe)f%DCy=;&6FsWBhX|PM+_eQNp1^uOm&`Hi-5#heLc&ZP8L=)OOO_gV46*s%r@*-
60qv%>%jmvH+eF*Ry(wtK=d%+GLC!Y$p4^oKd|=53Y0{xrbKHM4$di{^NqwWDl=6ho)JT}BlcLd`??NYBbktZcc{?Z1F7Gh@H0CbLSZl<n%cBUfeh`fFq9&p
8%VKS9q+^fH{Ji{=nYX5mwc4JL})<8rN-QS1v2Qc71>Ycovv`JI*P6&v==q9o!i({zxy93&$4g+=e+)AgkU7~ko$-
z6ubins>#8zbe@8v3$L2Zp!I9Dsx8l<Vfxg67X+M$c%Yokh(+Uj24A!YFHdbrAN7@7?c;TT73y9jL)L2w6@9psCEa98sr<lv=;+krp!@E@J0O2L!WPQ95>uP
+7lLJC68I#v%a)wd9b`7iI~Oa-8ur45dn)q16lDzhU0np=Y=mvM&%`nSzENTt(dUqPb6v3iTjuNQARJ&?A;GU4o?^5d7cIL>nPRU7<cO7D-
Z5j53G?V#HzkSR+wyp~o%f5{dof=2&7<A<>cF<gmU-
W8Ibi?wD_hAY?TT3T@l`3F_KM9pH@M^q)M)4j?e<HHrB(T$xr)4(z^WyW$0G!m&TDCPMISsfJg;pw<9|=Q`5w4C&Is=Z<gi%+1{&byi(ghSnmziwpl+Gd`LE
JqSiaONW$Wg~hxRcj-G$E!afUH~^ut!DQxD*G<%B{*3P0S$Xl~n_1;S}(KzHwTE2k0B)o*Rr!4^K37GPG}u;Y?Ek_6va-Mgtu?a7O?#X;lVK`Wyz*`;Oz0*d
<GvA?2o`s_+rQGSPRav`3izOx}QdS(31OoIync#A{CV!B{*fKTn+Jp@FWWq^^mbr_^Z+dZ*p784qxPdqT#{$7Hk9F4Rh<E;#e4?iY*u>xd9*?j{32w7Ef&{X
-!?h7G>iz2f%4f>bC2l&A5Ssi$HpgFag6aP{f?A*5hXfF*C9>i-4JsHEL^4!CH3Z{7n$BJt7&YvDr#7C9kPaVrgt8PS||F)p3lq_0D(E|zH7=^_F-
^LVIp3u`1?I4P9$f1tNXZEEa62*uecFrtn?Tq3^GolLtLeX)l7^z*GL%ALI%(PLYZ4mmD^|3*=^`T74(wCnzIuw;z^>BujDuEUfR`@Uz#IY({{Ni$QGLBCpw
vprTP(i&v`#*y<7T_^+@oY?&R;JwAVC96DSo#f4@Tn@Ul(K;WRkI!fO*osw`qk&%RNmqAEYcm(V8gMu*hvoI$6{1yXnHe;y6cT;Ab6{~Iz^b~%LEBvE}X8`0
Wt@4h>>`mELKd^T7xLAChGKUqckx!AHht%&eKT;3ybp-
@EYi#zsj3KsS?#8ouKd!JwA`#(AbV;>nn~D)o7k_iSFeiQScm=C7Jh{{sezxy$#WFz~;RFv3_5=w=!c&F9cH;w|_sNSRI&kEO*tPnjxh6G@v^xKO<PChJ#&l
JD35>KJ{#}_dX0OLm=DD6*!ePr%HfE5iQ%)a3d@8In>-Nfw@rNt7%rwtHfey%wl(}-)-
DP;ud0dX1FSE(~p!|(9*9+&@A{Frv$IZRtmRsQJx~GJ;@Jh@2957qkcl!$^hy#+Av=j=jr|`hVj??9_HgP*bJy#K;c_5EoWnhs_l3?Env8at1^6xQV{F6<Go
chQJmM5QVKO-SJlc^NIMM~%&df1gl-vmkC~|@eshB?hNhs;J-
HMI+@(SsvC!y+s$Ey=7W*jL30WrH?*cV%SzvF^^|)c)jH&Cw7r1>+oe=}Fz7<+PZc6R>X$roW!;Uh@fGBA8*Nvv_D`Yiyv18Lf5()v?gI5{wa@CiJZ!6Fc-B
l2iP@<;-C+SYajs#uTH}!+zATyR_+C!B5ovL^di}cAc0JjiCC9Bo2G<2^p5lf*|6re}R$c<}3<sh4+WMZxJbYY^gtU5fCi4-
QFKg)$o<H4Sq1a<F<mFg~T%9gs@OEl!kCMShG0|v}yOkoeVZMqCCm}8oJc*VbGzF|;$WjM8eaQ}UoVe>e98eh=p%Xpmn9YKI8>@um(SjdtjMRs7<*Jbt+OKg
)Y+VCq<lr~^CqI55q=l=hN>)~i_wV&#VdTIC;>+-;nA1=2-6D5oas-@6OPV6=t11oS*mY72i;`RXv`&CrbBvN^0q{jH@85)4YgH3qs`-
!T|KoQ@i?!yL%1~iRLIQGpkQ;-xukT`^;wk?yjHP?G^nklO&#g&b4hMu+4m0tuY_cd`+_LUf9!X-
+(JBcTP>UPhZopo@mJH+pKgi=b8cBn>o58cuDn#?c}n#ZX9^!K-
kpNMY>eicI<J|?ZOgtj5#7baCb3%%|AJ@GESRY^d+xM=4}%PQ(mUK_>c;RpmTmq&np3eKZ6|FVFYVA+y2Wr50CJGYoj2UR>mU@~?zod!T{SOyNqHXz)0dNN7
?hU~V|LydDB!7UuU9FvOQ^hlbLpu@e*wo~%SUe(s~ljrjG^RrpH_a?CRtpt?2i6kjQ;+q}^{V`{sr`X=E5G6wN1qG9F+g{BXMOkNFyeUyZQ=ToZ>AR8erLhb
qo?nN0Z>Zl=%j$@ZjRH#5VvyjWE_vIP2A!0J@zIld%-i31qLc`TR;2r^5TQ6SOF&E>eno}a<_YOU=!KBC_0rMMjYSC6iI}{weWiIP-$-C}SBjY?lmGzCPI-
k0aQ$)INayvoR2JIyf_T7IC4%v+6q+PE6)ys;y4;Q1JAI2h;=TjfKwB3fTnfma4}kn3?9|O|6R=f2-
)1_M&VK8TR;;53B+*KzSvqP5rTy$G`4oDuS*aC7=I*IC3MH2^eW$4sqw6Km8wZs;Q1XzI^8kz@sV%3Bvc9;UYR$Jqy?G#X<q6OfeIZ?Hk!<*yGju?ivQ+PMF
{L)$x)a`Ji*g9R;O^qTNaQ0e&)|Xbj>z>~vc2Sil*p@M04amBRcaWCztsGsrp{+*d4w7d-
e%6qd|7v|q=vPog|#(bdT=bhR|)PA?Hi?Mka<qnuoGz8Fl!XOE-JmO-vp`-d{qxUB{a9Si|~<ev8~>Slm3TGHnJj6vS|2ngd;RoErmV*hgN#3EQ{D?i}=-
KytMD?Uw=NcAh2y;l_Vc2cE8~ak|5uiWe7hYPQQxhqv!8G&*F*QBPNzaegi2nURB%1Zd@Rh_VaM(ltN|TC|g0Q<&(h1js#!5wEBk@d{GLVXkG11=?W5L9+3Q
EI(FS|BW23STkfy09g)!j5X399;M}}=6=`e$22YZ>07}P!nT(~Dk-o}QBz1e7;B;wJ;L0^Tg9e^jDR%9(?^8Jp2CO=GXx>ukM}bm!tM5@q-!u*vDIC}{LJA-
S0DJd7Z+!bHPSM*4N>(=i9KnV49aH4q6~^_bJL$-<AbRkvBgbAF-Viiq@@te-RbmL|j|nWS%aX5T%ms?NWC^S9LtBvTSd^TZx9^6Sz)dwl(S!hS%CRNsaOF3
oF3_1*4gcxxXRyW|FYZ*_t5(29r&=T=f9%}gB{yXmI9ldcaN$eYsv-o2MAhV!^K$GvfbFAsX(U`;2QfH+9K_kXdf|ZILCzl>tv=_G7yozN-PCYeEyRMml-
oMvx5e$psZ5~|2$TkD*4JfSm#zMgnO*yAMnxFqh9&q3Vdou5qv{8u3FqiU0^)LH-I|l1d{aDw9ysUyQxUHXm26n5*QKXsDRiA--M!KY#)(><{`hNrpqUvBP+
Q25SZM?y(Py3Cz-D9zIGUQZpEz+%=%mj$*~DFTArC%6s8zRXbTX#e%Ju32Z+Bo)AR#HJRoql`Q~Rqc>e`gw-
6pne?L8_GU{eEacgJ3Zw?cZNTs4p}>d(C~c;@FS=2~A?D=H7P#BD}<s^=TKAD1#9!9~CqzJ4npuA6t-
`)5Bf5ZpIhr~_N}cPum=7oXIeT%iI9$yB;HIPlVQT8EyG>~{Fq2|`MW>^?cVqlwuPp2Sc(kvLuyn9?!5wAz1C`f-Hjsn*hZa)97o<6q-#-B(cK!-
^!%sK3Nbs>P8a`59zq^ct-euNoj)5L|Mt>fxP*Cml)GY;9;hefz$|N^n~zy|p^q!+s{9_do;w6@KQ=WrZ59(e|Ye&{?gFB@VPH738z{jqe)kx|>GyCha$NrZ
nvKhJkDn8BqY(S^51Nkqa2Jy@fQuX4^Brc|ZAnC1+U{m6_NJqr?ByC-ZJ$!qHPK@L{N@lBf-9H?30vb-lO;@zevAdr&Rmnugf4uT9)U$rz}UpLBM1INy!-
=fA(^_k-43UB0<J953<lc$(Gx70n9KmVuT1X}=d90aC6@J6{6FUK{vRa+oVvf83E(QPnM7y$&0`Ac@7aokGo6gKO~F-
*g~O@OTsvmFb6r6t}x)(P`l(1g0KuJ1k3eu)KlSI`5Ee?P#uOcgk@O9o|+vr)JC_DMLN_6a8OpIDo!^rgqeu|1jb9lcF1NMFIDO>AhFx=UPbp&z?!`RC2z(?
hMLD{YAV^$UmW|m1Pfr0ElB4I<awH7-
X5o3i6YZ1*KRBTVx@Rf4Fl<G<lxSgXCk;19S1JemM_oOzNp70@kTykhHfA{^&MPpELPc2V)Wx_VT>QHh_y#&4n@*gHq{y#kaNRv}kYx`0n=@EmQTfK?1lT+a
FA0-y0EE;ps+!_F{ba`x~&kT*R`KUu&KT9+)JtQM=zl%~xm3=I_~77}NA<WlYtz@D6|UEGr~Nq%2=y+@L4aG_&YglkC5$SvubVc+p;XQp)56Zw&_nB9)pC-a
re>ebDXRn><ifSUEtO{<sr@DmGSJxmXPha&5694@&+|lP*%5TJss=ix&{x?V_zs{LV#Ax8vsA9P?S=A0NR31;O-
#PKHe2uh@q?^=DYa12|YZb6&0Bd?%jY82g&J=g!dWUufIPz6oPg{)nfgkB}4P8@|oOI=v4Z{vah*mdEiHycEMFI8I*s8g~D#=Dd)&-
{UH5Xq}Q6`?jQL>(g06H0zUzreZmH2q$GBU=96#MLjGSZWu-
e{5}+b27nV4nr|KylC!K1kSO^%R`;7Fc9@k5Mz|m7R{Z+ak`RxD*~{|)KS1mr9Q_sZC&rl1tIyilCTmWRhhZzL6biy$#4~bEa%1AZu|r=IqfFB2bzak6zw6J
s#xuM2X|8+=I?UrX!?>OJUV`VrsV!OahfvcHPx;7m@3{onV3^un=i;{yGJYmoG)m=Qg;%PbHd<udOY3*+)tRc~|Bp>TyTC(Zg1a>=?^!yvrltZJ8{n0)jvJ_
0Wu_M;xx(~+L8F?hWhw(PFYmg|Voi&=n91i_%)l#yY1k?v?TdfGuxD33SZSpW2m;#-
f+20#49Md=m4dTacF^udAm}u6C?Sp=w88eypxKT^*4e#b9F_i)++D=*FR&O%vW))|n{nVC*cxg3rub0K_jz_@cI0*-
$bhS*rO=^OOCk+O56&fSxe?<aa}~SsCm_6NJ84}}E2P9TRwxC_r>JJGE#xOa>dw((8<~+J(;uc!kK}wXq~ru)`+7k)Ix7G^zh|Wcw%>yg0LhUxE*{o-
*Rr|n@tYtBoT#dC#pTcDNIRh#_krOQzZwVJw$lq?Wr32HFDxm8R0PY-hEi6C%@Fg5%rKM1?IkdQQHs9)(hZ_QcMIws^~Vo;6H%ctSe+@qYTQ#&gKl4lQ>ZH|
?^~j<EOE(MP2n|$q=+_|yNQE^npq^m{|fPIEg)&7L-+g_%B0t}U^A62K7{G>Jd{ortKM3naBeAZr<RykV;fgl4-
}nDP!}uc4n)A2zl+Sy8{AJycS^tlE=22<%I{x{e#%SwNBT4_(a84%tnPx^XvL*lx|@_B>&I<+3>bSdI6>uhLUn~ppAI?>I2eIXQ3edraO@rn0#8#D0)EKYIn
j*@*H-rT{L)FgUYTOXOKKX-6P*)2QoC%`(mD0<wl#%VbZcp0jgH1@7~VT^vX{Pl$NGlLPq9AYna!lixn87Q8+-oO9S6l&_J{Bt*;xOZ;6?#ZXb-
_xrVpu0SLz13)L9y$$KGi!xCf%-Od=8dme`OW9uN)fn0`@>(hXoox@Y)YtG_u^Dg709Kwh@^GTmV}>X6#8!a&DJqNegN-uZh*IFR3BSe_IqThgQij<tu7oWo
vWGNAw1(-V7#Bw;6}eBgdCfVp&2c~icw41rDoPf5k}TL@$*iga$DEebsXDmSBy80Tb#C6Dd@(-
>#JB%@;r+XJk9gAZ>TGj8#^|8tu0oMD%wLG_CT2)$VN+LIo6V=`>3)E~~H;CHc9Pype6jpC7?yq(g$13^Pg`TfgGz6R*0!o$P;#o1=6uQ-
*OZqZN5ep*c`PyFF3SdJfAzjMzyO<xD!OPN_<%9B{u<bbz~t!*nSJ(k>Ob&*<u5$%a3?pg+~;~d197WA>)M6W<tt(|kSGqD=GN(2va0bUB4etIj}p&Y?ys>0
|XkT+VvBGNj1ahsWQA?gISu~X2ttIyK+SKu0tS!d*pLBYDov)R}xKL9Bq%54~L(vm}p#B*aY+5?2pharcS<%1*^;mXvboL+UU9+VC0Dhewe5gj%^;|`cnsay
=eENmhfHCyh%4*?*n5AFFYLFSVm9$9s@gVE{~JauFGzKflq%Bi<x8~0kcYaP9aTMQd#&iU)0NA%2;XXND@9i392SUA`_DoW$A6f||2!%XPg)$e~gnH|SPr-
)iYx%K75v`*C8-
(+(r03*$IeH3%NbAq!*5BWo)vZ$}x2iDkmatcJ^N!Qu79#&YOl?KmThKNe@Hv8x%1At%F_AD=O)5zWuF|ED~cjQa=$fZg)?d*~q$aqg@>;(?~rt5R$R4Qjdh
#hTm@xACuaNS9~sGCe2k<#S4<2f%dMtdS^*GFD9rT(*k&q-
r?15Tl!&hNqg;~GQU(yaXpV`o{q991bRh^MKmcsyNPRj7r?uP_w?vCmWT*bJetl~9tBq8QI_VLvZCk@rJE=Y8ZjWM2#SR&B4@u}66}MsyfR?p!gcVIuJG_J2
14NXcQiDq{yOu#_9Gv3$mTy{tx=GK?HP-
8a(IJSa{nR(JII52N9E)zd4hQMwAxF2<KqWVs)_VIt6$Fn;viI8<lOG5TDi4o>g=pt^#B9L~;TIT0FqUf0?1Gp}6zfvILA{3gS!C~fL)r*bVz<$#}#-
ePMky0B$e(K%~A0N<;>licrG7B>$v1tnkH_wrBk92r*Uhy9<f-fe3!cq2dytB>v{y)5!!GvQn;;icpS$+U{e+Kf35$8eaGSmm0`dd!1IH0RwDYXEXDwt%-
i2V2xTnP=ccR@36SLa(A$?SDyNSz*(lRWxgo3_UbSKp$a7XC1R3uKqxJ{pN(pOrV5RHQ}>d`Ti)fY0#&Hazh&aIkHW?$Rjaut2Z&U;gCn|HV=&q!o04i!M}`
%aSKqb4r=uL8dL`gl3}6@=el!8r|KIHI}s;|1?91YjW*RV#&Y>>M-6VPqPr|c7t56y8cp~gCB0{{Hh$c|O|yCw>WAO+E(2BOUs?cE51<$KL%G3m-
G$9JO7+R-
P4P92P)~S5C@R|l+exk{6D=87uAm1BYN7o!lSnkyepSaBaRkRpfVV5&+}_{f^Zfg>k{Lm2&fs@W?I9x#n&k(93OGHRTOc5bEpNHmswx`gKPsN<Qh~JprJ)LL
%f<eVBXDpY!jXnKTkbCMLI=i7S(1H66YqI#+fzs!-c>Lyc&a;P8-X-
I)`Ke)I9$v?;Bfl_4KZhSkN(aFjFpfsQ?V}V$0XIL0Q(XWWoQbvbmpZ)z03bC!M_#|_@D^@wwKsw%NLDwF!gQh;hJooW_?4Z!&`w}+}&Tb;_&*EO|LqYvxsf
Qi4??5pHgPblo9g=IfM4s;pIX8m@a)H`FW_9pS5Oo@snflo)h_R<C0s^ay~K_C--YuHB6~iXlt-
i8vqDdG{f$`Ot$aR2n@BG>KFOEgC)$ESJ@1z#;xmSy)mi!Zz&5`Ja)7&fTLGvTQ(vseD6*!X`*tU3eB)9@3sdx@#{Exa#n>zseD##jg?(z_)zOY>2{FWZDVY
fFsq_(#GgIoQ;+FxbwsX4LVzgh#Z9+@5NK3(0-
LyRdZS$?HU&;0+pBO}VQxg3z*_X@Hg)6jvRHPe5R0IE@(*)Rh2Cwp@HUCuGwQCrFG_1M4@!kl>VG*pL@dvjZbrGInSX-
#O+w#(x!+m6tsmi^nYKXRi<BcumPSHl#EfIgkLnAKFGeW0IJU#O^+9~uXF%2C)rw&wZMOk5ZCGk5re$nu;rFZ2bkSR`Eo146Qxr|czT8rY15*Eql%p49J{~`
5qkhL%3pKE<As5Sa<ddh=q6w;_*K0sD1lWds-O>F8<79_1ZcG7;*5{_q#mmG=-
hge?#}ZaD6c^3gwHLlD!d{}bjR2F&t{i8(RF_55pY^gMpE9hDB^DNdn?O)gL}+u6)el)ouL{XziIbiU4NmsP3@P8bkkDZ=Fhtck-
p>nlEL1I?_|w7u4@Q@WcbzDMU%2btQw+<HOxmxUBb4ypLtL^GcWTJcj;Y=zqd3A)?}>Sy)qTQtIFr0~CLKw*&>ld|NhJewU>gg(@gblL<X^lwU&Y?$a6WNSw
L=|qtJVUZA%%Eyr8=yc{aqmdj-C5}ffnZ3Z6bKc33}EanD%}{frd#<W8q@POwPV>`GaS*yB<vLGO{GvkX4u2S-
4=sK~lSqsYeuBP+p|e0^_Q+StS3^xN|nl&8ccjM^$)q<}<G-
%P`B@d_33S{O>WKq!MzL{ucOM8n~@6{1u`yh$C4LbK#JUY**LT3L^CWw2_jhrHHhwo<Gv=efGe|VhsevR)-FrvpHM-
K3YP#t^`m6!sf{<T%Xu5@uU+);jZwhP;n^{+$&-kCj<{HxEUw~e*W^o6twH>btShdnRtfS-Mb41Ui_)ngrP{+Fnd(aO6nWCO9oROjJp!y50K&p26#ZC+l^*D
x?ipCze8=tHB@Mvi|-
MO?of0?=r_{dL9$DUwSzH3wAO{`!m|?`ICv6)S}V#wBVeWv_nfv*KKQ<MPcS6UBm5A>pEJeq2y@NOfMQ+gViafj1U+;^g_d`aJUC1*HI8|=d)HZ1w|8>Rvlu
gF^~A%aM4F`Oy7TJs=XY$?B=)IJSL{hu$>_mQY$lB%YqO$D;!*{JLqSUou$9;rnJoE>H~oojiIbw&8bFAOoISiBYMQjb0Sa)Jhhl>+zMjwwaZZ!A$dW_eho0
A{xkleic`2;k6;6n1#EP2-TcUJpi|7tx`lOHm-Gg|;OjREc4qo4mMlIuBb4QiST8OSy&7I?FSOW=O5G5Y<EF&jbmC3&W!N3GoQPmy9k_}Lav*uf>>s<<G+92
$6$d-4AjjbmriS)xDVh{@T8(G?NcE0bvGnBO>A7}IxekKxWbEA-7rW)XGi+uS4IoXPttkTJIiL@@|IP#%3$CkVa4t-
S*4|}el;Vh68nq^ig@UN4dt#URJz2^9vLDsiH0~g+}0D-)uWpvbd^XX(yS6_@KVITH|$puoWi9|9-
AU3u0_^XIhTj3@lq4c`wB*WuHL9Yu7C?h)90h6RcmPmZ?i;Z>fqcs1nD1S7gO1_TpFB~_cibWMDXPODp11d8|3xx;JRk0v0cZom<TWqp?s#IygCgFFZ=>ez#
EJ@z8K>EVh0kf5oFVr+3H=@_;#;i!I1Q^Jt65?IJi_E<Y2{LJ2o(HBf>no^m(E9G1jPbXGb08Y2jH!M0yk?R0-
}T!vbA_z<Iw|oR?vWI*F{qsk2ijp0UF+dt)OX|9P;>YMUtpeqLHznh^l@W!gq`npUZBM-Mn4qoc(B3KRJF4Uk|NhK4))9HTy{@RI9~aaLxP1n{*2)_G4JrD+
gDAwrzy7*MQ}@wE0j;SCpSx4hPZm8^^+>P`}O+1y|sa2PnrM|4}>Rr%~KujBm8|49ucz&vs3HXsVAjYCn+R?#GsXaE(LlAjLrPMAVgezRxMt;H89Xb78X551
em2&St7gFpWRQEgL{%k9&ABFebM_7mLg(5F&r(7@+<dKfYFG;#Nx|1!nn973|)fc$HGf=!#if+Q1Q^G{v4IGy3O51)Q{YGquLu<jkF-
t)uI=gkAvLA{D4ZUfU8;VuhsILbNu-1ODSgxCsfxlUemMFrICnG2wV<@iV@&vp|3zEEVjwCM7U!cJ`QDkyRXmaue68|v9*-~hX_JCnbQyN<-
@qmPCQ%CMXzWs?*GCa5-g)xN$Znn(-h+FBMZH=@9-}DE(tPfr)%{A#`v!}#8UyUU=Pb_O*NonW$mdfmqFQsA;b|(j-EfS!fA#W5PI{dS(Vq0H!l7_-
;`+Tf8S=PoFmaD9eUHUS6!n`yi>6^z3dRdz$L0)b3k<Oq^V2Kj{yCiHUY011I{U@^e$2}e5|*U^NS*vQ|)?QUM9rn5Mhus@k`Own}kGrXitg<%J}F8?Hzh9O
&*e4M-
7xMkqD$;C_sRWTZI7uq!2x5(>ivLw*34?2z`J;LZBJPm`B+3LTjnXg+M5KzQRRh>Rpp?+1h@RtP$rbHgEViP{mV?R5^8MYF%Qd2IsGLcpO!Mz6f^F5*`pvzn
`4pO|rG<9<7O*lG`Dm)B&L3bz)f?a2b#@?s`mNI?Obj3Flg~Ry&pUCAH119}Jst-
`zEF^#_8L470Ds_#c2Ny5GQ3d3t6K27oaq?C_!L<2G`sI+oS`Vr)B}sfW>4-
7}XI)?BMJ9VrdWX32wtx3?K|30qN3b+xI(E+DCGV7#WN^d_cxcv(;&@T6<XDk<MUAf<y02xYsUPOjC`f3~6Rg;YwsB7HHRA&sGYl<^lTQaz5%dNbP6#^(q%^
Jc=HTH&e&mq+RxX+NO{P}=8rn1#fZ@awZ(!>lf@dhIHG!Y@<5=(_gSQvh2JuhCIdL4Dx-jl>%x>&vV^;No^alQ5q%5zDf<`LZ11v)6&h@y3$MpJ3?utOS5sd
St4aWy|Vh`!YN!!64i6gL9v6XgZYI4FP{%0Fz`vQJ96V{1T$uGE#Yg32hsFx9u{?JMp`tsqjp%L*o}sY#tB;O~?RVf?ehHULsgkTPXZ?rN4FgH_MG3#+7AIm
|jx!$8tef8}`f&-rI3?wy0b5RR2k2YEx(7!aJ5o-t5DB95AOn9W!cGL(5P=ybTh4j;c=qc!+&niBC({g-L*`-3xQnW1M>-
^eX`UYeBPM=Ft+?`YZh?fkvpvHv74%aO1g=oHOWij8bXk)6VzvQs*FF`bRnRZ13}vIF`4y=rWQ&ZG`3wS{{XtKVYJ8iPsyj;>0TZf*!^}nR^`~KyA=fO|8E#
E&WB@4$zyci+cYsx5{sQPO}<@GT5bGR7g!PIpT!0!6t6u7h+Zml3{F8{5lJj5fs-{&RePrRZRo^wP2_)R*{><PnuvioYP0^dHSEL<~Bx$%08-
C)}d*I^3g&3&85XEBCbf}UQe9aB_ojs!C@U9JX}>0534OBRV>w)Sl>dqPgh2NZ!(8zol#Q}&lnnAGSCt|>Bq~A8C@ocFhLVz#&QSHa3CL)R~w3!;bFPIK+#M
1s-FyUSR_tf;DwtUT7ZqRuxS)O1Yv|uj9<n6K36pe=YI``BfJjxQPG##0*Vj~^v5B<U#vu%Dg&gKE2Z<nabc$Mcf_8i-
Jh3?`s~VMogE$Upy)~q_3)5CD?^U@z9LmCknyXbh=}pRh87^Dw0Ag8;j{)(#ut1tK0Nee8q1lErl>{l9TF%fD>9OKABNPsJL#6}*OBaUAO8VeoB$u7jt9k$q
|+jFB*mH-
6hj;CyScY7aUKeok43wM>GC^!gZ=kG71h!Uz1?{#+=;3+MNcwPB<0rH3Q|S**r|tv_9W;$XBIHfS=gyOS{L@s!nKx1&Nw)kgyGk;O_F4|TgjTfCi2R^gn$nd
87``E)5;5XG_cT#@o6tN^A&PA&NP0DM6h(~zppY@7!4vVc122=P1;>=B>{$MFH&i|P_pVxWagp=8tQ=>hn3!kvBawSp9g$A0Fb~H7Ru`W)~wR6ZtbyT^Z65I
ziMcCkqra<V_SkEiHB9*b-
T7zs_U+|YHB#q#hqeL1I&aYl=2@@)EJH}Do^5U0)it<(8}cEi&KXRpAbkyAbdS^xY=GFab4K15$ru93d$2Qku4QeLVc541_TCWb>F-
TNpE6|yf%QUpmLeYwiDebSJn3L^pBADDk!$gWjP`7(<PZ>O(l-cmJw+Yevv~N=OI>Ax*p8;JpJLFE7tLW-%psmSE|;bnv##-)e%woDG-
3l8VW5^wCUn{#_qrbPr3an`2*cD$X}Q7hdc<3h#F^ZDzq_W;%6`BKqCB1Q2`+Ov^-&7h+;2zX5#Y#GR7mJX77xOs+V;woq99|m(c--
*l<m%e>9VCJoRMyj)7^#(<xxeD|L-
(!&B5Cz9E3L16>I=?Q+QUX>T)+Ogf{SX<HWnl)GP1qMh))!me=4gUw#@k39!v5&oWMi%n2)c?}MI;nu~aEwC2I8hJ(K8vx@tjwr4zg<XBAe3{UnUIoD*@I|(
W|8fuPFl*`(`%GGn;5TjL0+8;_Z6FSxs(d%z|Jk)Px_T}iyUMp&zi$TO{4epSW_xuBw3)_??guFz=t&6UDnv4n-
=#uj+g|G#fV7};3CRrfb5Z|Rz2@EA_P!LDWyPaHxv+4`(=`E2DLjJz1ENh?q27ZQAo2Vw^~sS=?!o-
vUir`V5o<XmBc_N3Ff5C&Ni2NB4M|8mgrq0U>7aSACIL9BG}DWejBr>7o#sht{(HNaqU2n;$BhqU#={M>q^8^>^tTa%>e(f=wVC}&*X9CY{$(I%Mku{CAr0A
DW$k|w!|+IG`Y=@MBCc`JD~>TOmd0QqVeGmPmAM=Y$F|JYM^rW`^xrkrTaf-BOLi$Zyr2XS>q<O5%ztN=j))jg&Io`E?0#66!E49bQTcIo0M3cpW}|hAdHle
b_*g4iIIIAU4aAbkTMKo&pqy@VOj|JqT$j3`b`ET>omM??tO3B&&;V^?u_rOnfLj1vWFJNQ8hGbT&6qhsat0vH$x-
TC*)0;y)&Wmm@AWt3$PZBFl3;?00ZtmGQ;`V_ix(K?JR<Soe&Qf}!dtT?c5|`1ZPWN{1I)4BnR<^~yN47j%E{xxI{&Z*Ed4<$tF13YJE}yRgJO<m>59yHOA`
|7{RO!>>N_25PN=35R(mDW?EZa7^yD@X@i>x8i;pH#!Ku$M%z8e>7Gw&|zV-LNH<C=LDAe($A#twaU~Yl{siUy|wTv=TT?Z@~MihH_OkynZ_xakm69`C55*$
2w_U|N(%UJAL@<s`y8~~cpk|74SQTG}<eQ`q|0;@b<(Xb#z4eBA8o*zqR2h)s@@vv}G8kARVQF+)^ozFe*2K6(!PAPA6-
_l~Rm}<a`BJaqNl}fTJTJ>A;R|Bb=Q_@=X1%l714Tey_+d6WY`tdjVlqd5*<);}@x<W5x_9dtcsHpJc`lMc2tQzl?7{e$i>khWlI1Fu)Cpc%~BhT{F1y&9M8
vs=p;kPu;<+zD>{i~!^&m$7S6<}t~WgM~LN=pG&Ct4*<A$cvx{eW+(*Sm8q<-
;;R)5fm<Sp+ud+iXyW5<)hyuxV$*9nyrPIbygXCTuzPde0^w0ZM8ULa>q&x3=nLyb2-
XeLSXaM2}1VjYI!N7ye(eDl8gD8HatoyeRZK{%PSrR3~EITxy3xGN>djcc6*6Otz~GQk-nbb<8Sm9#p1J83a5fp`!kCD*Lk;QK1lV*V(`4FK6%#l_Lvn@MF=
RINx+mC`8AaUeD|vZ`P5%S0#QcfIJaQH??`^(DGS1Inpw{Qnyx#>7u_f6Kgew=SU`?{HX%Q4m!Ei>>E5r`#HCB{s<@++P3^gKLoj>vH)@LND&SLftWyR<VsL
nihzd=_;GDHF=mWeDTo!~cXh^i;6#cldEm91{gv*`q~^EDG=ku=7<XF#G=m=UhNu>e?u&<60yy0pWx-I-nw%!;m53sVGdAJY{68razieZLw8+7%(?PKQ+iR*
+vPuB3ptphA5^)W(2e>;sF62wzWCNS@3BOvWCgz3>t=dZB9l_<(0AZ}zD<knKLEA)e_w1pMTwYfsDHHE__28%%o|qof!8;3;`P&v0AL^~qsb$<1i}E@35>&%
el>?#CnOFoA#R53DLobK!9UlY^?32UM{$WYRc=8Pq<T^4ef4W7Ev^v!UP4}60O%n)a`bYQT_FX_r!F!VtKUM_!b2C|9wt+?7ZV%h}vM%3$@7#cStZgSGS)aX
#Q&hS)M;gQ^clBP*Uqqw>Bgi5mkiQ0wm(MU1EvAhoD`)jd<6j<U{Tbgt!wu!jPNml*W%O+2HnR3j3Gwu1HaP0r;9y=J$i!t{teTa3183RGLx{l+@Zy2XrwE;
4aXkH*)Dn|u(4Y^W1Zyb<yKvyKPI#y-
dG~Y(Uur{kA6doLGHL3z(&Y7rdTM|ra7{m)l)w<nS$|Rp7qb(WhexUhunHiqu$&<xBq+DYAZG`QmLfLvWH;2Sb@j_W;TQIbGhslmt+h>**iI&3EEEkH4rQy)
aV+F)vuCrY&8pWY7gP2VMSFlB^Pm3SsG^LvB@6lz_qEEqpk!CfoLao_sJ}x<L|sM}B{)!ahMIWfILjkm!FItYNzN!?R#Eg-W;B`I*1-
fdt!K|a>m{Jyi%tE5PPBJ3HhoL21M;07*J33%kq*wyze6AnCh?l3%s^L`Ll{%m4mPU#pG_Qxa%m-7#wL1Ae>EptSVZNw8`1cw>M!Ts%(-L8{gOU7JX$b-
O`sH49r=@!T-7c=@zFR1!W(zc_%N*W-
=VOS4<73JV=!3MzJ6vymJ38gD0=;_zS)FVzCzqDM5&fWX(6T?pylYE9G+waFuQkec?B>*jX`G9$IAteX{&8S^~M{^_U-P@c*S-
c<E%v?Hxdt!F?XU>J4NdN+EdgEPaNJnM2{4k&O{K+VQ+fCf|MVn^)dCPm0}P!uG{-pO`Tszvf5DP-@e9lwzvqe#FI9xj+?R&BO?SjVr)ttB3<W-
Qjz+nU7MQ%`@Tj-oaKz?V-6&N(-
13}Na)fv<g7lRGa6fkW9XQgiWclBf5Uj5nPIEd{H0_#ZKb%z`^G(mvV4Xx_#!)$GDlAT%$R}Q|LBCBwp(OmWo|hKQgRAYay-_%dFYk;hi{-
Sn`gJ6CqF^+LxlP1c}wI{9=LhnR-fqKO((OqSY>EmC~bSMMFKom5r?xvE7F>W_OeRn;~>5s|1{Fn(wHOdrf})TZh?~SY;Y$_&FOud-O-Te`XRC`T{Wi*jELX
--WWf9Xe#P2(_$VgAeG-WsBE)YTA0#@On_i3z3fLu%%OG%&8@14%&$i01Bu&Za(#RSSf1VG>o;A#auL7Z?sWAgAzXYq_n7A`Zh`P`P(B5k&62k82gGzjbsNS
s?N3&t3=~a$HmU6GA+HW!h59f;>7k!q{8PFvc1>CwJT{5JJcHrObWus_v9?wUuN4l#*yZwz;<@!GV}CLaCg_<yquD9`72xJSgwF{rR&?_2tG&+cZgQ(&M~GY
_a3SHw1R(4)rTvaKdB5tLV&i3#_*cptRxzlz%Me3f%VH}Q{HaEC>FGDdhdM8h5~rmxWFUZL4$d?-
etwqAP49npNrvD<dIf^*5rb#mFyJaNB#?D25dV^@p!yYzm28Pgb?Tr60qO}Y>EK#gmG2G4f#9hE?7*$oX?sWrg&wK2@DnAMZREnJs2?Rmf^o>bt#BK3Qdw1j
On@H`>4-NTIf7e<0><_K%!HZ`ui=m>4jU{u73huEM$u-T#9R4Up+GIe@v9!!?mc-
@1I90<Qb2?l1B4y<tqZWi_?<JpI`mOSK|5WO6r`WOwif+m<S(3C*uX~)lwgJLkQ?H5d(=Tf+fR`B?~8nviDJb_figK{X$YBjJFQ|DyUr%1r39I=v)w?ejeKZ
p)sD{HqLkH4zB#@3377nu0y|UDB=@%R1D5K-SUXsS>zl`AF=zgg3`c0|dKH5Rs0Svi(NOVN{jS-Xl*@Igikr}xftvRqp3}ZovrwniX$OEKagnCyln1wIoPH&
YY&gWxrX==^!4~&e3?{39hEVNGVW&WboXmIsb0p3c|0N_<4y~08@>XYb{hJF|nDlS;tay9Xfmhy0Wa@fcCZF53eXzE|iutJcXYxCQ94CU*xTe#MySb?$$x<x
i|0sFh>T1_1sIA7372}fZkGu(dWLF)E+4a54ka~xYp9N*iI5XsUFOFd|>LkjW6bgdw5of^Z0r>TPch#ZD?^(zNC;PSw^%@UP*;U&>+UfS%Ez>yG*L9Tow=3F
D^9DqeH;7d2nl}x2I>h_OI~^GYw_QyVt5*D+73!l=bu9|j4v<@GSr&^HWzofRnzESC)lm7is3bizTaK}Aal+XmYhH`YTlR5Liqc?2Qrdv<FXJ;|5U4;a?}P`
wU{IIobvl5dR3l1`JI|XKH^!rvCMj$odN=ja@&z{c3<|b0iwalQLSEL><+J}op&&!`hb(1>ti%gFnaFo;P2b#(h_vC^EX?ymog_l3N3>)yfovLwZ2F17_%4o
IdIVf1<}<;$d&JI@ZHElc*^m&*9HS?7wi>i4?<4O43!HR^Nck{nJH3^f@@{l_K4QQ$hfFzeR&~FA=HO|yAdAH1nXF}O^6Gfl?V+?Iofq&A&f`mEJum8t&8;B
*8hj}U+M3HecLFLsWjmAUy;HUQ0T+&fck4L!;0Oxcqo%sttW;1pWmN9S#~9IVUa8q=MEW!>=>_ROlbYTtGbFsF!+N#DuScHN)y_}AU~|1F@@$5Fxbb9jS#aN
W1X%Z7I*TD3>+BMGffb6v6XsxA)%vMDR0vI2T1MP1%&Tsv&V15s{BrHN4sPR#s31su{Tuu~6`xjcC659>J1XuVz$f`(KG1w~Wx&SBC~Z~dwDTv$K^reWSOd^
sn%fS%CiCcC{Ie<wpM8)7B>?jqUrAymvtx4tr_f0enSA!9$&zz!Y6Hs<@W@R*H>isPnH;{=5ms;xu(;wBRBI>)&jXa(0oB++zF3cSkt4~Kz|rZ}O{tR2ZBn&
*Bp0__jx8~^0r_$B3{^q_IpjaQ{jNa^cCvOgLt&f?tg2FNTYYbhr<y`H<1dW#O&hjiRN;W{haIX@VoGJKE6(Ww+i;!-
)UGe@X4rRzLVTQVSa!Tf%g8K8F%A)4OE2@Zxv_2<kf=#?gPI=ff8>o~KsHsx-qAZFJCfh>pBYKW-
4?wSWQkE&9KayG&S{6Gw=;iU5w8zKGNBf(P(Fi?;ZD|l%l%%vzWkGE>#cVzF>(1?7(63DPRFW(;811V*as+|m5i@pme7@qZ8Z8GiqHo_nHT%>>b(HyI52Kc6
MO8eYxQR{CZ!|6e0UhKi*xSVkr?34(OLRLN71R<BLTaURCdIY1IPRcKZ32IYH<E5c7Z9#U%q}o6SWt`-{}+lNXU-hyUl9Xqr{A(P0%1cp=uW}F4NdIEYu^?A
%}ur*tow7haf|=v^mjMe*pVo8<Rfcpufa>VPyR{8{D=^iI3x(nM9B1hP?uxO2Rx+)Dy-{c+(WUI>hWr+8=lh!(=<zP5TsM4M4Ef=x|uZu|-)QkHQ-unJGe8n
+sojc|&C(7yU}fjLQGx3q_j>`@Y3)o?<aG8_-Jr#h<%7u%TC!h`8UJR2GBX%i_t;5tF=(|1(`nfJSa;Obr?Jhp>=YiHe!ytOU+-K-
70(hV9AxNO!#MZ8Id4u3Bo>dR{gq;0$Bl-NpdyF7Ku{F!swg3sIh)kvu+&d0k^+u}FzAS%Si^09?DOck;=k5T{tW%y25kXQ?29hbt3cz@10jWEl)*pQ3bF4@
vy5JI7EUoaFFct_BKI!Kwb+#uh?o>jgb{1jo^d63UW-{i3wE>EXLL9g~Iq5BF*mBPh%4$36KjVtlq?EP_0z$&-
K)Jr!?gWIV(_4QHocBvf9Uwy}PWHIzl0Vs&tPWeEoVFLt)ShiJYf1Ku15LW+*G=`z*UT)Pm&9rRq4m4NHjL?H#DQM=KAd6^<>0YekpG?lVVz2i)yddAYk`T_
#S&RrR;TSTw+^@>-OAzy#E*bm@@;ykD5w?Zu(Ib<>WqI1K)D)b?BwEjvu5<8EWsYR&Znm<@M$eeT6MP<iO^S`6CO7m^5Rw_eaapxNi>;!#P_*a9v-
BIV7#G`YuaaR|DosYPX<;on;G1e)sFJe#cq3>2Us)Ut*#9mZ{6_(;gs*E%C=*Fy_o7xGSxl>Ae1Mr=Am*rp3YZErlD8QmWYX$3=^3I6NsAAAkxgoxJ>CRqUZ
TqYpf||jqJ=pW6H5oBXV#lqMe_118`v@K+J5Iu!DM*&CEsBq+C4$uko#hlu2F;P=RM3a^C>e1hOnqFQpKj=+5~Q}W)H?Ww>ay^2Xhd&Hz(>KEN@bQtxP%-
JY%}?#QVcp6=zjN_HEWCE7U$0KW#hK1%KS_%6Q_tXv-ZB}1W;xFOLVuYN)~CKI$kQ)JKHe>G_m|?q2NytD}8q3x%JV_P^!W&NBBj3N_TuOrYAGkU;m~^hxQE
^Pb|-FkM9<EQJ4%vZVx<TmA>jIspvq%&|)mmpfFy{kJd;Inx*WCDs1BTc)SVA-IP@U20Y7{L}w%d!I&b^m@yq*i4i6p)OvCh9;A<z;2}yo$XN@aD>!BHP_@k
{XXk0&D&7Azy6UrWp%dG{4Eq5dWw|h7aluD<@|8EQo{6RR*poqqBGN^2_nHro9u-|8DpB)DwlY``wKb{SDsgy=kzs7x!^~QjdI0P<m)%a-
rM?0dMA&RZ*OA^{1_hgNMHC7e2a`+(L}aHx=c{qW#aM!x#{|9m*ZOIAQ!oUYO~j%M`^=Y1siO#)Pb!hQRh(5lGlowuLLZ)4qU`d`?LgeKd1;)>($c!9kzg=N
%-yqU9&Ohxs<{S&mW^lPGf`2S;fNwRE>nwzL`BlJ_n92E8@(~35X|vJu8k6|c2EK_XQ&;{DF_k|O+-%}YQkC=N%?%3np9xj*PZLI-yAwd#K)UaKF;H_Ab?Ud
tWeY^Yv^`3zmY%co_%Ev{b-@3A{3+!J1UYbz~)}`Y$O!#QL8;`K)O3k>3r#{j;#}#xlTaqfyil+Fjm^EWOEZ&+L$EB>`vq^HPiB9-
HsLmgY)fy<D>LV?&CoOt?il4-$7#L($Yt4!&+Y7a>_#w0x{29M!sf3f6oYE|F?oJ=FVnt>?`CXNwOdssNSROMgUnx3XbLs)3<BIcC5MoER4ksc<AV-
V*To}6q|84ZfAiScj&sFrL@-
)3(YG{ZKiIqY#xAQAwPYE<pC!fr=;7Cg$KOjH5B0z6PEdb0LpRef#B?_D^RT==<sJ7b(xvSj1n1YiI#%dAn5*B>fjLNI2?<}kaa{5Os6m`=XFKZxU5Micmz5
`_jkDv__KYyGqfi}sKE)I=OtpbfY#D&z1)J{8kWTGXE5WQ&Ylk1MgUHgAJeD{#&(o6Fqn(Mu)sWmivyg#u4CnS>luvf6~l|2q==nipC*5lBHH*0<|SmjdR1Y
?(O)C+$K#PsMM50A_)hS2XYD7AU*lKP-x(pFzxi3;oBW`iF2{m9Qz^EGD>$T|cVQ?N0H>m3E#GGFR21v$Z3E~0=10a~KM>VRZJ@zvLE#Vr1RA0T>FX>=-
+cHc%xbM`{O#o26H}Y2zRtw^+qfk64YrH|yi3I!eh;-C7;wj$du#lM>#TtXW-qon^uyja!VLJ6uvxKV2x2G(!zOSL9@K8CQ|S*HR=M<ko4xILb0tfoMOT@Xs
|$+wGYi|>xxLXvZslEJS<Rrbwt_KCdHHW3ZANkR#>S@t<YE!oonCZc3iDomxXbvUJ5_e|MMUl{pq|*@azW-
ai4cAKp|h-|SthP*#(;HIc2VrnA;rwx35++}kN-dNE5$mk^igq{c$*z1?;Cj@hwfk^VU{*zSXkLyDOntd=LA1F9@N-
mUIu*?hiFsc$tf+ZkT{0ELNZq(te!M{tlKmaKSD1gqaU3rM+ff?5edW|U7atu9-yUjh%#;;qBRp+-
9mEThsHEWo3GR+7O|y2vUqq*Oa~xJea}djC<$4_85QeevT`!)D`ivPC=r(gZv<mY0k=J~pmg0AX;M9LWKfni?OziQX6ag~1}ia>KB$1OnqXSnBHvRolXM@)E
d-f<EPe%iY+U|)M)2<8^Fiw827Eb@(cG}!pn7fQM3>&npfN-yT<iF>);D@p(tMI=iKLt4qoBdybyS4!j>U|O5_=nl1|ysVXPbntJ<GBn{#B=${&rg)ba2~e)
SD+!kUzXuaHN)bktFpoJb@h9pU|fovLEF#T>Hv4T#>aCBZ4!EANgTOSjIDWcH#K^>L`0mBCY=A-uA`H1drx1_s#OspuimkCmPZ0J3-iliF-fVSL-
`O@kV*H+isOJD|3PofgpR8J?Vx6z^!d!6r1J%dj3pMCJdlmn43@5gW5r@R>qs!?m6ZBeun|Mb9$FGIVATt)5gm4Hdq*#pj=-
TF}Cg+a`NL9?dA`}Kc)U_unF7$*BxrJWf|x@38=+6I~@uij}H%zssShkDF$xrsD`|DVzqxe4}y_N%F5m$S(nUKFgtGT;3&R#um$Y60MI=@N(O&Ox759-
D<xd@b9@k}-_I+9uVG17PKZ-haYP-c1doY(`v#BWk)d90A5l&A#1j-Mr1laelWAid7%W07b}<b-8~gRhOMA1p3=a-xA9zshOB8IlK>H{CGz?<-
i{6Vyv1kR=XTutn0_3gaXaBkAX@zyZl%e6?aG#pM;gnmYl-FeumV5}1SMF9op3;a5mI7N`629jmDAg&;s+ZO8s)~os{3gLpmrtwV*j{t1c7Nz2X9gHO)f1H{
@@^NfQ*tZEE6Vy^D0H_rGj7(P>$JyKhQ?ck1V^s5Vn=OCFOGsb7#SWLEgIhtl1ckw|3!<GZBz|fH72)|SMl_o2-
q<r2^y4i2O@u;U}A%H<sT$O+;<b8#M_7p93Di0$5uxImE4b*;-
*aDmF0$%DAH`d%{bui4nz&MnMT*c9ib<M>LwMwu8LPWE^EAvB9nwP>t*pRTgeJ(rk45=o@M72YOxdem>32LTygiH2`YOK2VwOPU7@H^JUKi+IF(gx;DywcBU
KZ+`S)xDaw1pfQiyxg>>w{!A6qhy<k)gm>*)4jGJMFwCG<|K$(5`>>toyA(_VqKSOpapdmy*t#RoD!>ZFnIzu4smgNm}5RSTo55GBNSz^In<`htp*w8wFdeT
(1%ttH3JBz^(0x*yddA*ts2kq{~l3iioxq5AHPP3At=^arOnY7JGJ7u8TtHtd5q5VMk+{9U-
;A=5=hGERm1^VEQU+X96rwtkG0CQU<^C$*H24OgUHmf3Gd`)c;Z^t$xDv`ki!uLBMvh>w_>R2X-@NStC_^CEVmf(#pi+aoIqrP{z^`~T4v-
W}i1FVg?bD_?aMcoFT^XYnyd>tA~C^oD|PjK3_M+#Hfd2`ig|*QY$^4&QSFXJveCi;5~7p&(<pn*ywu?0QjYbAhq^(KqkRV-;eF2pp&0|II-jZo2_d4l0qT*
DvY@7Im4zMjSe@=RD^8bIIBFnZ{D}S{fL+S<o&Ck3sV@_9;+#n#)jzNRQ5{;JK-=VC{BO1JuG6sIHrbAEj?z>gjHukpzP%|6g#o`WmQD8P-
obt1P2n2%A*H=SI)e9~ETy)L&Y2@u7>yf5)7i6IVnMJ^#Yxp5S6^Cx{5A40L*w&~!&^ktE2}LI|Xb1qISVMQI4XN^NE6bVFSm1@uq~#!Pzupujd~V-
OEppT#dqXB+$HD@Q;M88~9&BWgKUy}`mYcMopc{0ES)?ql6r(#8AJ+5*}&Do110Sm3&2aDH3c&m@S06CW{BH2c``680D>D3+qVDXp_!^*}4hB%OX(1r?mNzs
jHH6D0q2OPA7DhpPm)KoZGx#M=R;5pC2ZQ2?0-
)3T{SgE^9zdwTwahZ;J11y1X5Ah5ByK8J_XYAjzAW+{3qbWFC|#_L=B%B44_V{{!1P@GE$>P8EzVTg6|RH#a6q(fLM(0Q7G@8UJ?abQX~sMbp?lO_+D+AmKb
Ekwj|Sc?&X1o1NT%r8;?R+t@i2j!zAyj$<+2PFPQ#Bc{>OW5CxjSD3MiE>lN2RgnA@fk^G7_DwS$<Od2nO@v=czE+~<0vU8rcaI>h%{D;GMDrl<2aKrP-udg
YFyJW2RXAx_IFYf(+d-5rnmgDu1GUChib_H?{4F+mXKFK*I>u}h@))c3+_q@toW}^-x9<og|q?`wjvNuK8b22SIf(=1c?uPiNNedwms`&pMd15-
2ad(c)gtTfr)Zz)@dnu%)%MlRF;PaD57APG}3uA4sZ$QHIYn>=fb-jwKsG<mUs6}1Ym9Aw^$zlNf6Yp2b7?w(S#_AhNxL0<XzK&0FDxb$iPZ^%m@Qv6i>!v&
{3>k%iQxc+W+TS6Mo^i##Dm2OS0-%MM};kFr?g4icE1G;Fq^0Q*%0hADFFgta^romj>*0KfBCgK4eh2DxnLJiHd{USnm>D8qxFdF^0e0QB9CE5(B|DYD5-
jX%kjQVO=^?!v~>tkpNH=J7M8r;oqb?yo(t(c)=12XwB&^Xmg|WZhxw(yXuHi;sh5WLgqs-
X$}q3g1Z*PVaofSA^Q(H;+UeLi@wy+xbT=&b*jm_`bFofCj^x#vGVs5&qIDPVd?r%hOCpUSU2mry_HM)T0E515#XfTv%G7|v70m&LAOL5a{6gI1fY^sGl`?Z
$+@g)rfTzum_KPJ;0CwEKj2~rRhSF5q%v??@(r$ibt4iU<1^ma0{27~lK=HbyW#_)b4Zr1K9iJ*%3y2}#6bm_WZqmxD9hacPm8j~*frqt4+SCU)D^dHeP+{8
6I}grj#0`bQ5El$!ELkDsaShjB<!BaY}Vcb=+@@8jcKl|yj%Z~Z&F+-O+TmgK4ZtFC;mYuvLx(-H75-z@BE_leI0)KBgzrIr@|y-
cX4vq<?x|+L%|)|N})+ag=YBucAEn9u44gnKXY@@!1s}DMJI&MQ)96(<FGT)*~jQnbHQANjK|6=W!rz!CtsIJzUa|I#+u$3jR<n3LIVUx?Yk!OTl>S?-
uS8@%weE4%3mPr46mB0=l6~A4Yh1F^vtN*P9@mr$4!O6CeKYjvu}^)hyiW=+~ngs-rykv&Cr$ib3~BB=|)N<B(r~ZiA~e=H1y<j^FqQs65n&51FD!+R;{+DI
(F8c5vz+P<KfKtCtd$cJS~Ow{bJ`QQi|VUK2AvyuoT)B+%MNqR|6HCbW$9kOqh6X^!>j!aABBqCIkax>AZoWeTD3IyD%-hJ-dpdN@-e-rwH-
5|IF6;E7teeK+oU$kqrT<rO3a^j=mv3*$IGI^f-Xssq~?rGq|9k0OP4a4(oyp08?hNm7&g3AL{?W02MtZJ%QN&=(ZJU`P-
i&cjwo(I`i94cn^BcG<D<3S7Rg2oCc#HZc<hYY*G2YxWG5-
YX4ZTYGIF=z#Qkg_ld~)SbRcRa#xGUESR>1b{Sp25P36X4T@ECth8bXuQ;(F`KaYRt6oyVo{f^p4VG1Yk`k4pjAypfn>zs3UcchON>~!E|8M4X-
*iD>r6qd47gi2Wgmzz(-nOTUlKb;D@bdO5CwD%CYF@5oae0c!XMbEc`eFP-
#DjV5q&#25E06Q%)Rz40KU?>j;{9LigRhO}Q(r+~nTmC8=uFmruEj+|)ljqG`1$V2A(8|7Pv78TLihf=qg?At$vs5-6z5M#{^KFAEXU1mnG-
fk^sI^*#97~BCQn=n0-mF;CaYSs`w1su&JU}J9eveKDLOL@8AQqibdb-YZeOFzaCSZy&78zbFyZcYAwpcWXDK0-
zoKA>9_!oJsT3GrjbzBb1^c^x5fo{=>L&KOsNIof7*;?rg|@lp9d(cI&RI%9_x@{Bps;mDa0!!Ct&>9W^bT`0zJj_!<%xKjg!s<0siV3rl18t>lm<|nh6$6+
nQ;lmy5A1!3IH<ik_-j_U6~TocV<k+X)ob<py1{87fyq*;1zUPR<qVZvlR$J;X$}-^tQ<)%4hkz5n>vJm&PG|&0cW?B;n5>;BX`i`9nUt&k?9b_>wdB-
RmJ7Xe?CKZU)K{RthvgBGd>GUz?KXTFH_4Z!S$vL5Pq82Pytm=iVgd-xDlu4>9Wb^2jL?>p)}N>89LU{^X*A&d971fa{}DYM1DV5!SNjFmn6f+-
{Ieda&|)`Fy&8H1ZUszqK|ci0LM#qrbsO|6n-
z?{TdUgBf3&kcV?c;!?lw!YzOSX~w^F^8wKeKVdNJ<wTT#jAI=LzYj!F*=Us}78<?1IU*M(M|enzY$V?z@0UX=#iDHrH52V@x?j>W8Uw%E-
_YIw;9f$P(Xqa8J}!fyMMHoeT|QbZQx#^BHwIN%t=Q?MhN}KJ2U}v~A#?+iDJR?wt-
BI7rn0Z}<UvEv%Oq*ADTXksrpFH7i}8++)TgG;3kIQmK_FKeq^f@_H-MP<lwVq<aa4B}*AUax;&2|-
LY@yZv+%qedPB|1=6cEKu;=@Bn3)nz0cW_f?`9n<e?-
IQWd?HZl!v!tMM&5t*w?Xv<}c&Z9mS(QxCp3e+;hsUh#$VPgD>hcZwGP_UWsaVjvnDEV1VQ4@Cf79JJjlAh0o)mK23Sy!U?Y}8>{q|redOaQdYu!i-
=r8xdT)uVgAYAw_}ngxf7vDH@Ao+ekv>d#<|Punx!wAkN)W?DEx$(KbFee{aOc7E^eV4V9GUaE1b3|l2+&`a7yW+49VW$P%{y>Ekb4)yl)a#UrNOpc-
p25!z&C(LSZ(dJm-iN-
t>s>42uTqak62_ygxwiWxZN1&8&#I;|)<DLcSnmej?2cTjae9H%0hyEQl|N0xNuWvT!3Ltsg*oS0O()>VryBgYz+6etpsgw(V)SzZgZ+gAxL6@iglqSeNU^_
oMWlfILhp@y;LaWQ@!zu1zgwV7rT+?3Pf^aPK+6c}3{Uq{?Re5Qu*tOy-
$VyJj6rKEQHw>b?~yiUxpHG0UR?zihSm&$7d;0S<MSl%4{)h{LrvUB6v%FI*!bHXB??v4|+_w(o&zov-
`fiORY~M5$a7XGJ_{6dAn!tcI3)L8}POx$Rk{h)qSj|4KUpZp-
o=3`jkJrXzBjnWGfbu0iWA#TW=)C_38TuhY1L7oZHJkR(h&CWw?48TV(9?f#hMT=6kc_$uaYI!#8MTN&amrrL?heBt^#MxHLQ`d^`T?6$PlEWOtzUP#R&%uY
-&(U0Cb-=?&NGTl5JQ3J|cz1OCtD+hY}#D?J(Q!WOYT$<)mf@V@i^#|tl>?*r_;^x=yL2O_5sF0Ik!iG-X_t1bq#w@9}>vFi0QXC{iY2bPXjij$v&#F^e|H5
+M=FtzpF@^=cnhV{oLD3D$1Im?(1RVpRU~$-
NL|l(2%dD!6a49GAvn<NII@j6<q%)5FLmN$KVq2do1dkx1WMf`^GBu{Q4`cqDoIW2wZbG{E^k9Usv>vyB8^U#?uT4oU8M>Ux9^=gY-jvmyyh*)mfT{GwK8a;
{VooP`rJ2%#*9%??8WgF1*A)PB-2nZmM@1ACn<&37zy5ie;bP3eez|(3g@w|-acc#oi|YF@0I5IZI7(8sRpmm?{SixV!)5{@4#@TPP@j%&xqaPaC-
^IA``5}4vSvJ$mS4~+El3P7t06BorSDeHoMr*gUk-qp1p3$T$23q9I^VbZz^|F9K2<Az1qg%xBNT0v>F!4TOSh%e{YqU7r>w2TJrI*UMPdUf(O0}H@y%+VQ&
wp+)~93-
{%nnAOhFdz!H+%^uVsdylb|g*ftIObh9QhIlf(|1c5oVT=Ui+>1%CdezIqwx8C4Fke{%Y%AL1OqP#sNJuUqO+F8I*(63F}bj+XV<=FwTG%`L3VL)pqei8@Ls
l#<BCUH%4&45f!273}_0V&j_NS20zmvEkWop#W%#8Y}*iOp`1k<%&FLghX<Dt|*1o|J1s&OxC>+%+C}s8sQ(goC<p1)V{ey=s5M)F8!MvgEl}N39Ks`Lsr}U
%W`KVO3~G^&i1FECQZ=xs=}JsC^kqTdajenH-=e4^$maBbPmnmxpgR;|32^|kN{vvCP<+e%-
jjP7TcTWi~zVH^a6*2_Zr1wp9Yk!XynSiFESbK03R`{IMupNxt-
&^kp&L1zEoYiRO;A@${wDe@wI4X+HTWCd8@@llPx!ECYU#%GBOW}dbRF_%T^zvkif(x9po|EZ(gNQ-
)AxMHt2F`UncG&(7Xo*I**P!AgLpaYQU4SqUaO!mvI6UQad@_FAHnc`tj%i9<;RV&Z>Z=LM<WMg$oA+(=I_Q*K+?*V!8<)z@tVAy$(DAx|ZdC)LEq!GOHWmf
8Gd2s7<6o7rm!8UfTsC)`D6u(}dA2xfRHS9Z94RavK}Sdq>=ZfV@NQ8*X4xe!T$kicgH|HUqK{`<IYd9(hZ1II#LXJcB&_uS}fH+Mo`=-
i%iUG<=}9Z_a!dp@S>)ixU!<EscBCdnNjZ_+4?&Q3+dg3n7>O=S;F^3YQ@}p?_<5iJCE}aH?b4!NI2_GCmN_b1#n_v9%Z(ZC5losAR!I0azose=aqk)LN+>U
663bP8ACkz=?XuXngW9<ihP~j3~<h*~Qyy*o7C<0)4N3aoV;)6<><i0qUU|x*+)4PjWje2h+JZxbAajLCaTW!^O0JpNSd?{5^j@Ys;wndlPoL4iFxt)n?HH`
xi?4Fezw%rm?n9K)U~_G-APKk7tU_T#&rAy&NL)E?Ci&Y34_p7Hq}JwiNnt8RvaTn(1-
80jU?Aw!3^kgINFdaEC^38Mb{SXLR+qtAI4${u?#ZJ;<!6Db+spgHStWNx(mfi4nJzMT<u9bX)5k7eX4|E+k;6JzFBoOSgS7ZqHS2k&&$4ID}>MZ+NKgtvJ_
Ptyhqfn7&1QQQkQQ^xk?L(ym;=)eoi5%|G3{oCuy1H}6)P>D!)4Roae6px(a)KI>ZV$_qWdm<;4An4S)g3g?75?jBxR1YEm*v{~O&qzfBxt$8zx>8Xuc0hzB
gl9M5eRvfCc&^*4Ig67D3r?}oSyh`F$;`(4in|@}O|Ha@oGh6Rayky`fyO(CL?gC_Z%KBej=cyby+~8q`50U%dqxQy+U#qI`Oepp#eg0~Fpl|C|_lk9qj<~2
ph6;Lo;-
7Qwmi6@_m1mQ>p^q@`d%QuM#!k8fN<B+tjnH8!E5^@pB$1Mj5oG6(0cA7WiadiZU6#R&_uxZVEo5VZTm(fLHi(;+1@$*Nz$=VaM?*MLTwl)%?RbK8_3>oZQH
?j1!7!3!gRz3u^owj<w#>q;44<Z3@-J}Bu04#lKucqH&U_Ubv1+$FbGe_GWi>11L@bRj+(D3@UO8^8OKiQ<*!6R|>$Ry!*Q*8?zTLY+=%G-
D!u^4Fex>a8>IFU4DU}O7LT4eHz#Ef|a?9tK%lS8Tawdh4<K#__uA?LgFG2LFYdt!<cR5;4Nu!0YsXd(=lj^2zPb}kz|L=Z{*-
7L=zi%(uRA)4E2nPLSd0W=j&I6V6C#9m69^UFuRXZyY^rQ5j-
iw87TTN51?+M!!;SrCOjtC8$IiC&RGMgBoz$8J(szDygEit2I6~|oX><Tq?pMubdiC($bYRG!dkERg$lL;A?pxGE1NJ#U!Yk998p!zF$`UoFVH$UKlWzQ5~m
q!NlHIQOHu&ky!1*rCGnxp+HCvMnZFQsogUg3lt3Q(HXaNx9ru@2S#u5DI0xu0Ev-t!`P&jQ9>eewt~EO@|_qIx%wvLPlQ-
9I8_81oI;B=iA2GTjwEqHCDd*QHvC?aTU)a-
Tg<`_^`N#p1`S>ZCGRW72L&p92jyG@#0fsHErW!_uaPw2Gph3bYvA%(v<vg(&Wih>xe#aIamoESOJu1l=BVaSwpldZJYDix^3u*~@G+khy=Q{E2#}314!i7E
eu&aiktTwED~Ko_wm)fB-jvu;NsOT%LPsGok}TUGvE#$WWMrDoz-
e+jh9G!jaHXNhYxEJ53LpGKe6Oj&CKt3iZGZOr!abX*7wEa3WGFEYrXAxFe1hlf?dm+vG89*?!ulo-
{L1#yepD8FH8SfImX5SQio!+dZ%&ldCG@_6yxz#|`l^IGCcOEH)xHw9p={p)5rnWLUfVAXQszV$ELpj|arMLpUrp?e)zSrr~ByNY@>!y6GX+;}-
rWKV=uy7j4WJ`fs?4VqhbHfrBgFGG<~Mu_y>B?G(Eog-
)*$7N~w}ge`LTl<1Sqp@H+fdb~gv2al09EIbKTMWSEzk)95N3L*Z2*4pP~p<5XR`J*X=CIMru<Sd7CaQACNl_Fo;iu+llu3Nq%0>OK;DkbiKJh|Fd=O#c1j{
Poyc#3Yk#;*jJ+=-TyyBQE%TEO6wZ0;|2x>@U?B!V%}-h>3YP7+g|wx{^OB<R1$Y>q{*Yy*z85o!G{BiEO`xkw2{&qpjttq`!Fbg_-
ub(#-*Xj&#!eYy%{qRuEM;V+QHeji?v>}c7zgRE)}!wm&~aga4BV-IbBSR?B~*135l0;MtxP^bl^`LaY$9Zsll67suWDK4vUAY;>!Iw?g|XVa&go~HQG=`(Q
}YnID@_*1%QEEW0mhidWjR6V))i+GraJUgskEG4qvsuDI4b7kGI8pTHA-Q<<*jFeUiRhMgpr`9G`A}yS0oVDex`8cLaTNQMZNc@V#-
o$%2+?Y2G{ozDCDGF3<qIuJV1s0>-Y|hKGxc>LOK$TP7@7r0(FX7QFM`~|WA<hj2Nuo@iJ5!z;7rCStBwjMc4Sp{(ZS~~mZAgl>7Gi-
67Od%*5L`I`x`ED|DFKZ!efKH@D>XrGJ#QgH{M^JuS+@YwKsfSLT7{537!9Cpc^~ed#UAw2Zwt=CGb0B0SK(x(L@sN>Kmb*@TM~bqUZzoc*8(xNpTN~7Eb6r
0vzay<6(R#49a-
yO&{R*O%vr@*v_{Ky)H*nZTI`hMV(G(N$r{geU!J&;zh)zx^^e^NU1QwzOu@dtUSw*$P*~RCu*%1FH8!lQ^C;b=cw6Olhfmt$cHI>w0v92jE+$7>Q|r}u+l5
XYBb_#M1obUNkc(TkUezc6<7WQX)H$`B2c`(u`~;T)j^INp7yjp@agNK<DP~R|DJt5}r*s%K|9a#+Aj%PD?7~ofe1h|Z<WBjryP<VFde(G(j;2&AbSHe168!
x$W@Uc_U9gLmf%B&`iz#=(0^7qy^x~_t^i(&G06y5kVyVy^P3W+8tG`4pG&2*!Zz>M`@JuK$R{XQasMYWDYwqz{refy+HH89kc67(7I`T}GL2TbA=uC?EtZs
1d-%#b=!@A9s9qBpKp-<cTnIm_%s-Exv_b6_)lxtkdrF5@^Y5}B4f3PY9@2?dDI|L`{an;#Tqg4&*c0E~D_o9k^>-_bMjy-!FLKbat)0~(Ijf^-
!lF2EIwG^9Vi!z5$Y@$xBLTRJNf7yajU@&)r!^Vdx&XFB{AW$f()}VT1=#D706RDC~^W+e;MspPwd4p|@z;D=zr1x144T{4fn|sV`N_C&J(4TKRRQm0?=WV{
{0fV_egNYq>d&+dz$O?-_Uf<#fJrvH#yb1&s6$UNgprR*(<#+MK|Bc@=_d8jiS}Ft(@qVx@4xle7xKuF}*;Ozp9K+<Zt>Tw4>in6rk{xQU$UaF2-
#lN+kp){C1yA_?rn+YwaBlV;Di3Dl@Kb(yNQON2RS^%l)m1Vs!%P;p<qA<7(cvxY_68d!(MBLj>LqlZhR)=nMQMLgJzlZqz=9Ov>xs5;%xePc$6p{Srvmiw7
g=tN^7ux@)8kBV94{`C-!RhSSQEz776P>kBRhn-
c;qS4sulU@>{Ni03k!+TOcX(c$tOBNQh*%l+q#Zvgun9&@?i2_XYu(ZD(U~V0~ix^S+3HcPg|^!Q*C;x<cywBj5WQ}4s-
6M{Frj^{On7L!53Fb#~Hdis_Ci8u-
{Rr$@<?6iBh?C4^>_Lul>D%>(4J9LHySWij5I{L1?HH@iA*G)n!GKdJ=I^sNAowp#E^lG~VY1$GoU{fMS>U=o5!_D#@7oSv-
GU<j7T)nxOem3ASs+{$wH8__~h|6K9X%C|WPO=CLe(gAgQX`RZ?1YDupqTf<zl?qrhOkM`|iGz@kpN5UtrcR<V~KwsA64T#B}qnXC%@lQ-wltasam9Ze}Xy#
?D6+@irY46>@%g-iyk`&@%RAT_88|7@1;!XK`C_Z<!Qf!qWd5)-i|4&X-
EHHRuFK;89rt3TAXMR+Q%SDSfks}M3|8M=f6S(%1=Q*XM2O%1uwn77LI5wXxS5zntqXN%yk14-{9O{uW*jZ^@2e6mrL=MoWM^M)tiM@-
Guc<aCCI}es7~s?5JL=16XCo#PAkIFg3XFG3V7lOhaP5D@Niz73;qokVc@1T~YnOw)WpdNFL3M&Hl;#$4{ZYKhGSB-lv5BhbgGd5Z8+uk0nlYd?K=|Oc+v2o
65@x;Pij#DHl+<>dzR6ZkN4L7x3nTm1@HrJ}`W}3efRXkaHd8W<)f0jF(dpP(4D9&8?E)#z6G>*V*wT=Sm5Ey9Q@_x&3cgCjl7FMi4hY2EB<V83h&14$S&G3
fg%eO2KpiK-f_*14;apt^UWx_{q0_#n$K=#2;`xG2pN3=h`X&R)w)_FEGKl_?g_UT0?Sc`?u#25ht_*0p!v;5lk>ml^L+{W&CB#oaQ(SJlfvWW2HXIe=DNgR
y^FWUR1O~YMP5K`nwYV_i2}UoTqtw`L2u&Ag-
!~5yULO%CV%>#QY`}Vo76VC;fGIC&U}aKGjf(ngVsQG`D3%Wj5%1F#mox7?@MllNzTUQ71U$61nC)?hW2&bS478`JmMGj$-hfHm%-
UR__%C$H$%10)`LDryFf2)hg?PE-&*X@h8$DEUYtfY&j^=#NwQhAn!L#49!A8zGv-NBX`Jn^-xMNch4W%&Z#I-
tF+oC3NG}MMXYs{F%m|^a`rH7j5jICYL2mHv?r1!>&bj@>iEa3=+9mbB|#vj;3bl4$gz3i5)g6aq_w{u2LVvzx2&F#`_kCzafHw93|WEl@!ES3?x%jaVDTZZ
P=J4Bhc4PBo6xBcQh`6q8-
cr#MDb%Aa|Sg<O8#aZcu*KfgEPaQtSrS{>be{S(iZ3YjRYa&~BbhFWRjgw^OG?e^q$aeiQ<Pin~unee>sCls$tXG<pPfcNC^WfIg$s7hY<$Cej9<m-
i9gQ$8o{gOz(RGZ_e$%vi%`kKBY<r~r&(Qk!Qp4jng+8G~`M_G1XVHj4xW<|HZ*yJM2}7UAkFNRMx>1??ynyrI<in^0FH!LJ&7w>;iA=MYj(ybrZA(%xgON9
l;@BszHdotMa;^^4r;V|yE!Rs!&*=R$-
y285038+DP5A6)Z18;n%%i>qWCp5#jNwN!>+MJeD3oz{`E1FxHO)>|5K!~P<eE}Yj!6M(KJUa~p3W||I#tI9%R3A_8JH#nWoh<$uPuD<$?WZ8t8#Trw^HS~W
-!l`x9b+bR!s!Mdp}cLiR9lEAe7u+h~rQ^CT^Oq;XrM|^OT+Q!KhqA(3bm`=ycFHT_r1|xqfRlz*m%}e?Jy)^|}1&0J++h@uphNwydBkO)@LY=`z5O@SjBF7
R(L))Ov<=x@i$b?0^8KoF|85H<N0)W<E-{QV-#p!-
SL{>&aGkuIs^7=K%g+bVk#}jcUAIY4MmLLGcE5*u&t<M_lPy+b3{}qMaUSA0KOGobj7(CR4LArha?JY|LnkL(_JnZZXFeaS76Id{_flGz1PCnLILyRwCK+dl
w+gxVbNS`L~7a+gZOICOO6)^CXb)jol4q?nZKpS`|?oXuW*Rmxk#8Av(~3v+A854YJG&^kf(19lZ%YHlD^)pIhD0R{z6RqtD-W){J)lmQp{3#f)6Tvc0-
PyLIv})k(~Kn}NzegM*S3h73BUfZ;?hqt*90w$6L&e>{^4e^iIiI9kDJ4SCASZPmAoW!(`>xP<|b6JMz3eWu6<pFa91_=>EwPCC%@E2)3?tJ)bvHFYF8nl8T
x1(%Jmm|}<<Q9<5agYq_#i-IT@JWTnh5?F%xf(f4o5eC`r94p-
78MKTuj~vOFsfGl+i2A^gfMTPc6aEXx*U0wIMyG@jY;TG9iudvz0%HX6_MWeMSY}NJiyvBzA|WxOpk9{GQZNe#&C>byRPF*Oym>2Zb^`5K5E(GTIRbei%1Le
>MpwMLOF-
3>lIbhHkc~$ygs+@GL4j4wD8Y7(JDo@upNqE<kDummUL93H%vA2;Wk12qE*k&iLoZl)xJ=YL)L|)Ms`VL)nlNP0A4F&az1?PUb2GCS71OY7yW)!y2m0nM1C(
$?XfW)@%(Q5bb22`3$}tQK5;r}lEi{R%-okCUXQXlf(#eo<B1eTiC-?FJVE(St`6c<HdE+zqV{_nO!K{q2z4|K36ge3Oxvp|;yIK?QV-
S3=2Y|w^*QBP(EI&sUjfuheC7n+7slv$5YD6=H04ghHb&LT;3(bdH@I$dB8a5|eMqq6o`q@+04)vxX#oqZAZKYUy4vdb;a7Kr2&>=ptcW`2*TGaVQNdhJqm~
Y#(#8SqZH)dTX^rPnu5vdnetnT{47g+ilU-XExk;-
yl>Gpe#iIUmKM*f=#5L!h9gIDN0*H<;4)YD7`BQM%(sJGDA??u}n=vB!#;vh9BMd?17wq#0fN1Pec3avLVY=zrrCar9O0-
z_PLiZoSR?}aY$&2>4J#Xog#(kBC(8^a{avcf=%qDFSz=`ObB;|lWns&N;5>G1y)zLOGm6B|01v{5mi9cVW2%Q|e+I@Qvq!GZLNCqJgSa0zanrVDZf4n;+`4
1)2Xi~1V@$vbN$TM2RmJ5e#ajvo&Jq(+qQ2>Qz^y|t`G-l}E-
Jbs8dp^US6c;Q#`r`8>Lrl4H6TwpNdXGw@Qol&zqHpp0L%2Q2@;vh47}`DrW~9>^(DgN!ynvRDO4kArV%elwsQX=o{r^-BkZrXi;--
e5B$g6!SNlw2tWO!Lt8d2sqvdWomCUst<t;r&dwh@_w%UbJgKbl)b}5iJY9^i4zFJ`D1Abzgod3YN_|K=njNPn-cl$`1yK>u8Kq0x$lBTJ7dG2wB2m{=dsi$
&L=+07&lVj+)_8B1qs#8RN$r)>JG$wmn&O7x$RY4kn^+>SStC3tSEpF2sE^`M~xs2UMsR~c(QKi<IGHu%&%K61ft(P~TI&O{b=bqZ>^L4=8jOkg|R;>S1o)N
SRv#FH-)pfQTjJa{mnI%b;W01|WXApW-yO--?vRSoe@zOYUyn4@hNx-wu#^Gm)#+ZpH8MG>L4zL1-
pagA+31Iz?e0TbWnTESQp!)4rBOYJ+H3%ed#dQ~KvelX2qrFNs&Jh&?yAiEcBHcW&j%NG`tdHr?3&&3$h_(~oQ?6x~t+fQwQSD=7tCaw0CAgT)F-
6@dc+AmIp$l7gaRN`!LAjh`gq#2e(b|IO9Em}f-
$F0Y4JoT*%F`&}ZBdT19`Hofs_!Bx2Fmfe7=RPv|FyI=8OTA;y+n^MuHt@h%(wdOoys1Nw+pYjL}eR2*Iv%JcK9^J?6Dl?_-
tm#8BOZKR9@!E(<rNj_gdD3UBoHY`dWPf6}DSM^-
M&k_Z*QLIEN+IUKe(PTS^q6gDi)jEbVdI4|G)%(C18=2Z!gdY!qAp!F|Tjvc@)oZbWQ1*X)K{?#nMDI*e30%9_`x&NT|i#tAUd_4!b5s8)+~00X#A8&ZyeT(
*j}E@|ga=ThL%P-
DwV(3Awf<d7M@m8V#LzBi{i<+>Jo{}o$IlbGFv$RKR3ihhMeNb+#1anTeu<v#Ug*KK=y!_%%@2v8IrG_Ps=^R&sgZS(Bm`wkaX);&6;C+)6Tw>h$FFLKkq)A
$m1K}59~ZlU>2_wAIv&Bf6QH6H4qjPNfWLw8reWp=Z`v53`J@0J4G!@<9FVt>)c+$jLg6{gk*FLZsW9p6<^91{Ol1=uwrR!xDy=L;1P3EA1@oFoN&*9%U>DH
toKV6H`NZV0a#3|ZA^5VFiQR18X>(IN#-GCTnU275&bohLD}oNY9}M+|XzW?z(eht7o4!zY)3rwQ>V8&E!4Qj02(V=%7Jt@+-
1zlfMA|5@~|zDm8jF2QR9tY1L!SS<gCuJIQlCxhSV7?9PvsSv2m^xS&Kmdj^?kiB1uo$wggdo$P<7Pi)Hpd=toS8%!x9d_m7rKf2`bk+g}3Wu{RL-NCVKQiN
Hkw3N(_pPb5Wq;w{aOoA7JD*>Dr=yd#a$Z$U{48+(G&$OwJ7DG;a?f?R<I*g1S#W|!W-uK}quQ=lZZ@m;FOmFw+wB;E589V3id(XVz2mc}eo{pEr-
0H<C3GgyE##Jki^)@jC}H6!+gvlWO$Vim$LP!YQfl!BNLKSe^TGrNC%KX+O2GT#XTh<}wWqKL!urN8m^WmC`DqKtNO2_U0)1ZE%+|WF%W<!n!pD(<^7W0|rl
G6hcYQL#ckOlWhC_KXmW~h*ktjPE+Kio=(K-QsiRL=A+F}$-O?~paR|lvKL4#y#vPAc^>cO&}*m<(-FjNZHk72l9_D$kN@F2j2Bd1k=-
2?7wup9)Z+;x#{tE3WP2K@Dn5zOG5FQ3U8UcZD!VqA#cXS>SnQZn)mv-hndQu~^AqVg0qW`o6;)A*1P*oK$0IlWWFj41v-
s>Ow47IsN#nv{ELbSwSl6xitA1>H6PX*xska_ngg9izG9<p|k2emNIm2vn$;FX@}J9n-
pyg{7=o=1hBXrh{_8W2ynv^tD&#rDX}h$!oi_(T_5<x%XnY8GAToXsB-4s@qIAKh<(RwG0338<@syr@BL?GzMJ~G2--pMxN8IcG&J17)hQ@F$G)&LlcO0I(x
WtxMNr>MG{J*#XcRe?FNH-*nqO^(4dy%!{XG?Y&|i7HaVhT)fCLUbK?|VEm;F-
6sUuu@TH>}DLI=j$Aa0NlZYFizZK;|o$l1W#{q~KUp#Zt={^aIdCea}HGE|z-;g4;t^?7#-
CXf}(f`MOH@J_bkWx*WA_LRbE85XJV3fjMu$1MJb@23AK#@m_W2eiIXXJcyKwk=vQ%CLQYpn=Pk4|;~X#$yLtI-
6Bem{o68G>cr6oey}?yO}iM6w5;_8uiCeob+2z}O2x6#2bejuI~+9w+^XN_`QO5dLcl6rQd@kf1M8cRr67#~Ll)Cs`_Ut^#1x`cCGu9RmlzI(LpI8vyJ43X`
peRc)0!2Yx{Zifyq%?=Rcu3-`27-DF*SQ(WVcX#8>tqEQcV@q1vM3dRH2v(6w3>jeb?s$(Y=@rCH4iZDM)dRg<zdys+^lSTnl2O&+5OOCS-
d0NI~9rKU<hL>;L7eQ#1r5I;6Ux4*cmAd3iDDC=x599r_(&x^vVAb;)<VhrI7MrtcW(wcNHgu7?9z|XZ9OhF&?$cC>869Dw-
u`)Z#UtQ~<3n!%Jb~7rzGVLG0pwUZ?-~q21S}k-
n+7w?1Kn>__JV0eV%^d&9r}PzjEAe5Hz@Y)bhDvzU5U|&F<k$Dlx$4;P$OSAW+*hOT1GP8xk#V8PFU5dxpWA;?C;6ND1zl_PEb;4P4b0Jo8^gHUltSYh3&OE
0lk>G6w^{FQ%0OjMANOwa-6%m+g~fvrQ!XcDYn49ihCR1&rYbz-gaLC3Sl34vCo?AXak)OeB?AO@-
zG)#D@IOBRe@im8~^#4BeyYX21l^uvFWdByp|nQp)AX$CqY;ldX2(+;ie_DkQI)WN~xkPji?oiFW$-b0r>Y#AIyNv+ia=Ig$i%z8jM-
ivwAWYbolsHzsj{ab;B~(1;gaksf#M?a`@vbC}3+fm@IDK!89bWFn_1XPWsS8xxfYF4;et{J?`kEBEUY-h5y0Cd83`j}_1W@S+&oka-vx^yuzGCuwd1r96GO
fiF!IQu3sz;0OBiI~#@PCb~K^cXkUO)nx<NK*}0}zSeluH&@V|d4*7aUm)RU7k7}=WId@kLDplvZUTi20V}9*X6WX^&s$>8Bpz<LpZmZeVfgc`nFd1T=g%;D
&_Y~;;m!qT%5m7B_#jMIi?;{!Slo%T+1S&BIR<y?ikeMw7H(r^+AQ+@Sdb#ys$Qk>3hjJ^b`sece}=Pn0_ZYXy8u))z~pPX(^&!tx)M%+)7#Zk&R1QzmDNjV
#cO`tAik2a`Ry4&Dn^yS<<*zXB^!j1y6a#1EzIu{HB05+b5dMu^g*kf$mMf~V)QM<wxFw_(|9mh5Rf6*0*@&8EgD1Q(rNC!o_mOf|Naq*nicDTT1uF_fT(Vk
l*&V+V)1d3N<X^?9K)0r>K3#8>ppunG?rbZ?-Ny==vmxx(~Grj_KnG{y(pf^5f#nW$u;2DwOdKKyTxvpTOkyOuw4qfXRE@j|KxL-
5(qQLGLuB;KkzSBmcJ`Bn8F)(GO+t#Tl5Zo%xY~v;W%B*Y?1}x!NH0dL13zC_`}V_v@c%63K2aUOOsBf(Xyy@co~#<!G8N<OPcfbdlhzPevTxWbXC(;H(E@V
B5mLSSejT|n#Q00{C!ZY(dh3|cUiEb55Ec5+YLrj^ab+F?FoRq^SwP9^@b6%QHqAM$MO_3s;Q{y*>-
n}3n`WIHeN7nC1Am*?_&xR3kU1T+@L!L7mn<vC~ZTp_R+nct$MHZ53DD0+4(Rq2SHDwRb415lhFn_dXo*9)FP>R;)mT-
=mXN4sMrx%ZKZqxe!pzB3^F#=_t<kGtwx3)0t6fZ?^B2fm&8LmE)Z4rC++GK%6W5i)D-12M@mwP*wxANFh_LZ8o!`b;-
2gb9}*xn85zHH)VZZyC2!Eu6aXF&(plU(G|XZ^2az>?c#;8OwMQf#(mo*#daD$uBhjc&BLK=2(0_7ks=|S12#gj@2oob-
)*5^_BC9=VLMuO2P<*oJ+;YLaqvigJXjS=B;v*{Uv~{Fs*SLl&4iP7DQ^_c`SET3mv@Dc+O_2VlO^vvA#YG{x_A(<sG0(BR>pxF1^)w3koLiU6@%D|T=|}^C
T=;U*z{U0|lB2?(*&5*sD;bqoHT?qn2Lo@la9_Z5tiS%x*ctw>34i-cz->-mwMYG4do_#)2Ux|yFRkXH_$G;}nriI5Jwqoir!fM!1E8B>QEbPFRU-MGFeXAa
{ak+FlM42pXQ4Q$t4m}fBs=K0IFNH~o=$b0lbuiQFuqo3IMW%-)*{2i)kexWf3H47L-
jeYgyY7`;jGp7kR1c^tm}ndbfT{jveWuj{mnnYN?+y4`L!&sayJ8*uL-rmP!3jFTz&1|XsQYKj*Fe%yZLE&MOTEVe?`sXERpnkg54#)$)r+Wz#eVja17@b14
;K@5HF1VKoxKAYPtb3v%Ji*$2rc6LL5m;uxjDD9o~Aga;bE9;b21~CZN;|=2AtIUomh<$yFH^Wv~syG_5gQJ$S?D%58~=Jv!)3<+82e$9^J{=jfg9#2^`gxg
F2&>ujBL+(wx~2e@`Mw*OaQA!YWi?xpI25JZ(KdypS!L$vpsRV)wkA4|$v-
75@+b)IaHh;Eh9!&uT)i2IDGy^6>6umy^u01oalT9aAC`(%nIvdjD&foV6z<1@C~5sbGsU)fS~Pc8_jgrhtnMidCMfFu1f5Oul{=tp?#Z%eE?Qj*AE!v_@(l
@XUd2?iCyNDw-c02k_(l0^=UPs=y>`Cg}*wGI*C-Txqi4JZw?p&`M?VzlkQ5Jt@h#<*=Gtf3FGA&=Wm4V1k)76)ocPi&zbAb51HiyVyG-CGb`UMj8$CqpEgT
)~;D@nwVu;x63-
w)41rBM9%A7sIE4R0#3IpW{8Z)w)n~6{^tsYkf6_5H^K^s05HzX8q_Mp?Z^=%lD3kwI)*IEblIXjV|I`Gz0{7oN2}HoRNx?6Xk{gKY1=GgLAwZ$57`N*-
j^N5Vg22Hk1596>b?@MS$sFmx^OHVeHR~Ss}Gbm+=Py$WeSqEa&a*CfOsiS=0lln^smL;pBN9@FR&Vko(p3IrsC2R%exR<)d;5n9jV-
;5sH05#8t9P+qSVRg?IzbI+qvs>vMtRe`vfK8y9~shaBLcZa2Z!<0JTezueY5#h<q(n^fny{(Vla@`EXiaHqK2lE1cj_B$pPm*)3wLI}&GVVsy)srU@lPc+o
L2e?ANY<jU02NRcTa4-Q-l4#F#BZN0=W733U)->qBs#tGb?NnOmMx#<rGC_-mN@b-
O`QaY8pD1!WoXhRB2_j|DojwzguhHQO*ZLi@kQ@sVhVgIJm!$yrPFl1O_ym~K1WEQ*YW9nY6E;HX1qbP5$Z^nBuz9z0CLyu)h%8CZx~Rf;!YFzBi`2Hwju5j
HDk;=03q3v#A$l6r`K8e+9$1G1!Ps+LSiw*1ReJxKiuKXB+V~lHtxe81w#AnZ?4uu>E%2buL-5I@0(BG7X42ijjaD)9V}-
M;fSrc0xn#wk+0t6STUjMm~>~+7$#GAAf=;sXwbGD7xu?4k(ogkR?k#2u_3S+g~8`KmA-?yJR-^3kq%6Go$8Xp#R~|ed28OWX%6%T!-
X$CzHNKzYc)ZJ^m9#3RU5dNwBTENay|3)qEXB{DJhJUyIq2;^4lxF-
7~*2uC6C`Q^f~>sB1DLy}^t?q`)u<^!Cjt=@R~ur9M4ea|2zc29{A}|L+FMsk<u*n;9F2XvLeRJ*#W=fx}Id?Vrm>jkh6kr2hWm+-
I52J!Kd1*XA1Y#{>D$jN{!kv>+GyaI>&(8tZz94NCe^qeIh>w6b%o)jdcBHBFi3-
HpJM>E35Rj{o$~8S(Y@V&{_bmc!5vfcbMnGti~!G_ccdYNMtSI6bHONik#Bbg^Vn`%xJJ^IpwTn%aqxP}zsW9_Mslo2_tY${C)Qe=y&dihKo-PtN*d=9Us8L
QKW}h?I{2<#>5ph^ot?j5L-Rh7AihPA$%VIDLj34J1P)@@MU)mn*X3VFoS=-=}Nve4Rq38Q#^FX!0MYPFhr4!JSyV`GFQ7IC6-
#tWy=?VoVT2{UgN3TM#rY312uz)S#IZuEfHWWW&sl81y)S3pW0(I78Ky9l<tW(wh#Fm!y9!rNgVbnVgQ%koZyG2DuoMDr%y-
P(`*%4j*c*;=&$q_`1YJ>zw3CbDrgl+9ZT<SyVqC<5?s;*RYHmr)a=zmHN!o8JJ!a-
AI!J#wk4Kl%Zr@5n(2`di$?MDp7}pG+b7b$H>uoXB7jKSvgSSqjT#yhND6RgpRr%;kW%QCw<q=0P^C%Wl$sin$U_<#>yDS;eQebGy+yA%~xu>F@KR>1xW$7<
MUdXOW>a+6)N9c;6F0-
VJrF+5G^{aELpwjJ2yH&1RBlqx!<oFn=G4DI2E93>#aCaI73)M!$n>+k7HF?)Ez}CGWe5Qz+`lu3;{NKjN*~qi_6?a&$w3pXugL4$Tz=74aROmJJJMTsfVS8
Y1wJBtI-
9caM9xEHh63NkCARP8WaWs@xN#~JCfF2+R75fvt*tDtO5>x53pC4g|kP$SQaA+U)$`QwGT;Yn~v!yHq$^*?JM2Z;3hZ}+tU_Ojl^$@vb*)x_50b9BVnnFfn;
A22<eMfl%j$*j1;D#wX<hWr8Im;G*Hz!1x4?VHwgWJ-
jdd4KN6j8V3E4qM5}9Rggx$_^KBCrS}ngT&te=aN0N@Zkj|(LE5;N&7#&s6#i3`Lyd;|y&oViaOVot>m#p*fzCKg-
mJoBqqQ^k{B=g9!TEB!UJ8x1GG(q>-B7-o+;B9!bhY7&d1Y7&q2$v}?5-KpO?)S;N#o<YWW>Ss61&~UyUJyv}Gc-bRChb&f#6zcy(&S+eVErQ$IccSYWXbt|
+VB;t2|5t0=yKXdh1VKL$ILVQutJFZdkWWBUzqUW5jYP%A;z<{9}~HBDZ{mQ>C2bDCQleaAewZna`%*w)#eJfDq#|sE|7YYCljZIFEn%lt{)IwD(ubZ&2mAd
M#w86Z_%Q5r|8pE6oRJyrEt;4gjGlhQ-8-ngQ;4KtTs4QxT_XAmW|@2*ci2mzf_h$Cjxz5IS9J(HEm~n-FWd7Aja-
@lecnI>hq`nRklDh4xUv#!4ijzCEAv^NkV8Vvhxeb@R?2C8<?q*EaospH9k@tppneg-b%31CYlfOq89Da<ClAOS?@M(^XC7x?-
_k#xm9rjayqO2zc;PC1h8~3B`Cr>Lo`mFsx-3&=!PjQ(3UG>WUuK-6>G6=5>J)sv0e6Hs!jKBw=Np8Pz)KGVIzL~smJDdp{p=o8&{+Rd-_S>UPg6#l`G=Pcr
Zs-%zJ=W%~5%_?ST>c-*nB`Z71+6hU7UQlGkLzu$O=gssDZvuXy`@-$Jkg8-
TUpe4L+IM(i9Wg+7_8^P1|cb_<E>TZsAERnU~>ttA1T#K=2vl7foD+>!{kXyhob|G1{c2CsTARy;5W!6Y3PAKZeQP#)Jyb8zKUrl4bhW^q;_i}{>UyC74cDb
6HK{MDu+H5!xksKxgTj%A5d4|kGtpFGt%6A6`o$z0TY>33jgZT|srx@j<1<nYs`+PZ)t$H?nU;ScRq1{WZcn)1YiB#V&N4_1vczC+2^ec&vtOXmi?I|1P+5h
=UihfM;tu{E)-
!rzl$`&jn+uWNp=a#BN<(oI(D`DyiLl9#qqDxqaPQP_a%m^XQ<SF^bD6ltsbnp<_3{m?Qb;4a$xlXK|ph_>OAkFs%^JOX7Hv@YW0Lf(Ex!aO5>9d?5KO+U5g
1H}8Ts!Re~&ZCiMUC$c<Z|GlT>FzHlc~M&z^TfsY4XmOZ|G!`%*0nbvNM-3MP#{{N0IC&c@t^=x&_QvAYlN%xHX!SZ99y9!K*am>E^t?K5ncTt>&T>r>=Y-
^<&W$A{MRkao1S6SJGbBF*~n1f_eOUSjD-r8DUu5}L+ol-wHh^sL$x)5wrceF4pW}%x61tF$q=qnTJq_(cetwk#*AO&;6-
H@*GloexmnPwX~uz6y6=~g7s!ana0R_ND}f2nN<Yhjq^RKe;74UKNjR&N&wPCfGnUNFHJj~q8qoefh%63<R6v;=@G~>s!JmK$v%%|3$UKIAxpC$9+%}=i*!3
lIg1)*&>($kgdOw7qgp9~lg|NrVC?bj7@PJVy{Xl8B5H_tKVjHd4V-Tm>_!L&2G=D_$mMU6YzB@L~p0-
;rcV}ZUX8e{4+za+uCARhC;NKzpVd<{GU`KQXq9r2t^>qRV!_uHtTZYBltLU1Dx0#|f547F@SdxeD#jLwCvM#y%D}o=%t%IZJwx<5i2<gP?`N6h0l%qo{qi7
4u;Nh1xRwgmTdFJT)G`<xa?X3q}3vvpfOTE44hVbZq+_9MM8QeFVQJ@RJ(L8&;{DEK3n+6|4Y5e9qnD1&%KTRS<UhmAa#<anP+CKB~3*5nx8n7My*e4iD@FL
TIQM}*>RI5fgGiU6^S2sgwuwgiK9U4x@{oH1BqNd<sO4bN#4YPIiAj%s<BZD@ryX^=8Y_cqpC`cSEGF){2I+7GKUi?JHeMG~o&Uqa3L(`R3Da|_rQW5w-
GuPp+lPjJyz^$>#D};khkT&8O%R|RYD??Axj-QqiPrNA@X3qwG!hthQZ`j9T<oQ{{n;$v<#itL^eMjtQtdlbM7!ay;)f8K3HvUwgM-IM`aGBGKd-
B3`z_RZ;Z3-8Gnoxy2P`-`B%XHGCe|Bq36)L5&&tQeoShS1g$5S)4Tjqk*d+c?-
qpP01YxmiLLpY{$a?z##_O*s0@2#0%O!y%_cq_xRI>ejEhr&2>t2ZHqwq*t+Tjn^c%XWCQ(IwJCge1K5a=}0`R`Sv>;)n2?P=nlK(TGnsoaZ89w~%@#;67H=
#(!-YnE&<fw3MaFeCFZ%?1K`1U-#I@r>J6HTP7&3u<INZSx=7^wWwacG`0M)8ZpB|$kx!StQxyheGrtDb^-
KM4MPNc)0rrC0L`lUbf~4HtzCQB8`S*SZ0cCPwZ`GoZ2vjSmlYBSH#r@p12=*l*U*(qDkGaxV)T*E@Sa4O*WQ#7k5a9MC<&!M{_t^N!JorOW~FQlGPR^e_v=
)HA|#wYV*SXY;Y_vW>f5e#*9X$OsRs9wv=u+iT={40kf5agj>z1?anE!)(B;z>!UO8=a*dbs)`LmJs#mGWR=kR}PQQ80)<{Fsn6X~F8|@<~jmFAi;G>S6@ko
_?Pdhc~UByeq3y<`<d)vLAvCSa55UKi=OV_Fv{={v{dtwA6)rld}@O1AfO-G)4^NQ0sH2g6JP9We6P1U6h{kT_1XUgVkPJ2s3{K_n7Qoeoo-fo`fxzKENUB-
opW-
<#q$X_^w9>u+_NO5yo|E^IB^UI;bxxvYOj{Gykj~z(iKcr`Q)*E{}`rwSGS8KkvR>g8?s!UMbf!`fbnEx+FhQDwGO)>p#Od#{9kG1)C1zrH<$jkA0$$I7g8l
L4Oy5*iT<Pa`Fts-680<^Kr6T4AMUhtYP8v)5wj2t{-
*PMq#!<gm!61jC0LFjtdTveV}1lDXf<bvulCK5&gL42mKJ4S1y6As(=mbYi<CU*+HpZZ{oFxw`~2ESL`SheZV?IV0X#n}Kd6sYU9T#d_m!9z0MA+lIu;VJsC
3fqJYOe}iTZP;Efi<1VQb|0$YkDI>@?Okap#J$kVif2iFEp_MP_1&5k)y~+XJaB+w<v51*5E*fozDiW_m!8jD_#3V6j&7{4VlG=B=9#?;1v+kU^6gF&!vRBR
bC1-
W96Gs>Mc@m&YCsso{f;jCEFv2am@=3JG{h;ser{JkNAh@Ry>nqY<3<J$Lq@yf=91}KI;uaK9kbxxYX0<*`ffj>LvG$~$NQEg6!*iT)!;Be`sb<n2q*5AzqRX
y3oz6D3+=b-
4z7hbVSx_%`1q8U<(puBD?ck{H<}g}2T27g=3OqZlsA*`ihR$J1+nMO#1#8Rn(jZdxrQn!L&l<W3$n_tB7&SFfc2gPG2^FIWua*aR|<Xua%|1Gn=9^demTQ#
rQ{_3BYvK{kQIKQxF>??43wn$lpB~O8U8L?Cwp5{n%WPW*{`e|9EQJ!g{3MmP1Uvu2VB3)Xb5Bx0ew&6y?@<CYBS+Vj_LN^{MqUBAPU!fQl&;JUITP;YU!yH
r)qCrJtnJuZ6Fu7$bqQFm^gd6;@S-kWA|`%ACp)48gL`~Z=@Dyl|x*RN=70`=wD(U3lMKz!>B{Wsu6IHC-
M({v6;gY(%UNd<U`?_pb+2P9G$AjMuehLG;r?bowZV0Clu4<d!uOG*MFidrGysuwze>5-
(XQXJY|#(C)C~InMNE1G<v~G7C*|9G)CX<ZLZ~}YRZnpIoCjLzcOFqB?Gt2RJG4lX4XB;q%(7>30T&;3Qb`GFH-
W;;y(P5ug~MC{w!&TX`f1K4!{wO#pW1J=^~SgHWeI+!&aB`t4=t>`~z~!#Z+IXKfl)#CnLb=)k{m&xm*3!gRAR(wvU{J59;pj=oU@Jq;vPpK(qHsf7Y0WV$2
9Ut)tNK4#8QL4wBh8wFW?&riLrvKjr)#%}vg)>=8-
J^xz;<RM<+nq$&ObJ~_c=T|o~H%`KVyFEnhm*kZ#bb@eL$`^==F*c3eWq$Fuc`<v9IasvvgrDdc%F5Sli<d|kSKC=Q0lUzm_1S(5m1X=kgLzh)Bi3dXE{$VN
wwyD8e06-
vRxBIQA0j>zHJDwTW`69)b9xeecyMr%vKSJyIB^tzLa<<7)<9U!mB3Q%%vMs7?;gokA#?SBno)TVGKS(DRoL(U*Pr1e=Twq!a^(p{ficwv(WbiLl6(lUAnM0
FQZykfY`m7Tsmq94K@p(hLlCzVnr1EB?EZ-wpNz0`4*!pH`zMMaZPzpmpYA(BO7#Zh45E<nU$zJObz28(E{?3Wn;#X<avFi`PI<bs*g7sNQolEVy28Yn6sLn
O3W(@!aM|DMBN+i-@pUp)douHe+JPy9Or|1_N1H(iIUKCf=S(@-
(@febUY!e<GwR9puVktmv@;(j3?9nMUp?8>HzI{h(OjPMegq^91G*c(1#XH($?RxuA)5HRcJIGi2mp%h@UU@Bvyq{t(MlKIVv3CA8;;TQQ#W6h*qUh0+HvP^
O)_bD!IXtIbVWvO6zJfgijFKNcXm|)b(ks@GB@U1!7Z5mAa=x+(ow=H_@=*UOM)19PWZ)1TtI)&mz`HwU+rq8{yR$GS3Lf4c4N)EK$M~Y6AuEx+F0wq?odRS
mWC;|&M-
~@~kFbRNu(!91g8+23D&Xg4U60PX&?Yvz+sLIiT&=0^1WgQTjx<?Yi~CqF8Cimxn{#xvea8Mmi^Xu}`0~eJAgY?oyzMO=gHF`s?{#EsmlhLVwX@U9AHreR5K
d3v7tR$HDk~*AA2pZkr||}=I@1RP$2-
skFZ^Tgr&FA&=~F(Bcn?=GZt&;mEkA_iW$0vKkwKnu=&SrK?`yQ)=_8*rnY|IFpg2+W(s)}YFbKSg70$80S&AgiZ6pss`QSbnewf;s+p{uhsiFpVLmKM2laY
uL@=;Dq+Z3HBv|vk1`yp*cuDyqtHE&VY<GX7=Wm7L;pa~ya{x=ibyQ5H4k?w3uGs$f97X%k|M@y+S`3U5M9O+4Od)pEg5|}ndbDgbv3=Wl3Q<K=5>=$;xuNg
0Vgb4}o*U@If;@I}M8PW{3icnnd?TOM8J`)c+UaAf_d+-
p{YUE#}L;@#NlxCf#vxM`)N(6#QGFs9+G@#g|(56LGmEhk5X=f^w&zyVfp}<*w_(JODdaeo(@bMChkT>51=BE4n@NB#RqCd2i9&f{>#21U#yR<PAzqb>54q8
c7$$g9KPc&8{=FNcrmo#7Va2LIn>%k2vl<M|@K9c{GjST0IsqtTXqk6bm8tqsrW*@U}jpEsr*?_pux-$TGE-
|6GF#8RD?w*u^cfeSHfgv`HWLljOSHwY;gw5WGXRoSf&Y7w-
v|z{Z?86V#&~C`m6gB9}TTNZo100#GfNM)VEcycyzU5zSSb3o|)tX2)5({ht&s;EXN_gK;rXDrYSyTfm`orgsIidIeS8Z`hgmkLht1KV#HQABiu_{K{bdlZ4
`x~|*q{$4}B!Np+5ibLgtZ`VAt^r-GG#fq-
NKyi&C!RR2{k2^~H(sxp2LMw#w?0ZaiFm=HjEuE^`o6ssGG|<q;n{1B9-+5DuusFX^v<oJze%mF0J=^g2~tt!WhmD@TefS=D1#UZ>KMRgME-dmYiHAZ0+aR^
pUBzbz{igmKhXb*qH3NpNs`d_{{Z}b-D;G=1HdaTSTVOTGCU{(LCf#eW$&voAY7#$fk5GZL^JsCG*<eb67A-U4fD#L2`O#a9;R3WnGHd-
l6$B5W()_;7)*=uT<PzhJtX!%HBGOJkoFi_&rR?u&|I%%>W@`}rSgWY{!f$cNmfcyH4)>;l~C0N7v=EbAyQxB?pTLW5vhs^e?6(yYj`uH7D(;n9XUhb_r4pN
T_?x2>S(m^7qwN;ijX{i;0Bv**-ed(q~Y9^ALFtu@n%8v36aZy9PCGt01d3pa*`xQU)hGVuM1jQekYfIGLE{o?nKFan?n^m46Tm<mtlw@!Q-
@bM0lK1P)}^bJy-Rj!mY%~YBWthrW=bg7d(}4c`F(HL$(P7Cw%*AJFC-?RN?R&*ZjP5*JV37I65pdstR5Eg**PJvQ3T;T-
QlOEA>ksV%zVut^I6J`+zWg5)K3MZ>K&Z@^i)Ccb-&D7{8i&qhk7?9v~>=TKrt8w%mM}2gj8`%EzU{_F6%!2-
NA+;hjE=VkG&Pza#%Y%^3>6+1N{AYZ*$Z)s=jew!Tt5FcBYa(LIrj&Z?B*-
}AL>a@Q_OqJ=o6j!qI#pS*PqqOl)K`kinLYJ3#6j>fSZ{;b5f4f>R>sFoJhww*BY1CNKOL?)y|ze=^WO(Yff0d7n~D6FAOvZN+-
Y6Px=n%XsH(sENW&>Z31O~(T7Xz23+6ua{)p_glK(c6CWNo}A{utq+*+j2y`e-
4E{I_)8I%Y(|tf&Z~hCmfdRAV}izT$5^b0YOJ`YpRkON7uZDEWH*g+{GJEbj$ouZs4AjiEiwIZ;xdEc>TV1sD?txOm9xMtD<M}4!&1RD!F--fydc7_yLB{AB
wJ4cAhzphDWfgx*WeFEzp#F3-fm~A>eNOgX`T9%_s&7lA9hQXs3~rJ|rg#_J5a?*^Q{48V<n?x>zN+^I*iXE@9SR7b$Z=L3VDC#i-
sX21Flfl=(qSK3l>ZtSbn=zu-8-
@t5L|F_vyVRv{+D;lqN|)S!EcJXCu9*5SaTd<d+I)LSI18^A*&y$=yhXZPQ4K1odt7X3^Z%6fu@b64e*rHbR{M!_0QxOapESPnXqcd*7~zU#H1BwW#N3!m?#
1XzRG^nw_%-l+$O+J@W(GI)*MwulH%T9uE?d|;fCjvq*yE!*pmEMaq0^IWRJiY;vz1Y6+6bdSvvgqZi|NB>s-HX`0rqJ2`^ah{S<fHtz0(2HHH>0;+p?gh-
#P?yxW9Ycx$6vEUZ4AlKMSY(Z6l7`uGysi3ZTSE#;ts<AI1FE+n8g(e8P0i5tf0AIf^x5jcKkOEIKn5VRHt}cSJ8jrzIqA#EvNdD60mp*{h{!qHY>zIVe8~K
yxAjD%E3COWy$v?}A6WFtWgt=;Q$t<)gabe;Z*B+>UEo0&Oy>)@m!y3h0x(UszN&(Qb&oHw0pY-
?g@Ym86`jlW^d&OzOL%YIdQe&iXPu>+d~$S$C;5z2agjRpA1oHsRk3i08!Cbbxk6uyJp{74@)%xrIJ^6lSIetAT#AxHY>X}HH^yXUKJ_w?7UZ{5Q87WRt^x!
~yNUMwOjt9{Oha{)?37Tu*viQ&e9L;R-1A`;29rMoX_Z>@pQIROzM3n(g&>@!MDJ_6@K#HbZ6tP02T&q(3B5<8kbYPb=Jy?o-CrCVkqSe(>3-
Mry152JsB{}&$Sn`F_~S-E8g|}PS#c^>;HkzNEQ^H<CX}3h|7#ggSB!f(<A=$9k|{hJ+-bq3XjglzjcCSLIce=)ZY@hDN|3TOE^gP}9r*~ocV~>d2wxJi-
(4sV?R;Q7^ADW<UfQ5Ww->^SLWuzhXKk9j-E$>Ttc3Yj09<SEfL`2snOB_-;T%Ye*GBsJItTNt^Jeju`ob0X!50QRcJsIKquwY5f)r7-
e|8k@HheECOIvRN?Wdfyx}QA`NJIRecqVx!==$~E406kT=wnf5W9sBJP5_UWNsVnOV}5=EH)Y}->XlQ~(od|hSNHO-!@@;6%RH4ld%TzZ3r1eIc_Z`V-
cp1oDAPd9d7k-1jVY&n8GBZ&-
Tyl&Y5s8N4}}a#VtX0I@v=ZtW47CeI=(?*V|~evBRpBTh1L!WyV0%TW@$vos2ual<fN?i(j`o3ni_Hu)Q%f*)5`=9Xy2BlHKzeyj#EbtrOuZW)Ku7%MGQnkK
F;GU{Y7W5my%hocr!7^OR~BofDo|Ai!3;?b|@Qxc#(p+w5P_?<0Mp)*X8HQ5gOpAD&LN~0xBub+r^NWvTaP;lZvp0#nyQ4<Yd#+rn1oRpae*(k#oP{Up@&4q
(hCA4b!C}6X#Q;d;`~)lI@crvO}%zZTTsDP=6I3&m~U^{8<kTiUc&qYqw2ORYLSPsBPb5sAk=N`uZWeY~hL(C+|B@9Ck$<aw-
QP<VBMv93l!a86qOsTBLWfEW}i}fA@RkJJff_op35+$5bErElbg@dJi5fQi|GjVp^sRth6gN-VQ!!5K2CB_5xk%MbLDwuqSsKTF!%L@JuLOK#Q+RsY`}$&^E
+c&UNOMd*=?`r7_s=sHTO(Mg|z>QzaI`hIe7;57U>Ez}F6>cD!@*KN$wa_sJCr05i5&7Bo@CaXmf^XYy6lmG>3T;n>p8@>^r54o;3>^cv7^+y4H$C@2jSz7w
m~K%OSo;>R!zEQu8ZlkGHp$eI;s2)9zo5c2at&%8|?0_^SGm9oHFFawGw6HrTYNvY!b9*56ho5y^`YMWyS<aHJpbEn79o#BW`EeGd8W6jAX^(FRv6ex+B*y}
Z^4cA`F_gOrY(_wqy3uGg{qwtY)(~kigNc<TegG&PA^KwGz-
1%YoY*Hk;ZMgn*a9CXn1x~OG<q$c**M7O4MaAp<?}1tfgWbQD@M}rMtI>hjTFuNQg{s`7_`4_ukT1~LN1vl4LY3vL78B5&>f+@8DQ{orbm9}Tv+E?Z@{L1f2
n`KQ`_t~rYvTZEEeVfHu8=x1W8Q6Rs!G6*qYCKx8Blo?iFS?#-5e;Xf(dH>EEaAq_hja^(++s<g<6<mC_G{j=Y=c&U-p(?+oQZ^4-
5=F?xIg}<Fz`>Cn6WtwCs`-8IN-
B5q}Qdn8BC)bKLS6aje>MLq@|mv%rohSaUBpfXMc*NQjM$rfy&gYkDR0^t994^SrNg2008@C%@(vhID8)qnaVU04wBEIw0N3a{zek%eJyJ!+X)73{^H?!*&d
xs<pi=>a8Uu3(h4@_-
Fg>rz=C{iZWqKJ<<23%CJBp^oNrSrq8Wk;PZ&KP;jKM|1cdY{c(&o#zTOWX>3C*KPS>A(ykjJdA|1j4i$%@ZlhsGlcpoIa7WACeB}WXB6&1WcDZJRQN17O!n
Woiot3dE5)W?D7W4W^VHK+6CFkOF`j#uGOt-
M2PZ#tba?D0HSow&Vx%F7fAlt)5Qt1=7bLl%=Sv;{#?sn;Y?$B9^s3$8mFdR+_tNT_$`V%Y=*<t`m9sy2g;!`IA2+!FaxUPBsQkiB@^^h%szWRbaBgA2*P98
SZ<)71-Rpm(gzQrf^NeABAb~#&2rO77-rNKNDb1#oRY;;c(LJPw(i1hTYyJm8+)jb<G`ssoh1JBBP3k-
1+0vpb#<lTN^+FBjGYUKqY5b<m{*IWQs2LV0vCmu&9l+qC0mbV<mR>6zZgcSKA`^omRVk;zltKWY^3#6TB#Til6^n}Zd-
9eU4rH}oPF>x2owZfZZldBxq30|n0vWS&{hL3`Eu>YI?sGFMIo7kJk$=9>%pAwr;evTtdkFw&5{V6LX(5Nj4YFG`P-
#}=R)z|D=OXgHjFC#?ei@c$6;aNwxr=U;85j4!}9iS?6!#W1k3{E^vjeT*wS7@6~c)k57Wd%3pX_(^`XnA~W?jEZ(@IAH}HV@p2iw}qQJDad7&AhQZrrk8F<
wfyY*<9fcyA6c9GiDJMmQw?ybs~&bBm?Y!Pz7=}4bV}<4e;(*rn|M0IuIt}=eQz&NR*x;2$0f_c@MCkXTR-wm%fbY-TU0BseOT1L1R6#B#j^$v7?k?OMj4VV
#=<9IqXmqq8AP#@m=*oVw;5MMoTLi=#7snbsC<Izt`{@_1%VPFhkS{$tcGAAu06YswreOodKfo5yv1fgE<!+l`T8qjpoIJu7+$O!~Wf`5*gI1ri8VwME4dC%
mg$A3$toh+Yq7{$lK=I0I1UH1Q8`>RxH5sJa4EjfGFrc2MKC0KPab+7KR@z=Vv2-
D^|B!q<)0qcj+JQKR&j!3Ro!`2?*igJLC}1D|ZBG?Eb**)nh}aPfIfQHVW&Ca~pt#*f4ds!T#Y(Ws?VTg;zQ6rok3|pDc=&r>PwN7O~9a@drIrHc%G)lOOS}
%=i*X$2h4$UC09>GyBVnjA!y{e!qQDO*}MG&1p2t#RIz(fk;w3I-u!F*zhx(d(gD%;dp+#CAJdj*<}+3l4Zq~#4B@GOes(Gh2gsPF-
6fd>K3`jiAKDjflfK7lm^0r22)X|V46%WY|%=F6jfS&SP~=3EIBJjvmsotL>X8Ndf^$eVahTIT+o{f0moHrDMx!RW2vuv_`jeS8Oz*AlApLY3ervp?4p|ZG>
aoOm=x&@`?*(!$Rbnt3DKD+Ix^Kq$THC!%TE<L$Y?_TUuC<zNG<FZ^i|+*H{v+VB-
0A1`v)Zv&!^ml5eVr5<o+4MEkRkadHh=Ooai)AI3d`}X!@f&HU3}jdL=B)IGF6bBKI-{bE2agP<RF;5~>NT-
%#OxFd{F7Y&8o>h6(DOfYmCTgRwM(lv6R*=jq8s>}63@(tyy+3BLNF@b7!>O?RzY{!!ISs}#IW_*(5P{2<P;^cUP0-
Q#Xrv#?dsmm5;aviLGC&wblo>?mwHin2?{-CC<-
3_PfFIC#{<ENT1FY!tYwP=e6nV@Sc>EO)hO7+xmld+;3XZWwCiI0^#h@34gG?F)+^k`N+{A+)w11+!D~0=iVngh-
~hQAxm26`FuTUAaj7$#B9qzZWd7QApIS<~n{F`^UoRt!s=vCI0v}B$r`f<9^B5w$Ty^&M=wEat+6btcF>Yp_Ik)8nl5bAp7Wc>C98N^UMJR;GSbgS;WB{BPx
9w;2SfmFVC_JdRh!XNCe7ZZ$4-Frk`0~j}AACILlNl{b1rjL(|n94iljCH~SB0yTag2K~d>jLn+q?z^ix}rH2k`X@nGUr2R>=y@3u$a-
+u1?;EmK2}h27HzgkT+)Ro3AhHKl-4YodR-mlR)ya*8+hXwBX-uOahdZuZwI_Z9Lt7yDDf!JeF`fQ|;9LeDw@eo>d?T^#4)PWIc%98tqLdsC#&R<_!h!X!Ht
ogyA{kG<M%5Qstk>hcvZ=q|S^YpH(V6&+47PT;PN~ITOlj?4MK=~$A%ti6!Map)ECVoJ+;3T|RX7bD>HTEjhH9KsX}ra&Phq^I$vwtTtXi`o9fgu5x`3^iW6
Yqr1rq-rjw2av8HOdrJLak42mm?J^`Ej62F*Sd%L`-
_a;a)XK55enx2=gVQfJAx{?T~gni4EL9jj8Z2{M7f@0jJykp^*YeogmQwpCcgU>k)2sWIZt5hlz`ogYZ!aVbtmSaK}YXU=KKx4I7VhHYM0rY!k>LEE9At@&?
*tgDJZL1PETqC=x_3L!*Bk1R`QzzOviTT2Pd|2R1WD#<9_yrBOP|6Wah=Pr(2|7)diGPWD0t#v%`K;e!XiJ+)i63d*921!BbeB`eG6~ov+E_#3R(s^6>p%7=
RS6hx6NU-
B+HIcBS*je`Uv8xR;au`6(Id{7&g1f1wmg>C;vQ*9rr%)xnRjK~o{Sfl&evdm5a<vgU*~qf7C*UHj{M|cT7OZN8m+3le8%+|yzTE(hbS6I!s5+hMy99yfCNP
%Qav)5HyzFeXW2`|tZ)Sv~NUCtUCxs=n)~GZ*21qp&Dwr#NT~YgE3Z=EDPG1vJ#^y?x&bi5JSyOFn4h<J}WR$gsw%lp{^!V*9e|*vG8(#6iWG=5aKz%<TxeB
o#q)<Af#9w~K*Q+$>&X?Przt{ix_X)}b26_<9g3{&Dp`oK6yB?ci%*05IZjN_9Jx4)8dL8T#P8(Dsr_-
r=VQ)`?3!&MHA$_^J&vJ3zFKX=*te7ccMp}l9hF<-}DMf6lF5kUa(+<0zaDzv*GIg_{Hp4cvj_1~bAp;R9ufEzOSB<Qknn}HFp9^$W?En$a-
1L`b%#nOk49OclyF@F;`c%2~jXx$2#TQm5A&`3JwiizAxmtI-z<5W$zwd1(N-S=@o|0&#`nI+Kp$~iUzGFmgtQhK#-
}JylO1^^xEK>d_yEO6gjWO$#e&V^IP@?0bGdG14-W@(HuH|k9VkD-
6*h<wDQq*p!TgAxo=6z<?8th(7^4|Ap^~7=@_A&wm(|)ZIO2m^xGjT>roS}0Ap9(FH__nuWGZ$06%#3`Yjwx}{3(r5N6)U(Z`A<1}cV`4OSF36^NHBm4`6bo
wD{dah0C1EOA<fi!<sUR&*$y{5vKWOeIvWd5eaH;`tXseNDhJX!sz=vfsf|9lYa;L-J$Pwk`=B=~o2d#%MP?dOOG4w@jdU{=ray>ZdQDDcb>-
2~P93eg3^CO@$rL@+7jqmU^3xEWwyIOBO%N~VV@k?d0y8WxWl*HW3X_BKdA=m`TQ(zgg1;v-rutZFOLZHy2u*i^0?GDrNGk4}te?L~-
>5XrN_5Fm7<?5FcdMrX6_kBZ%BA{dT%BAIqY^7_+~;lGFgZ`Gr%g0E+k8hG-MWubKbNF4z*zZ;DGW+i*3>stGK^3o_1@>LtqV9FXb%NIlO-BzEd1OJ^$DXI^
e+R@CB3AO_vv-9WrTY`4TJ4Ir4``RjH!edqC1^!RN%_K#-3%;Wt)IePQ<&T&-xSA)=!E?O*-S(`Ij4-
mfbT^WDF`fV`{pp@oR33TL@svHF~btM_e@t`^Js951LN>jIxb)`Fn}WvQdAq3%XAzG#|E)kfJBSu3kz<uYilvI+krP1ROzH%}-8p97?+t6>)Qr#21-
g5;rN15InJF-?KD}j#0bH?vagEuM^{Qg{tp;*TgIM7Mv85qU7D-G)lDZ1*}vBfdi8#|Jrs4ls}_tL5Fy7*+wXK^@1Co-~U)N9j1?;kITOVLN-
W?Okcl!)u@Nzw%nqMt+s0)Hkji*^hjBo24mw<%MSd4N>!$mx`Xy+D6)#ft0W=?IT+@XLoDAKN3at#i=otLM8E}#@H46;EO5idO4c`(49`_yvQ_pu_rMFirSQ
{o0+)yto}~h+q!>9ZXCLy*)G7AatVH6^6n&Hild~&~bpdXpEGq(-&7rjtP9+g&Wb}tG-AnlFD|ZfJV2wn<JFpC2xx}ZRNkj4BU>nJEs7&KEx5tx9T5~tQX}u
$-eUbjA8A|m~zTU97H~8e-EW)+Mpc3lxU=B?TF}}?bI0=mf@UD-G6Qc_z;_~>aT#O4ZVAI=+$>Rov7ox^vS%*H#cStgG=h-
4Aj6D6dzkX*GK+~bIhd*FX7{w`F4&)6=!=~z}4&e>_j5636Y2ral*teK55O{9M(M!W;lcC}+bjl;qeP^6Ou4?{R0CJ$&`G_?lN$zKU#f6xjV5OO@lKR?go^w
CC)XoE6Yw{pSFhojxW~6xw!paIg`BEEK4WU;|<I5)p40#8LP>U<Q1Nvv<MSlXwZ~0mNjabC=T@{_<&%Rj}w<IH{I=rEjS&t1ZQKf|Bzi;tO^32(8YX6jByYe
!M<{1khW*6e>G3g4p&Cw#Tw28lck)ilVy&)0!MwsZlk?`NR?Mx6#4<S%1^z@u)Mpb|8%0VyvYm*Hx$pQ4udZ~iWlfv_4feXyLGN0p2)EUutm6u0jXO}=S{Qx
k^sfs+AkWGl$!KTI7AB(!7p|@XMcKtz|B`<9OO$xslk-kHK1r>>99lU>Ew!A|Wilow%Bp=~lYrzGKqm)Ef`&l8HpFsEJPaa_B@)*!)g-
XnJP_55|aGN70;bcCQvw4nX<KS#PA9XJ42pXSb*kQ@TmbIxS+3NE51N|2Gs~lVl`X?n=GS&Vi!pcmk8V)Y=9BN@O-
&d4iTS0Qx9w!ar6R(uov1cMYM2yA;f$=FOjyXp0aCHNisu2{Gs_xXu<L3G1!AL{~SNmv%_?K4yT#7nPWvxc>Jg3Ep)5qn^s+HJH{8Zyh=PX0&^4a0ioOm=Cz
86x2(KANac4rlpkHQ@L<^KA`i576)uzfpa^UPWB1F=3gOkh(jI2|!<;=_=(Ll)yE<5Vu&DzG*gy>t7I0@<`n4Rcg`+VJsjLr-wqG<W<<ei?!xEjaS0w-
D`VHOX8xum2WbOWei#NI#`Yw$23L6DV~ow93H1fb*q`!%5fS53;!Mf!6ZRAa;7O!eN+MZBck^dsFu|F^D^L0`>?+RWrLp1<G^F8^Q_BG!sjWSjYW4_P-
H0fD5yt$h2+52CKB)I1*`-uH9B>Q`veEBfS{7cZRB`ao`94M6`WypLFIv(k)6yUDK-08t}(p9?1tUAw_u}Fi}(~7}mMWMo2m-&F-1C(HxK{Gx--_&p$eab0(
I+Wl_{@rL4oYES|0A(MfV^O<m}y@(w-s=robsRJ87jU(`zKQzxxfHD#%!_ylawYS(Q&l&18*GXn=ohV{-$_L-
`&?7>6gLm$5*t^?%@JBB3THU!f#iWghCxRmk2)&RkS<FR3u4zRUau#${l@xjqwh6Zd&fva5Bd`~kq4<esnD|Z1iF9>Baf6AgpYB`GE7*=bU6KX^%|G>ko?i_
MWpxkDX4>7ZheHU{kFn|sBM;fhWM0!9;ckH5k6g*>v9knNqB5nj3>}!hRKa1N_Zp!LKwz0Qv*WipaGCpt^iXh((!6fQoM`+I+5TJeP`n63!ReiQ#aZqXy5gZ
-tUu0=_jrSX(cJYxbp}evSD<tOKh_<DEC1M%U033m9gjUQ$>6Fd@U!e84yhf@)Hoy=jO-
1V=)Ulwoya`JCvdme$%3jW%Fr<dAB+BDU4n7wdNl{!b4v3MjcI2cZSa|B&%(&)k0`cHQtMsehyOC(8f7e2MX<m-
Z{U??3j&U$)9h4H)%efBaIN2P%o2h%2<$80PSQ{-}J7%2;K}yj_Estg1ajTuO4u7t`*fyR;hMkJ}Ei+eVkueCnW3CZMeF^-
V&Uv(f=c&Wg8IU>2E6eLp;}222VHGOle58uA3yxdL+|o-
uN%&kF>@We^l+UjlI`Fn_a0suqyr!Dg<gYe!5Mr*8O@U~kZ<oR5UutEyZ0TNwqz8In)U{P&uG_1(e(JHGL#AOOoUx_@xe*rW<`LbRrZK47*39o63ahFD1hQU
%sY5(j7s!N5%O1d@#u{?|hsEI{HWKk-^d!LC^dbf57%Rz^b&C-
pQl4R}kOuxzT1o3M;l4_uMO~954=lCZdWPB+vtaXzyq3*0aqM<k43CqH`CedE;n|$1P7%T<ijL9F-+f-
Mb{EN}aSA_1fyg7)y+xq}G;fvqhqxFDF4Y`U_QAe1A?s#$ur#N<)@B5pb+r)`ks`}<*tb2jNciP&@=`(^rudi|?R=EAZmP`bE`gi1!Oom_NZUC%mi?ssSjHR
gS_Sg2ICnaHrV3}ova7x+^lwaich@2{AB(phwKz<THzU!yUUPn}Bv$3~chVsN#0TOFSky+xW>%G=IbCx2*MKr?bk&wkAS)a1@V@~<jYXVE(8kQx40u9l2FxW
o>=vv6+>|yWHxdCd{HmHGuLo7E?mSyW20mbtef&K^Y;vBEw@)(|KaO>4tEW|@fzP=%5aG|ugI`EhxWv|7yhRcCAG5RjB8DifY;(vU=ltTi(i#xym!TSf;=?7
FbI}1<YPmgg&86xp){t&GAdgY-{_uil6<#{)(FZ+eJUg0b-alNgcM~~Y*&?5aDz@I^vdil}G}4G{y1K~#{^7c-
d*p9)$oR3wc1^E&F4QS*fC)M18BE$?)7L)l!PfP1R=_)ZheZzLwDg!8{{egZXuiu|U>h&~J(k1_mq4G=O@Av$9?v2MtpK>FI7dF^N#h-
8<9u=mHe;x};{+zGh}B&sFyG%oujR?eS#wq~g4>7!4)g@~<L1Vhqip}n0P|{b=TocgU?#^}?t&FDb%w<buE|84^1^E6m8bQB;yi;=P`zYq5vDna9M`jM_)z@
+Nl`wkWi?ttAg@JV&KOp40o36?-
T7C3r*m9WC&eo%Ww$)ytMKc*Z^!0Q+`IGEWDK`Q=n_lu%wi9bE|Sjt%#TdL*Ic0vVANwpSK|!R=ZF_3g?u|VgEeZwPF7ZA_BXa6c0dH<?WanU04x|_E&15ye
>3Vk)f3yrYdW{+^gjW5vx@d)Rn>U(CLU##165~=G3Bb3ESviLasVmde7vtb67;WP=-
|sWPz9bz(&`P^W?CWM*pLxe*((?laWNt!RV5Gm=2b8QMH0=_|1Zm2G{GJMqU}e#FV@l`JEmNzE+O_c&0oeGH2A3>MvJA=0ff>a#YWXGfN-AXUh)XP=)@9?LA
~vUUe=k1@PMT{PIhRdZW!V_ZKo8~H-
^9MiwlQ<2j4sJtjo0(FUXS<2)AVjW#>gj2~^!oYDzOlDo$Rab<KYNr)^6FXBD;Af_X;T2Ydw2%s_Cts3kadRoFsusfMyn1U9^@$3)bdYnO!%3x3-
<5=_4wJh0p08@2T4DWPr{HFGy0tbRR-xLxe_{Bx66vCU<mj#SRAm;U9lGvuT{fi)$Ff)mJGB5gTXT<460bAVCpk&~@n`VolmF=-
G$rl%&@b65|9Yqu0uQhtoqFu5w#&oy0)Hs5tnxsam!e`qkRr{_5Is<VWQH<!&Hn3aQ<w-
4=K;g+avlx;LM<GWtFu3diVH4Mrwu&IGwamu+k_yLnRg%kJzx})4%G1=QUO?#hlKEWZL^l)K@(9vA&9Mnx^hMu&BY@uz0T_M^K($rNg_Ch;d4khnC{WEV8$u
GB9WqzU2(eFb4_9sskSrrNrD(JK)Z&abX^Y?+vxcjUo*^HSj3GfuBh@!J-
)zl%h(bCr&IDgQ!uLLA2xiM@b1gcXTCzfzZlU38!KFj=ta&T=+_KuGMbr5AB>`W6<VN2zJiLWv6E{dxHHh;emW7C}ORCp14wMdDvoW{sF&eUq+qnNRLVJNqh
Z*2^U&t-
=1mTPGpKs!%=1RIiyqZiiuRFcm5uZN=}W&s=?E3KgZP0#RtiUuLSmvD&Ll*#H#BYx?)P8ZS=6VkL+1C=S?$xkjOU9<ImAeGV+Uv6Nbs8N7k7P6XF{Bz9ZbBf
6mE${H-UNYn;MXyBAsq{BUcGqb#-CyPjJpv&qIwi&gR(Fn9219X^T^EMNXSRA>o(-
a02EbZzqITpQ%rt>V@u>7(Jf(=sU0!CjP?Iit0XIx2uc2Z$>E+cB#t8bMLGc8lrX-+hdDPKeooi-Ijo-L4W}8)$STtGO(V~#u2Kt}g(@>v$#aQ&bBFuR_R;-
;$AklA7B!JUSq?1^RhGVzOyk}qze$MMp2ZF6sw;=e9;}`s^B72>v;rP|Q?9BSRAOWYr-=SY?EJJmbX`;YH;;M1kGxy#Gak>1-
kGy(8jYZ!B2Jge0jVhAxip#J><Ut}rp>yeOqCg6DVYXqx&%bYa6sPg(scRwo-
e?}8S?twAwF&F~AHClI>3*@#ML~zTq%gX*pN_yVTd@+bo){sOdPW}e5~!YS<4Stc-oh2_>OkyfBo#@mD=Di3c0WOOWTrH5@DF^;X2KH5Vq2RqE>$aCwo~~=j
g?Tf2-TMtQFK&HGZ|-DXh!eB8S}1D<t(GImP^P&q*-oPx821@uTbEv7zK3wumg@OgxW0e9D-EYQ1E{;hNoXJW-uc|)!!kRLKbgYf-
8KDCB6Q!4j}4uQs+J>2&sN65>!y991Uzfv39GG$~1UmO*Tj(ZC`wvy^%mEopdgHny?iFTx_scW+x3KLblM#f`qu({+s{xfv{1Xh*?uCdb59}Mq*WmR)8)(bO
p3-
fOCj==W%Tj7bnXihZpJ4mXF55E{f}%kZ~pdSMMkzd)Z<Fp%9Xcb|#(<Ta<2|Y0q}dQUqQ509cGKH@Cia;lg2T(eb$72~n1;)((Jd6Y=h3X9!`!j|{?u$lqM8
k?-&Jp5oWC#FmG~o%=bROU&vHXhqZRv#**ga(v$Qz7;y%ikqN!?|zSrbSv|D5*=aOrdLdrU(fc*p}Y${Fhy!t1qfT2j}KrGsB3jPkuS{w6;_7-
h{3Q2=Av+6igzX?16(nC`ucmUzd2~1$h~3ih3)CBrIFNVvj|oI9!Oyk8innRu5|FmKqQjW?$tXHAdqHMHB1=sL+VSOn9_=e)mcJzL%oBDb?<Qg%2eFiZxI-
g6-8;Lv(0xZbcBV;-8#O21;MeP4Bx#A<BZ5mF(fB=wMgxT8V`Y`*wdUN350=ud)rl|_6)o%pV=r!hUWEcjdYkut3$hwX9`Yozsr)DA0Hq_i$4+K5v<)?-
dW(VS0y=z`}t5&i0IuC%}?(PTa+u-{O&9|HMHP#-z8XSd;JV1Wk@{W3Fs<e*QdOept>$LT^rbZ_}%nBy>0k+p_S*CX}BjYe+;+R(fgr!<gn-
8pg>_`p9h(j%<B8S)8!$HKGz(oa_r=4L_Miz(JUxcExgkn&Cy#~msa<ad{J9j_2!vz>rw1+Xb8CEf3aruTV-
+YEx`1@ZgVm@<vNqz%UW{%daNQn>El3jCGq7tP*NUhu)K{6WDd*TCJvQ7lt-r+2h?h=M?QFk%l$<2DVWDpMqd`bfD29_e%$ke-
0hgCea7$i66gs90}dZMYZIaYlZ%p49NOsPQt|c(YLw+w%mXjq)rX)Iwo=tQdCwwK0x~<^B#xwBeDoBTEHG?F&zFPU0qF*!@+4@{lV+n7kO(MoKH?~zner6$b
auM#61n6DVomVZSxqN|S=tNCjk?USeONF|a}D%BB1_T@Aw$8h8L|gX`#mGfys}G#e04<9KqFt9AI#fdn4Ct{U(@gPh3k-
RFBNkeBI|5{cxt~I8OOi(9Vn_taV4*b^SE>mpUdVpO1mfX?4T!uvYC^)fe>7SNFHr4N70govv{PWp^zuG?lUU!(#n+jiS(R4W&&jOzoIUKlMzV*Wvdx!2S8U
mR@-W;vf=Pdji@Af4j(o-QFkzPeY;>uFzc0EY*Vm~f~N;aQz|ADtldjkg<-
ec({uG>O@H#_T1wCvu7+BdW`p;3x*L#l!Mht&)ba>x1q{vtx5UX)&tCzur=oM~=W{UdT~5~blqm+W6(kMy6=4_4ov*KBh%eA;n)ouv(+)9Ua_og*LW+PM#Yl
1~q3p6x`?Au!)aXB;E3;-
z*}LxhNq|${yN7FXiX57fJ9ks~dqzc8|KFBRQQm*WK?sDF5`=cgbY+=zPbNmxWPT)cUW@K$p3q7}ZDsSbQvS?*5}5+Jc+5_)_u>`TIQ~vi9N=K#*go4A3WK|
s9HQgRb_9Hz;}=D#<JE0&B1-1w_IW`{ukeJnm3J8fKHG}o#SRsAE}Dxs8eJG@R@?3^bH>UVi?#SMsrbz%f_EgK-6?p3v3@5%${v3K!y~m9!x@qk<+->B6--
;0t>6dhDOe-I5tE;p_*ATM?<O#~x8<SY7>4I!4`_vu_}OhPajm7Eb3A-
Z`0V9jfVYi&EoNS_JMb3yAEXab^&Vf=9ZuZIZ0XNrGoAsR=zCU?1<+le<$`$r>FkZO`k5w^L66p^an8;mA5#QYc}xnF17#Qs9G>xN6<qLYH$o=|5)ASZmbz}
PkhqO8trG&Y70Fm1=&g95_IARu-(!P;{+Afr&V{<RilV&mA@ifbPQOugZa;DH(v07Vi8Cip`Iqe;P0z$mTuld?mSoX0-
1CmAW4#gqb4U09O~1F2^_i&^@nKlLjgcW;`?8hmwtgK3)!>5V^_mm9nK0k?Ju~Wqv5#8IR7P5Ue_(8}ZM&w{&cLZgo^j6UT9?~Vd$^vx)VTt2XiwEHtnqnRk
s=WFT$R<^=ZOpvra^w7%m^}0eONyLL22r)g_gM}21dx(^5KKI(mWI?@Ic#gX%KP$(QQ-
O1;jRU3nUWa(BE8BN?zPLgh1TupiLgHAGu6Td73MWQ|7L31$f^Uh_4n~2n7_=LPKW1;94|H^sb<<AU$mHgB~KCQvwmH%DHK&eNM?vz8A`eK~xbceNP=FG_N0
#m~t^lXwJLRL?`Qpqu;k5?3%HT@4&Cv;ITJg+n#VBu{IBn?M&~EYe6=wJv6<gA>DS<_mx=-b!X5-
eu+q3A18q!2DB9`eQvEbDsu~_*c1F+N}f~ZB5oK6EmJICXG#v}s#tBFg<u-&Kpvo}_TvE_Z6J&yaS-x-
V!ig{3zF*mWt<H}<ZusrdJIk|vQdY;O_$7A*~y4W`o6C?rrK*75z2FQd!PJ+*7Vnvrp$4Hf*V5<W@&@5f3U3?_q<kIXdIYPTRhf9rQQ@cD4AH%Lk+~3ZTZjX
rO>d1Zic?gK`E;MeC2$aY5Sq=MK7hdsV4`nKdlRIR;|GH$Vja!-
{>>I>M;ru4g&K5x8n{!=&`V<83S?u!ebp^HcmBl$%5<K+N+EiNOn@3LDY{jmy<B8jE>J~(0!%?)LJImcosQOj`E(=_i@G>rB0lOG#t8s*5+XB=2$ZVZxCh+=
A=sqFiVOLy>h@yAf>pBe<v!)dzPr+h$F~bC&F-
uaupqZlId`E&iSN1KvB{;$*1=&J#1X}Z#<25_iB_C3*0Rr^zi?#mFBj8Fso6ttzF>DL)Z{dMa%DcxXo3V`)Hgs6iZY(^_iip1$!A<^Bpx;CIs)<Gyxuc%uq@
c)?uT6fH(5I>AEBZ(WP3VZulFHbMXuz4`2_p2EbE26C5e-c<T5^ZrrP`Y82CxWmZ`GJx!ta`6C}Tduv2nOY<!l#nv@x-
u#eW`0Cf0r=>=RFf3w{c0$OR@MlWemXVmIH&SbU1H+O@ZJWKY?lp1C6j5I;Qmy3e$GRqdr@Gp$l1l4zK_y2yZKEnZ%T7sB<gq9q@=8z(a)RlS{)7+tjPKU!e
kZ<^I8>SG=O862R)J5cdxG_(?!=A}Ks@~U7O#Ye9X0n^5&Xh$RYcS7Ou)q@s!x_JIHBjV<hq6tMwr~mhTBkpbf=vvz=c>vKuJ|8_9yYbu%OC9)c-+04vF-
>e^vsINt=_;#Ih?W9{LlH^zK&FhV<WEU85gl-3d**Qf06wD%tAmW%eGkFOFes6-5vAQG%-s9p(szPUZXIBo4<>xMS<#@)mKFG~OYhAmN-
|(tad@q2Bs9bSOWNp1};%bBT5P7HY@%d|OODeww-r#4l}N)s*<sZJwM`7!AV{%W1K}kqWdAlLj{FdM^5F!6wrEC;_V>a)p-M>1IcH&WhCJejz<{bX-
b^NHhBZHvooP$-)kspuUZeSgI7jH!DRjV^hi#-}#^leL*I(vr%-VOP2&Vc<dmyhas@qI$`wF5GA*TcqDYeD6!qs6tLm!Zjx8&m-
lX)A|D&SYBzio>JYcC36QTw?6FXsjs1!Ff|wOFduv^h2V%gJILGYKIrUN=sI0#%@nDUu6sO5xU`A~C?pn1--
9BaM8^yt8#=x>0|Kah8)ssvTwL7A&Kyw!~K$JvdkC~e^?0)uKJIJ-_ch3(4$DE#h98A|$>U0}Dg9IP%=5c4kxnxXcCwmt4d4ahQp{~p<^?z4=j~NlTN=kn6p
JR9J!!aUOex8F2&HnV(Y7b9H45(n4p|mX*zxgX$Z7XlcuYlI;g_ugEfMAu(cg7&<@1tH;Hy)(=KF4Lv>IUMiU&Pw4vH<naFpC-
7DlvRJp21O^XFp94c~~`Y<2Rpf@%TS-)#qV_g>-
y&iL>&x<3J45Tgb<l;1u*722J}^>P?lJ2Aq}h;#ZBQ3dTU=ig>wS{<Q7%$*Gq={hC)i+6BkTU8J2nr+{<(kLuCcEEUOOI=AWjTP6hfz0^OlM>1n568c9%Id`
LoQIrAJe+R1R-
TNASYCsjdg_@y(kv|IkZgRo?%!hH0dn;5%hDZLHUfX2xg09r^2k(AJb;n+Goi059sXoN^ht(kfolO2O>J*}g+P7tJoo4!nNs<&@dLp8fwp<EXBun+x*qHs#v
+$@#pa9m3mD_;jYt|{rTbHuJ*rsJwG4O0i`zX@|-|yB;`5cl9Kv-bvpc}QkMcjlr?dZ{8U0p%0R-(#R+u~t0vqV?)71%`zvu6LY_tL+5fVA>hgLTE-
4piQP=<c8g>gK<P^~frJu!fS}BKvc}@LT7Z`0H)wQl+E-;yXX1T-Ku*s;V_R+0l>H-oJYgS~uP-oGCMiLA?<#i|+<H=p6D8K56<GH(SievE&nKV;BX08H&-
WAo9z8)|oIV0CH0%?nHZ5A&8Fm1cMOkFN4bk^U_DD`Yd<8>6{EMu>%w@$pzwrlbgSXuH&suBw&QW#57M%G06{=D@w1Rl2#W42bynY*b4kLWjII80Fz14zuiy
7q_B@}sA^-
Y?7^vHRmw57Dni2qjHlzL;*=4Et{atXt74m1^81QuDU!)&p^qJOAqp!W<qD&{jy4Gf)K7r_Z%n<XQ;q?suj2KjdP~DsC)h8N=4g*;V4^1_(Zz}AtQYu>Sf0K
I3=jrWA7Y7s57VGxhdJAka7;dBa%L&JQ{Yngoml1{l;A(c%}!<3DDLUP-8-
t5@8itd|J%0C^GZcxI&HakC6+o>#R;idSOU{Zuu*YE+Nifn7?K|dVk{U?=@cBLH6qrwXW@cOU1~r8f*DaUvqzq~phA@Krhay-
;A(OC`m4ycfc{=!pctRmy%tlAAQkcgI2Tt-%Jw)3Q-@<n68nnF4Iz1$MmBqNe_qg1Vb)@W&@ngd8`vc|V;Wk{-
>hGT&H$<_(%7PsdcY1Z*n0LPe%;EhQl1(^(=tfj=p&Mm>L#WFL(JB|<n5zk1hB+LJ>JCLV?$Ub@7<j!Nf3h3H5p%Kn`uz_rFoaBrTdZnqp(j#fuNaefLa2z$
M080b3}(Oa3Y}&DVpdi2om0PiN<xp5UU$wN7zd2;wu|{HrND?j)kpIpnLx50WQ=*FhE6Y9JY@zrw~WT6A)OeH?27DQL{bxiOI=yfv<`o;riTIfK38Cn`f!SL
=U#?Y7TlNGktSqC;e?V?7U`8q+UWvi)01jU^ODiKwai!Hl1}8px>;Dfw{P1GM9a_8-9lcu@_>;-2a)=l22?J3_}VLX03%}-
B_0|XdNiC#$I%~?%9s}vce!t+BIk`kkKx<r>f}(iZUz4sLHSxtQ$hB1&a%JhIc@7<7}7~Vp86|LsM6nN9lxB3GXDjLp=IOCF6U$VBlnrCQdPs?kswwrq1a%D
PD=We3bb(qfcE7)@JeaJu#YB^L>g+!`#GU6+=h+Z<f@=pYXFG!L@Y?fkm_}l^k7AxOF7=5L_O8h!Pg_4ZAW$P|W}39vT9<{NTue0{6o?*!tCpz-
}53P0JfxND2TMZ36NDr262PJq-dEC+s<8N1Ub_bf8z5$T>=<AF-
z7xm%43kIWAv%JJM;=q@DOgmb9n#V3i)L*ewmDxliOM`mvWisWdBpHsVKSHezE5D6agS8iZiN2c>Fw`j8`!?4<xd$m`6Vcu+BPs0bb>|fsi!xxdfBl;`wjX(
DHhT02Q^0o@TPTA1je)SgRq$HM~#VRpY(|_;C-`XBZ!;5Al``zR}@-
l6G*cLYb3FR^st~q5)W8xhbW7n`75tV5Isc|LGTJ!9t=tu3VN06>4!BV35Sc7{a(_imloGOaU{d;!DLD*P>|HH;Tt9AS(e)3&3t^~bN5%t;lp96dgx1-
bY`z3-eyG1(a03Avz`9)<rvL{E-!uqJ;11AGP*!`1CbC3lsTy}Z2e;eDQhi3>dU#MSQS3{?@aY0W*WUrt`^kjGeJH=cjBhZKUN-nlkM;&FQi%9gR{hemq$0@
-`IYVEV9Z#WF6tuG(IG*;2y#3^ZOR0J04&M}#jKR#T;lv+IBsBs4xEnXQjMs?K7Yjy&Z%W2Lg*JLN*-knj@A=-Nz?@Aa<UJ%t&Pb~&mF&UR#V-
RI#OepsX2IcaKca*wPXg1J(Fd`&mly$cpT^^4ar|46B;l^`tJezMXuG>YPB8cVG2`gpRI*g|h~`d05vJ6+IS2qWvG<LfV3mg1x7KUdx)>I6LYfi@5|!2yzjD
!4-=P-`@y-
J|AmA$6)r5!Erf_sByhI+MISF#&px1R`pykiFi4k;??Ps~sQ2+vFnwnPiffu}#Mt!f3oon<n%}Qptf>4$Qo7Qo=c!@8VWpFaI1i=lo9rbN)O{9HRe{l*cG%2
91GfqlBqL#(ygdUf6d*dI`5|i(Ix##Y&zB=tvR2(TtjjLViXG)BcUzlxPkSXo<Vc3V?5IVBp(nNMhfUSfJ>6yxKi<ihh+jLjf_0?JiDmuBnc>YigI;O*KHBD
Z&mAHk`ZtdO5E_+zUw8iTdAhhyB7u^^EDIU^W-st%>*O;II%u((RUW}kxH;$*bOS8L-p%nk-&jTiCJs-91lL0fT)MsLU@Fmi9bvYpW6M0UFvWb-
j#Q16tGG<ck>F%}FF-rWLhvpbYhi+Eqc?vo#O9PHPlt(HWEaGKFFhNfJBBFWOu?kIeiE#k@v84ih4VyrzqRs5lE;jFu)57-
9fqP&*wj4yIkgIW?AJH@x*JKOW>R;nhoW~}BYC90O7_`0V6|A1GV?vDQERlE-
8yzLe^y**4Ab*|zuwV*aC>X}gsLFfIq@r)*{3F_odo`=z4oMeF^sfK2%fU5kl$B#tiRPXEMvg<)&lc*RYf{j}V=!T&Xx7(L^Fa!k&5YOrjp}XzO>K~ORV4~k
6@HRR)9xz!=T9)IXNNvU_LJuUaoA>3N%i)e@DDI3(K0e%|8)uti~XDNfB7XVs6;!khKb!Q(kmEJfQr#<Rv}1)EbwARG1$se#mr_rnMt23rUtbEoH*yBEcv$f
l1gE{oK?T+T9E2IKOa`J>gky=jgxe-`htJVYyo@L-
n`X(>kSnNSNb>2w9i%csRdIWEC03|QIqG%Fv$pLAFlV~&iJF(bG%m)E39z%3B82x`unl?5OtH|O#!l3U@Ib`n<6)ic>!D4aklzv(4>qO88z|m)Im0)6lU{ct
U0kjxqe+CFjViUa#wJsSWVE%u+fm4*jCm&X1g|JT*Eh5)55e=p8WLz-
51qJ{+AoC&9hI}&ppw}&D4DVc_Xlpo6Ef3t0$AfRsT);z#xKS{Pl;4TjTX6l7YYwn9yHF7-
^TBIlL=xKqXbr*ULP7`>%VfRMWpcgSZd!+zwY#eL@5qsT6SVbqF6kZ~HgPOGf7t(H8ch?otO&xe1-_^WgqfYO<*xxKUpczKFO}gnG-
ZXRw?j2+bllg@D&xzcHR%@aofFv*WQV_x*4&i>SU@8Fk(Mb{(@evH-Hl+Vt}-0hyM;dfDw-JbI1Fd?#6$k#;0CE<1XquZ1AjU+Aioj)1XX)u>vBt_o#pmCVg
gCkPywg7tOYlw6EEVkFQ7k6-4PBIlEn#V%zRcAU}-(Y_;Fe>n{d%D-
d!tR8%58Sn}N7gG}A7<s+kvW}Cv0th1mCy+`X@wRAl+W+PaXrRW&CrBMhffP{*19b_hSnatcjbsK5d>vTs*c+};QQ9vZG}Kg>?R}F<DE;6u7ryd{lD9{Tj07
ZYGTz%xqZh?){I~v+jmeID*O@&!1yGCLT(kyub`T+76cA+a>9N-JEc|rR0DlXksX=1%2nr!g0FK{9A~<(8+sE#1?BjYd+#n($xB-
H6$ltX$vy%F~HU|NhkA1|^Yh2=6DOM$>1)qjPZTV~lBAks;oht~zo<;&GP_qCDB|^<8LVN3yz|^6DYX5Gb=!nqoph+bw2Pv?jW7oXSVZa_;+lkYJqZI&E0=s
p}G3v^o&!kU><u>BsYI7&`WbS_HHZ}AlEFP04g8;Gmd)Y#xeR1<5H-
`OqEe3j}`yuzlf>DSur4v{^gm_?{22$t%*fE0hP=@3G7#}*m|2*k0G_RV+I!35TxHP2fs5I!QM5TD)sfEPF!e#rz6{>d3TT>zgHMy=p=0}raCWhOWXC)p$yk
s4OFLYf14}d$Z1wrUOk=jo@j!<r#7+W1lSetFnwF|R4L6=Q^wiY>kU*>S+*4{7-{8{DpqI9;Jhjm!z-1vIO0pD}<bb3LACi1N1Kbo{qD4Xsp-
^aFxDIDr{<^q9^AP2q5o9%z|rmrWseE9Z~G@a+EHAIBJ<R#Brs8q%tZyyV)gcHBBcjC%25a#^N%REwqHh$_3jeOHnIFJRfyr^Bk3Nmn}DP-
*`aU>6VJslCBO=XAduygRx2ebmo_~-fGi&7;tA-
^m)^~xL~(EK(c$MT58&5QAp=(;_M>CCxo4I*&%YKQlTSCOX`&4KH;Qnm+t!7eyS;)8ZxRaI@S4mm3^hsKxnirINB9*?7?&(H07Ss~b+`0nL763Zhn`OvD6rP
*_lZ%N{_5-
ozb&|YKvw7n|QPvrC(&%pVbN<e=tgu#30CY%`@=Kgr?1<qpF&P17_n^1f~J;l^Syy8;saqcw)bHzgU?yCMH5MW*+yQanUMQgh>O10_Xihe%0ahiopg)c4;9%
D--sn*U7bY0#c)%97rYH@1w6vbRM6V$moiKpm%VlwkSWchmo497HkC?PkRj=opy%&f+!YBXMf_GJZK+R!g}eP^PxuqB*ApVS%9K*1CAc5)T0V*ULaOrPRxSt
{u-
Huj^GZNnfMh;(y*zJN|i=s%oAJhr`5Prz0#4A<`PVJBTnz#a1=nS(4`>|<3h8@TCRM*9;|n#qEUjA+VrL$Fd>o@l*ceMJ!4q4H4lr9nqmiy?19NwodeO+@aT
um_Zx^1^e(Qtd$YmuDw~XD`S$OBy{t8<uvM0>j^u6V->-
mIL&dT2HDoPV(HZfd~4ndNvEQw+%a{YY#nZkA_Veg^vy9uO~C(JH6B}Hn+BoKm$*jUq3{pdZlIf@kD7#>Ofq^H){nUJPyg8PLUU*^#)xBl#ngiB7VSpa0Gw+
v!&=m#&u3w|D*Hb?OE_;#C0xT>b=c&J>(v)1=~N<W?w-
yeNlhO{gH*~dHCijxT$qOiNdQ>RlRh{`l><4l3M*~ozV<<_eJR1AQ8PV{f4M$<n$F@`0RfE;26KXG6HfZ&=SuC$v}gn^agF6_JIw!l$!E~z|B~?@37Z*i5Gw
ck}y~fYGK~1KFV``tdwm|(J}|H76_(Z4^~NR{gtV>kWmE+{mlO2Jmbz4_SH@rSSCr>kpkH(SSBv0X1?a7bT;;-
bged<Xe>0C@KvL^3OLa1dHVzsxrIb!B}!W((ONmYU+&R9my*eV%HKh289#JE9(%&|AlTt3ot?I+Klp<>K>G#}s)lCXE;d(gq5qllkh$)o6__@!Ys!M-
PiImx2a@45JdT~^ol_~A%5uUBmjDl-8@vwg5LJLC5*_G=?4ktWTi-UmOyB{flT@(G@ORs9LvmCC{w=F+gA#ImOBur6j4DI0b5U5*HEQZ&hioLDhSviD^HzR5
{_O#FhSjbE=1=Pgl9N(dpdUvE$+f<19Xs>XNnNi}>L`})BZ&bK;Me*uY7I{~NuB>!OTU?J|9*1{oh@f2iUz0XfU?kB?!~MT;|OIxEt89JSv>L%_80YTSD;o5
!R(4;M!@u#8>(YXN)5bJGPfj;gH1NA=d;)Y&CA)v%$n>WAg$kfIgS)_(e&50QYtr^|K6T^*qGS<<7vJ$PPzinj&s<i<~G%Uerr$en$m#Wc@f))3aHvU&-
M51n4qYCM}s}_H_uh}Fkr)i6T<pK^GjvgmiRIj|JK>Pz~Iwh5VUA;agog_nvR7ec$z3A98R8P9<Ob|2DD=K3%`F1B}%Kw%r)@K*2*!{{CS)+gj&S*@=j@R!Z
M#snYkK`2XX4QSMiyasf8dtz<ev}2yAjUn{d3PVSb5>DzWo;0)iQS1tZiuUe$;iL&)HWa`r80-
C~zVK8hxiv$+Mj+Y1E<p;B!v%*U|2y`NnapmkcIN0LwjM;)ZJOEP}UYzZdF`3G+0vSta!p&o+IMk}4(eN5Nu+N7!7&__dGJs*C`4Iw>p$t)_cSoOzug!cbzN
+p{TO9C_AtaQWZ%D$XpyL`fw#<xmu6vpt53}s)(1~d?ii^XKy+xeG<aG7`|+YAXl`(k?!G@jE6pL|Y~qyrjV1li05o#y)N7f#^Vx2a>kpfeRY%k3eJr?A}_6
NB0HEQXnTiUSRPd;w>$=pq?NQnWk2o|4)|Q54-Z_O!Kcnk8}s)e~S7PA2@D!YH(*iC13@h#J<Q;}Z6PW36BQ;kmY-
yTcs_voPcD^T2xSi{>Z$S-3$UwX)8aJ-pv=1H~Mu=#J#V$A24$AP$(&CYjR?Bzht?P|FBxB2T(V=_5`SiBm<!|G}C$ipWeDNthpJdk#E;)ms14q1&q8O>F-
()`PY;PZ}*w!Q|%X_Yhp&!OXR22<UcVN}ifqx&1bs%oEsm>Rf>}a**q0Z)SPV_K?5rHt}q5NG%?8W#Uo*f;&PPfo<DY>f+;8#lw1#&w7{-
*3jCUs|(%LB|H<o2sYok-vjStrM)hVusgS)^v+j{DuYJxR9UXGNX%OBfWWQs7yvknk1q2GvoN!%qL-
F7U!J3Q;18Y(=d<e17q2#W8I{vIJpO|WI0vW5OL`g2bud&o5KP`mNP(l-
_iC*m#&0b~S0o)AfPQ6I6seNcQy3l+eUU5Edh+8BnZ69D8*^lybKjE0Y6qcT6lN|QtulhP@pTT%uly0C{L@PrNOYojJ=E{xuli#2UUuXw{*du?KP|w<DCq7B
6h+yK61B9D1f_+Lb-%R17a#*mK2S^#XQa23@YsQwwMjcv#uz2rJ0AP4>PLN_fa5g7{Q~4$yL`Iz>DUa*zZCNIw#`cqX;-
$|oIb0*h64KbO9+Z^c{)6NUr|s7`#HU=EdsIU*9q_}5DKA;R^z~^W94k=lG2tg8GyG4FmC6Au63YrZE#C1@{>cL#nv)lb|UVQDecnXeHvV8!`{7<mI6MkAwL
fe>eSJX%E9SG%(l8g?`B#@h?Ggv?1KPHvx0apB;Zn?P7fao8Wc>6Tn)z*cwAqYLv^aWexkfAGG{yoi#?_vX5vL`42GwzJUGwkFX14MF1_=Yh|T~n%jx!<qCA
=A2_gsCy$`mq;&r|Y^KE?MaEnK7Uc~NM8qT1XuUdS`6%*NQdU=Vd*8~N!b?q>cjq(d0_fh>`ot@Te;OdUp-
3p*Wxmx#*E^$TpY+Tdw9`Tk)<p@X7Dj{cJXykY&2qaRmb{!)N+K}<nflm46%a%n#LfqQw-
l0~ok!cDpvn$r^EpU;eS7jjz$sXNK(3&i$2z=if#g6^&X5nmM3zLvjrN8^M6vqC`2TyWkHNiRv{m*7RXnTa7)6n{qqu07#OOO7fKQ1Px&lT8Q>CAlZCBk_GO
AbjKaq!!550~|jHyR1sW8+{3gNx-gDQT5-
6mcT#fQsb1y2Z#U=!$P?DIS5prwzp&lJn#uCuh8GupwD(g!_n81xN`bdmj)N$cyg>E?^(m(RWWTZr`pGFqlgdyv!$F4VW@mn{z;%VjW7WW`7!>;S3uR&JCXd
)=)Xb9C+ppc_AY^rCpqVdJIf}4~y2zQ^vko4QDaj5m9kyJ{0aUz_l2MR}m4`CVb+N!G;wH|0hm|WaAE~V)Dm}0}z_>Ip7^7EmHeiZ6KBE;9L3(UAtglLrj~i
DCEdk@+cb8aeA_AfID>raa7&AGjTZoG7wEGorZj&(_1He6Tx(--QRj_7kjEG8A4@i>y}S|9KUmiXA{cy2pQ^7+2VbfP?Dly;QQ-
$N8CyoQmyPZ<!34eZY)N9CQs%~@>=%4#$v_uuZ5M&ia8&Ca=Bw{r-NZzp*JoQHvPimx%6vZ=D25Qg=I)BwPZ_S5moi(e5m4!RVX-
&rbUs$RfV=81aKxcoKO4O<^J<7K{hkI+8lZ&RHV%|k5fqz=MSBO%KfnjQ2oop%U5u)`5@|Vl$kM~28_5P2+P!1G_m96X6;U}qhHFkTk#2-
faB%X+7lc+glL1Z7Wz=%JTE)-
*9RsFxTe8g_YYa~TvtwHu@G!v_z94sH%;4jxO!b^x$=_rt^_jJ2Tew9)y>Def`OLDyuB`ja&P~1_lEKVm3PBKParsEy7{pw`cu|i*b0;3&ab)0mBO>loJu~(
6+_!-h%728b$b7qESgcNNv(Us=Uj)Ifh=FL?~oATy)6bbvciJ{^~6y2Ey6Pj6t5zk1uJld<<*TljiY}B;MqV;a-
dSuE!i?er(TB5kq}}jv$6A{f|xZO^A(%7bx(ANzOPH?I|^r7uWZSpF7G?m!2~v9F$nu*{1rqj4V*cJCp?C#uP%mwfy7(Re?4QJP5)GuWa~QcwFbdM&A<W#N(
FN_#4^U-
YPQZaS||NB0Au&dE2Z<7Ro4*0ymAoyESh~#!H%ah@VlL*)n5un!RI6L=tbd1*r}E3_!Vq2;h~S#coE$tL{SYSe~eCa&McGJ=c~RYbp|Kc;Va1u&H$`NiF}_)
?gd)FC}Ff_s}SspmtvBWEF8H2N=0k#IkP^hluDDSiOK$L?Pwbk+hs#><67^qsUakuF1xjB>tb^%g2Y;;AFsP{OabSG)%k*`bt9$!0~|v!Sx-
^aDG+CvEu|y&Nt(gwjC5`9Cxl?Gg)6%E0klPT1^zcHTOf8y&$f>i5SsA7@o}lr*Vl4Q<A2wwgIQm;z=imc>VNKJ5#@@Wz_D%X9c-
hylL5d>I*%fh90DDb=sfq|OTroC+{dTnF1!9y_@zB~Hc<p2cv3@fZ}m=i-9lo;Mwj1C=tM{sTc-@1q+gX$F`sY%!$89Yp3D}xVi!;M=9T7$Mqrvqh;-
DHvgZSvfMb=2qF7(=>DPZTExLJ0MgSE5L-llqXR%|<g9fMZ*eKfu2>XgAN48XMp1U!NL?+-JS-bD0FS&}PGftDBN6#LK*N54{izS_EnIfo;hss}Pa<N<G#s|
hBm$Scqd}9;SPfbd2AHZT#7~<i_r4iNN<=MT6P_to2dJQ#vRk(G<Je37$em>uz1)&Zs$iF)?5pB3YVl@#}AU9~C4wnvvv+?ivW`+-+m&+3@Vw^6Qg-
w|lH)Y!ap#c_c{Qf-&FZi};KZ!+@nIs=(UoSD>Wf1@XnJo=)c4OP_H>3O})fLm6s#9}|KNpZzC^Zv1&402&&rTUY2(*F^wo5Od<{u-
Mt87{z(4+&$Olzs^`FH@Rl@3I@0kE8*u*IIu3^X*t{aW33KNdenyO4=Kta20J)G|~~Q%+|P-
JW7VVDC&GT_Bu^b<P5liLNWL2i{*TCMQIf;3FyJR<aBki%xmebBG?)26v3!jDisOK%=V+9?_<dSp_q8tb#C?Gr)nyRXo<6sY*d9(2CoWKK)YpZ3W4xRAe`f_
E7F!p@5AVa_&8P+InZ9L3g68p4>GQQsI#W0(`&cGBT?xrg^bGM3GQvv%4u(VU8%Ssy?9I!npJS))v~gB?IOzhSAFIHC6R$q$_4c^QyjNUP6_9!a3K~j;U@<k
G}`H@j?SqEOa5;Fkv7lfW!lUFa!!4xR1-FkXp0g9b9+G>IVdJ2-~WyuXi`?x>YIe*MZwo+T|tq<jA?K9uVZ-6WOORm7th%cr0f!-
*hu%CehVoIv1_5<O49MkTtF0G6Y6>P+Kir>8FC%cO}BnNDpi$7mf7rH>G@qS%Xj|ry<l6{%F8)3U1ni@$n6H*sounqp9-BboUw%GD%riYVKkt;cAn-
&;axlnDe1?k;XQ4vtB6h=t}-
zlZRC_`s6wOa2NlZtHk<HapaQhQu%X?;@=#lZ_?RH?kNsJ1f#E2MTDiY*k;9Uw>hu_HNS@GIs}x>v=e?UB>t5tBTam4WE9Ul#=)M>Y`1Y}M5m(b4Ru1KKJW0
DVd%LuTcqA|-JnzrDxyqcxwb(2E4VS^;3>V=tLxkH@_&R*5yyo(wYe|bX(I%o6-
gXUao`Xhy0KC5JggqrBMfnCp?@kg=*8@{>GSSuFUk&C`<>4%?1{APU1fh}8XTbp3rVWv*$ELj`0YmWl4TmZywuu^_9KPH3Gzvr@!rtUURpCc_<bGqf5D}A=5
qP?t{&bdA>{vQ9cI)D)l{-Onf}iw$-oZUQwS;O#&GESobpZ3KG%psdyeX^g>~=Tw5=&a?x4eTJt~6q9#-
Ct8~e?A)Fx6H93FuG(<*Qe9m{#k|6`ZenE})cJz<t@rl_cT%!G$A#xYl=^t@tZmYh_x1|5-
A$g4qGce|g@>rz~;sL+irHw5%Ka>MDrtiYp+C;!UeNUFretk8R^(37UX1Wka>-`lGT>Y=zh7P30~^}<Zu5o53-
Lmz*QT6#+T3tOY=7!^Z{{N>ET3CBR@jSKUkAm5a6hYyD>F{rxZTqcL!603rVxbqSV!_?;M9o&WsrsPIF0BE3zzIiP|qUqG(Y?!QfPaJ$#(4NqolVM%PX~%zM
OZ8YLg)#DlqcDT-MnOn;1!?MrHO?W*2x>!=hq99s0M8Zw)N<X+Rq5w+uqwoVF}Oo1#P%L?>)PUD`|`p_rGzSW<C?k7_@u-
&!Cri<*E|~uQM@Vg44Azd6>)U2y;JH(e#mgddAsq^(d^@ha{2VF$B3J|CCpsL>5cY|*)a<GM)(X?9@1;bg}d-
N_53ud`ezN1g{pCg60*N1?+*nxG<C=<Ubo|@azGHr;?~Ig&}ltq;F!b^7s*b|j+uF+UIzu4ubseBmS&`h7jY-
!2`H6sWT1iq1E=Am%Vhbp7rb1)+93~}V$Bi_Yyy1`R++&_r0KBwDiW&hc>K_ic^XVdfnhG7OX?YP6F2xF9rZVTv4+%x;g5o%<C_vqt?9=m<=2GcZu@>y(k>l
Lp)9jVW4huu5X|Q?3Lq=k#o@i!jwDSv5rEB*x)qr(_E;?qL_YKD45l+j|2xsDLWx)rZ8ZZJEtX<$@VJGdi-UeNyDxm|5fLk#Yy{e&yjF22k}ZI}<!&nFVlr#
P-rN&S;|biOU~Q|T^CSl6=1mM|#jCn>mbMOU9k#$KPt`O0_&$)uPQhy`Vuu3@aS14IziS~q7PzUpz|om&?mVVvvyt?|_h99Dq%V-mOI~f-
wy>R|HD!^9^ix`4%X?{Si5?}bRh>f5pznk=F~N8~T+x8Ny6qKrsx*F@E{vhaypIrWqL2uFVIi)1zwqp&(5}IFza_H`VH8i7b+Z0~%km3r<TZaolOJ~9Ph&U~
#6YK``cX(_Gb}|x0R@{&=?n`dZTc*Ip#Ch1Ng)<;FaB21e1Q_iga88H!mkotgD@x1zGstel}b>ZaWpMmh6PoU9<lcNM2yXTr&$Phv_0s%<wumKc82ahGSka7
sWtwT6cWECsdE<yla2~U+@<Q<B(b^qYOs+&H+XkCS&C8oog;7wT7e14;pW9S@b^gh#@vLy7)qc?zVfiO5siTv6AlDE)OEe2;#U+yx<oT{zXafUgSVZG@LRfA
w%Qd3yGI$W%9n=nGmQWm75Hd>F7v%#Q`jYeqIH;egN=4u?A!H=<(<q9_OA&7shvOpTRQxu_kPp+F-
tesWU@o{(z2yj^8X`eo`V8!BpYQNZp@fxSVK9`<>B$c7`@3<Ux2@|9nAV%mxR|wAI)|4m2Qv$Mj$7+V`co=Q^9F+BG?3Ew~x#l1=e*Jh1Dpy)p6h#>ob?n4D
BPTBFjLW7I81=_;48JguF4xu~}plAR5el1nI|#p=t}J9wL@2(LeF)aggbqRYY4+OA6O#WldLJUuNYGTG9gOTx%us43eV7c7GaI4m?I@tqTov$Ia)4e=9ickk
F(emMj$D<$7j^tzr?E<A30I?vSd1K)n@_#L|6j6#>u4XEu&9w+c(^mb(p=fpVg%TgajNyrZlvZRhe3$q0ZK#6M4}>Db~o95HTxMo~ahs5zhKFV325bP`(G+f
Jd#m2c{$iAyL(zM>8pY*UCs;P?A^yxz#n=&XfolcqyklBsZ~>oFCSe?#kbEbCHd3#(3xPqQM|q`cCKKu<r<`FWKFozn-
@pBLXx49v!BOd(?qrU^%TZLu4D`z0~TH!&u_x_CW{Z5u(rHtpf+_WUF_LrLL^mw!fh{xG<2%M(2qfgkcI#Ou=M=FpwG!ua1c&e;}l)dn#|6U}c?+R%=!)3b{
?hjCp$)Y;EhP(uBq)TzZ&hwthHha&<+Yb7$Dex#lWDSWXR5}V)x<m;pN6>uM1W@|$=u(Kb7L-4ze^GUiYmX16BAL-
99sliRu1{?{`_k(Fj(j>fcJkTqkYLqiB1&SQ>7a@WWz?t~tSnDf^O1iy=5|th<WY!R|*m!A)q>NMFTpMy$ii#_&P|{f4Y_K9k_Q_UK+2WhPqAhs?@Cc-
a2vlZGBEKTcC*|+K4m`F+${P=J;v`A7$~#XI!@Of;Sh(Q7IpN#$8IFvBm3%cyZChUSPp*NRBk^}5inV6;n=jN{6pQvLuzW{n6Lj^TZE;`zOaQclJ<5^XU}Y|
1Hdz~8sLs=|G1Kz9>s=Fbt^I~f9nyEU8DR|U$PH&tOuIdbdC@;5$|d@-@P>vGCsR=zPb4MD=O;7_CC!&9V}Ue6f>thX?sVPM<7H5n1PnS>`0>SSVlKo-TsdY
ocfa%ZIPcK+?CJp7*+X~Ubs3A20VhkuzKsR$(7XAt2aM@oVE*>D<^qpykvu=zaDy;!Q203Tten+$DkEmn5}Wa`HQF5Od)FOXR9}%sY~44DAUe0arE&8K7704
nyD}Q{fMC~VJ#`8(>>{!(wM_GfX0%6JR3pW09S-+({xVa0Z9rAz5n30I!1e+BZXxrDuW1j*b}}SB4e#7@-
EYh&KJhsCG^eYmqooq)J#<UI7^0u@g%x6u<w7*TjtU6(stw&^@Fe0V7W%$KCVgp8!;xZ3-
Ms93MD{9;Bux9Vv;lDbDmkJ+Z?Ep`{Ub%&n=g7+od^W83f4SvY90>X{|Jk+k&JAHCq*R9QN8(yP0V*WjM+6idCd*4djFJ3neEV18IC3-
`8@5WG+nPEkRut5SPWA{z*gHm<a)clptTDQr0{f-<sB2;u=urRyuOeI!n7(j>ivr;bnQrT=`Om^Eq+CJ?;vM+ajDDNjhjHpV&Yb@3CHQbH!=1E8-
{^eQvz8tN&G+c-{^k_P3X~aU#T2J{>J%Q3jf@E)aFRZ1J~sLF!ho!#=bfIkIV-?Yp~>+C5oX=Ri;SP4oULmk1?XL$QW$)nJ$DcLQn;yZi3>1-
B%7M$W_cwo*+gF%q*7>2=MJ2rBhpEWnZW}!MpI9;JMkLU!|t=U^hFZ2Nd-YyKCJf^vZ=xDtG@TZ4Azq&8P>V|18k|@mZU4zrK8LK$(Ex+hVFDh9bA)oCfW(N
D}rCQ@1N2`4ZUOzEZtHP??j{0#t0-RLcVIsugj<0y}+i!w}$9T>@2pszz`1KGu{aT!bo3D11OKQaq1DB~uWsGyDsxz_FJ(wj9Lq$o^$(et0-
Ql6P^ActVdMq(JU16%59=HOl>=^>ls}q7qm%cN6a6d(}Z<dw4B_q=QLpZ#?3W%L+Thpy4f_egu%)cgAt<XsE;8VwNuP?N^IhquDlK&w0q!hpwtj+oCiSA?XN
;L2gT##R#XUfb8z8MS+00uEhNwh8?~}@SJ}E%DJb5gd(2l9q##V<LMW3i^lgLXCe{YR%C{0^||W(kX6iX;Am^C@&;lVz4vSYOf_mGzPPe*yEf5iwY8{(u)sZ
5_bUVb0?e#-
j^j+cNYx8hx4S!P@DQ#Y@^j~CT)Z_ziE+KT1pRR|a~5BX_fmvsL$?EcPX#hgi{YzQ?csO)7oVuZ(Vy(EMNXeI^x}`ezOOGsg%wWItfNCapN+vK_%k*LPMeq1
D40dr?nXAF2ZD=N0pwZ*;FdwF<I-X2HUqLFL>|dlTJh~xl_cdkfRjEh(xg)ocol>6zQJ}GEN8{qRfp1_M21a#ktCK~JVb%a@gz&-
de(EQd+>c&m1G=SDj+E*ikPBRT|%Y_q?}oz){CkP$)8@)@+p~ud*{4j8V&1&(uX4*-oI0i!iY#)Orl@*hcH;dF95>TBEJ?6UCl>}Sy0b&D>e;-
DAiMOjmST$p9Fxl2aA77%Ms@dJ{t%%6~_IViLA1d@O1*!_^`xR8}rEEcZgE#`tp_Ow@%3EXaxTvFrM01_aPy`S88s56`KzHjW$hv?A<{b;ZqmrI7M-
ZmUP1nuJ|euUmGvfk4@E4d9JEc=lY6cQ`<wIS@%>DVJ2!;C>imJ$`cT-
2p^CnbqwV@TLXWiZ1~D4BmJh8RhP%(*{S$c&9ovp!ptGe;#rZIgL2<B#xy+_abOZ#3!&JwKO}$r@Z02z`KrZ4i7>P7>oL4FQHUYWCH8AwYonDIz1m}T4sq3e
;e~JYnb@N9Sb!cYlndgqfuvZiIVRe)aOV};^VN>w->#-
h5>6M?RenT^76sxIrFV1z^CsNrUHRWx)Q}V4#ET0~dsTUuD;b|1seo=RmHxv|(Dzl;3lbCKUzMpbAv6C;eDn$Wil@?H)bHDP#dizSh#FnVa&^_?=-
zI!AY^_+FHx?tU(-?RgJiAwu#Gspu0aCu+VQV~AZd@^gv^#)&V&8h^`oN{sJN``)W-
3$MBS09OAwHK1;L$>J+Av{)4A?lOM8$jn^X#5gD4MA6&aZ2XubF%^|GHM+-m#@d_EO=rA&dWpoSRk-
|1Ac6={<Kf*0N{{AfmgG@;(q;}pFy>E`ftch{ussl<zta7({YDj;QwRVjU$h)(fc*y@4OngjkSU+P-1szftS6LC|xXKynMqzWJ%ijgXUKL4X0BI-CRL$QyOt
Oo;UMpCFKSg;*Xr6Q!WQn6=C2+F-
pCpa|2equRw@{&{vT!($C6a;H0B_HBst@~6YHKZgpkfILLdu4CJ>Iy#1yXn`zP7ZK@D>QRvT3tGnctie&gmtLh13ni)3ZRDCFP^k3sK?)LZhshlH}wI?13pm
nn@Zm5HK_aqw)H7&lHqF`F*K9Q6WSh@5Z!%&Q!XK`UA74_dn#Xtp0s)aO<eF3S+01z+W+pPUDU;SyJi6pSqixb<CtUraQEe-
l&mNaOn}4M2@<Yl&zsyE^&+eigV>foBy2-
Y3Q@}<DGOJ<`OCe+3sDG~zBB>e^fxmrMR6X%0rN6TjI8?;#!Ek_0p3UonvjU7d_Fx5hm*pJdR`r%)DfQM7RILHp+WDvL`?$;pv)g$Uc=`l0aoTzy_+TfK;R(
#%j=<6a;+UHuyyZv11}pzmzc(iblLmki;w;1+h`+0VHk7PpLlb8ebf6N<UG7^M!){E<4QL9rHj<nJ3J{n75Hfa&n-JnX9{fM%~H5s)BmLo$|%4e!OiaU-
1~YNaNBcab_oJehi+Eyq>wl`#4~I1i;QaHB2jSED*)Nq_bmT0ORqCfyBb#l+TnCL?#Wb(GQ^<GlApOQ@=7shfU_M>2Bq1Bu-~gJR*zJW-
aQ*tvPoAT8!}sQnaj&D&}K{f!zQ`>&Q;)Q9)krVG?@*F<<=k2+B#Dr)6ql)aAp6ekto-GfYBC6)XAnJTWX`B7QXs(@a@OU4zq6|^~y1G8KaHQ#f)#Nlr;Igm
D7wBPh%>h4{laJ&|h<mVz|@aJiyxGN4Bc4zb>&zHu3(FY~H<O5wX16M*irTmDBQ(fdp{XtsP6%wlJ`!A7*2B5Z?Fl*FEX?vhU0#eHONv_6uJgBxr(#?>KoyY
ytv-
t|L|l{{jGC)>#}Q2iB+!REakc{{u_5svXDZavVN8ucNMyUw{!5{h8{ix7npS1A}t*3BMa^&xzP${GkAm^=0h9TF9wPQ7Xh@afI&CzRi@SfSx;(8Hx>1>^}ZF
HKFD_r75HxFp9B#P=bT^FE_(h@wU9;a(PJ&+4mj9&4C@wy3q|DbhrhO#}&J^{UGR@aNCA0?K~>rXT1saTLN><*Uc`ZRjRA|s0nBUB72;CN2PHbk}zr+D!+lG
O3GyTov>tjWZK!d7%Ur)&7tR{{U_cb7HNdHHBOXHIB0>yi}&4>1VM4f@rlP=Fr+MKLrtlH-MVTC6IGzpgmyLi*PI|#!%7J}${oSTo;v58kLU*BUn-
6(d@zUV;5l;~kup^E*KBdQwgx))dw`r#RhE79FAynceI_VcA_p@S;c;-ABPES8=E|9D)NBZ?2$-
9OeMWScrKU#$Z!YG^OX=fgVsCJA(+8K*Pd*DWD1=i9o6#W|0)aIHY_d1PlBTX?b>A`txRBMA8Ir+(fzk)*O-J#>pKX-
?ga{a<bvp=uONvLfs48R=(|&84W=wcK!6{u78xHStg8};3<~ySfx1}x&R>4lV173DgX9Kx(4E7#~y_H>i%ydaeCkj-
b7$?(D7nT)&7|!$Elcxqy)F`>Y%9!i3q?{M6&pa;`s?LErs#d~GEHmBuoo-
4gSTP`jl}60=$dx*_wQ}nFJeXKA{PPHkWP~=E>^2l{ZBwoKCj6!`PQcO_g?tv(Xpx$~VH~Eut6!=Yl@UcQDa;vb5Xz=9%8zzee6ODHhRL$4on~eODtom&iMe
{XoI|S3o>K`nw3}5Y8L>xpOnDEvjf1nprW934s83ABoeX)LxKrrrlIU(Aisn_{%i`wlJJ5%ZaRTVgqR!$kHiDAw5i^XqUvBSG3W&%*%XKYM*?&O`r94uwE)<
&IX81JW@jb5mq4;TwLl*Ur+t{}OURhSSeHv#v7oY*JDS7f49F64j$lM(*19L0BR~9>6)WGRZ^+vI7HRcpEH8zAbV7ZvB#-$&gzSY_WLor4`>C*{tO-
)k%I)_l*U?CKXa?R(910;_G^`wS3UJ3~mnCTKqppM^VBU=UnKS7<KFyKho&~E^~uNA{<z4CFD2kH}A9_!DRJQD=wsJe@1!ADDI2nYsJ)i;ytxp8K>TK|JFD?
7^F)T~DRd7kcT$qcv`kJKqA9AaTEGx6Xk^9Ll+2u0raGc!r2LoYUf$2ocolg-L+L-
^1t7s#dkizJTKu7g7*<uqgwTD{f*hkN%B*(KQ;Mr~4uAf6~(&JwmVLrJwl*4(~$Q@tzK;nY6F$f#3=Kxyfe>u0G<BhO##tEDkNq;*?MP9^kPRj2g|058!~J6
V%)ymYSN`vLlod%{#p;?f};*mYi6VUK2L{`F9ekFLK{Ua5l|%7kR$CJ!TVp^}><mAL~BSCA9?Im~H#aOaAAxN-jnq6etNLJoQjm5OU3_hY#)ikScvT?8#`^(
=+mDiNx*=v<DPPe;u)oXe``&#=de8#jna`8$TZ(#Ht{mN{f}E`R+xli8H8==p@I@x+4LK?_7Ak=T!fKpA0ymzoaYU@!Rgio?ydhTER|zLam=m;D>`1E#@(hx
vra@Q#dprNzy@J14m`RhWsRKqqk|8stRQl@fRMz!EpvJUEk8;gc_Yr5eONDh|yM>|VCLr~~QuUp^k`IVP6Zws3r1YM+rzh2O8No1L)0j9KrqES4SUhVC@*|D
Y0h`0HlW#Pd@d(G&rdV^Zy*llcPnv+LFdKKv!U?+fxGY=gJV$v&8bh)9hretGB~2`F6#<&ztao(g?#$ZU38!x?BcEt`4===8+AVlGmZRk$wKh~6gELSTKyX;
C)gQxApazO~)I(WG;;>(>XrXm~~uWWE$Pl^7*ar7ZO1;kvzD5~06%<()KfwT4_5|9Lf<y^}5P+`PjdSZT6j4@99J=XttI$q*Va#4X$pF#dV%gXjY^0m1g14=
+Zyjw{YJ05knCVe}$1czgBeDaf&5fWU6;rPZVIJYneBV!Y-
{QBE^K%qMXZ4<hnciDv+j;+*zLdn^4y&DZkJgXT$F$ly8)Z>nS+I3ml@FdANEbGwg!<5g_6_3p-?a6$5$%v$9h9Uql|Vzp6-TZOc4pC#;Q=nd~z5-
U>8j(!yb7flY9Vp3jiB3(;pj3jDuVSGevdUiB5Non*9YG`<<_`aqquach#ov#na2r$#Eb`gF;T8mVr_FZELJoLqEXv$y}|CiQ2T#nM#K(ZFPu-i)*_@-
RDQ12R@fU8e2-kuGa+{W)LP#2*;TiANNlfnKp&%QQjv5i)1VR88TC(!pHTHr4&@w?k{%-
zG}Rw74ox%X`?UJuhalDF(?C;9)yJ4`t(B51&Dld!bt=6RO++~AUH67~ocF`57G4;bbNcLLkVVe}+)*w$G7nAGZm1Vq>_#4hfj|0)Q$+S>t}xNmrVGCAatdd
RfR9*y&OU6)j8Fy`BFZKb{|3SXl5ZhwU@eUnsUC{sdDpA}~A`+b@lI4!5lo*ts>;LI?S!@zpm^8>*egwRT!bN2n4qNUUt=}_|`W!L_cr%#sa++mB((dZ|l*k
_8NW)-
_T9BPdJ+XM4gr4^q{+wJBCHh>%PlBx}mVeb%tMk40mYv8Qz7sQqkK9+cz(;i!@vjy>3$T8?dQHoaHmYKEu3!K9{sId<=6JD&5!52a%KtGzB91QwW<y5IS8Hs
~IRredDrYCcc;UhKo)n>)ri45KEJklepn?*H>Tjb*zdOZsBAVK+Mm9a<<Yqzu9iV020<^_GcaFmFy|JehC%b7Rersx!A@}E6kjnbs7kDx_fMKAT}g-
E<?=fDtApSyh3o_=`)wT}<T#YWtbv8V-L{hU$1s4G!n0A;bg=;?v(aX6DZ^4J#Rx*zhVVy<a2WUaH#P<|~JKB3`;-
G1%}S*s7ktAeYfaem!i##22f2;@JXs|I3n@3E(m7Ur8J1!7{EcJ7u3jcPD4WXuvidvS<7>sEL)SljudcWS4VdbIpGojK6R1G)AB#lEJLDJ-
?Sax2R~0jik>I7&u}Ebo{j1x#BDJq<*}uJZ&0I}JdT>GT)~TnjD#dnm=M&gBe}yQ!Z>z#iO$@Ou#2*YgmDweF&q&JAHuxCq?sz2RxXr=;~<{#Y>IFQnH5;rD
7^WkYrqme05}A?ftTaLoS~vX(Iu)OVhaN$}T-nrMU=7*n@+_`=KwvR=?DV~tu<NyVr*Wd|2i--
^OMzAld1T!U2<eGiIHQ+<#h<QZbp`w#l;<AVomas5w&;f#OZ@oW*wjGp(3&a?lc;<(N^7WSR0WW?uT+&v`Haa`(%qop{!aHV8)$EWczJk65&r!52AX<3sRl;
PSk)tQ|^I_H7ZqR29oR-}V?0DLI3JWsLo!{!b3b5C{d^@jmJr&r-l#V_y;*E|ay78>L$n8N=VhpZ?J-|L^1wx)y{DP6o6*?&*c)YHyXsapqX7LvFWS2ms#-
t>Zr0cS9xG>C}8#?iwo;RX<n=(w1T|EQ{;yn-iZa+H{FRRFr3)TsRFO6V6-RxAe!88X2HFEe}#hfKtM+Hj>!PRRgY=ceO@V!*ft7S3f;%*zWjf0z$!e-
jIwH0}+^D1u5#%!(gnywHw<Mbze8Ew1e8g0|?^+<+^-%y0w5&TXXTOi(yz-
V%ki>L?e82=UMU5%C_UPnD7g%7bLq^@oictMT!Wtbm}oFuyOV!ZZVgu5HYrR9aGsgaix4fmpIN8;*gkb8+UtowUH5|7d*NN`fTm5caT@C4NVn(8@t@wF&-
y&8HcNtOa8nKj4#Qc-e?+GrLF}jp{bPCq{yM=G#)LxW-
e`Ilp@3A|G5qz}m(Q+1;T7h&fnEH!BvcR!W&CIn<D8f{ZBcaUH(6d})o=+W=?v0F}*dYf5{JfBWwl8ha^~Nq|k}-
M$ntk~Reg_mX5BuPFs^YXJ{WI%3zEFqMBe&BN?*wsLw-XkkmT{Q<Z#@*L+#2+6;cnO1&34a9x<ADF@Y5nyBqD1&?JF3$jJ(l+w)5zhiG1k(6sEl<+v9wXNz{
t~S-upc~Vn5st}j|lbw`hCbYft@Oe6mh-
f+VtbT=kxbwa|p4qr7zB5Q>KC+exH*k`48?~iZ18kLG=Nt{~Lf^Rl%R>Jy;a(ykLYtpY;b)NdKzrR<kfMAMle<x@_w7nct|A4qu#KXEBW}Z1XUdb-SKQRhm6
kr5E%%BGU#DT#*WvwE__Q0w`{vUxjc}a<Z7<!+wPJ-
)I&wZct6@jBJ;wngdTGhk%>A3|(3~af%HDYNujYWlTDF9njdtfqGrfHwZQb9B^sJnyc6g4%ug9S#HAyb(X1MvdOj5a^^Nj=^UvodtC-
9(Uft(t{bbhU9?RQt+d)OVrvCFG?am;JE5h;DD%;8|2W{#N9wN3F3Jtb)lj_>Adg4*#O3u`c5b^aM73IbPP`iwXor=4IE*ThZ|%e44HHz+m9<ysx~R*<B4Nd
D6$GXVg9<(&vnu;duho*#74oZ_f_FwSpX9Hh#`ZVtn|x602fFmXU|M_!HiW6%Zp`S^S=34`$)4jH9VRT@7j`fXist4y*9<mz-
u$P5|4b5A?~H6Ey%}^ZyD`_bP{|Sh$)K64e0&!5@xSihCr}>9xt_c9GHhWBc|YPE4M-
1923yU_<8gc2IY_B_HExSNRs$PF)>!CgfM4h3MG9tb^JKRRwM~P`u&k9W_DICbKC^*RNspw}f17*N2M1RH2mjr<Z>xtnrVU-
)Yt#u7)xg$0*Asb3LQ&oo>jGloOw-1g|F+%Q6QuaX-{X2ZGz*|0oxTMh(7kKhY2fK3_jt#ct!>57Gy-
s72y0}y8YNgH2LQpxx~5GfNobp$t2|Sy%cAuLB3NJr>{Orn<>1acMhnfk;Wq4-XT|%}&QR3V-G#**xv|H2QH@dq1=^n{r6nb*%K2H4^Q6V#tsN-
d<oVmdydhtU=eMIrSfpztYT>Sq@oD<xh4jVfP@KsD-Oq;9@a;(I@zpxC)jnBzxg6U(SG!h{joh~;@<)~|fo}s&J8kFN-my|Sfn^aChOEM~0H%|kBo3)3;n)M
c))@4uEAyr&O7()n<<;aO`IQoyXOr*WX6J8FrX0KwE|0nO!}S*S@ki>F-8$(?#kKKd>h~XgZfh=6q50L?EpmQPYLtDZdW^XfN>b8(i0?7;LXw-
(keEB1qeL0+AN-OtDG6XFIMUx!&tSWRoQF8xab?d9Vl#Bv4SUP6HOySY%*vfEeazOJcR`;95jBm*A&2jU6rjHOx(WvlO;`m}={hK?2QmoKKiSpoPWh`e$wJn
K`}`GK2E3!8KhYgz-YH#@dO<(da%-5JIkp&AbAwZ+%?U`lRo8-E1~pJ<zNqjJ>n=^aSs2HvKx{b-
0EpZxdS(&tb_J~1fd$zS0`bo1qO|YUk9jBm|5NJl8DCI5fHhRsEhf?AJp@=w%e$c$LL!aU!{sSQAp*G$vN&0Cg;M($0I($Y1YE>kuUt0b-
lChnbB8ljL!(}KEaru;0k@^ZlSQmR5gTc&;2VMrP#<wCdio8rM5l^@PSkhXC0m$*dE#bsH))mMoS^E@YDvx~kvZK&8i$>bVd(3-
M%!c>aXjED8iIH_x#W@b#CiMc@%)pZNgNOU>h?p{)SQ;=m=vJt)?{Vup&bNbp#`&l=S|3}`(@r5tivG5+u1vr_cx43yTYxptmFbF2G2<t75`ob;F12bOahrx
5<u^~i2R?`L%p~>dZ2F-
8dMRoaP=LyL`|r|OT)DgqlICRT`n2X#1EJ{re84DRSz|#sQFQH$qUsxtne*y;2=s4o%v<XAqK$aZ#NgARevLk9IE_A6canVP0|z&Zh0Qg)N7Y0=a!sNyX;k8
nh^!?3c4(%uNH&-
`4cGPY{iJBoIN@tnpWv^FN4i66jl;6#o|vwE8tAYc4X2uWgiY5Lde3glaB2k2LrW3HmIgGLVvZN^ksxJQI5%#57$kh1o%i7CuuukohYIjmk{<W4Aq4x@NB0z
>P<nnp=D4IWx}lNhO94M2`K>J>Nmklv?bL*7_BFRgufxez_Ukq&+&&*X@z8q|1<85jT$zi%LqRV6{=pAOz7VrL$%w0RR}-
|bl(u~59(5*NN;=UF_xIOVk|{WAu8?UD&E#J)uV8{WZ^B<>?Ff9z35ARn;8Uk?-0aa6M+=z{&0xhc+2|=1JDv@O4x_vw&OUdZlntS!cCWqT;N$5G@dC-lZ%t
Ds{3vc8zO&14JqH%%4DumV0375fC2m?h_`;Kd2A$IWTprhRP9n{o~jYEhu8Qbdx8Onat?Ah7YN>)cTte(kQ4(6Ixd6hI6XZbmkvVF1bO#DluS~+y{sRL639|
@pFO*+RnGRfgd7KP5_8o3&*zaZ?>x?QUAgh2@vF-
8F!>9Nu^t5PHrmoos}Le_bZ#hIfh5b^*Y~)+zIm~5Um*lr1DcHHl)QK|O`3rct}mGq&Vzh;MF&$q+Uf#Ff=>ygA)jI-W;zxY!6PTQS5GN^+T^3_6(3awbq`-
zyc4zG5;yN^fVej_Nvff}7J&JwzR46K?t@=_7<JZ{8-
~Darm~ATpS+Ch^nhVUI<KkGIq8Cd)~<Z1+1QqY?SkF+q|^b#XISm4AXl9uh+}$O?2m?q#PiX3`sA%YGx4udvKib7iA8)Pt<_ot>OMi8Srxp^>^YY6Dx5&xWD
BcqbL_P)Xb`SJ$6Q7V4C^BUzlz;IRv-
0I5?SEn4k>EswYYy>7zNLsZ6YAGeXp9|dfvIz5B3E}ds)lsO?84JFDPUK+(jtzM>0je?u*a4aEwMDYV1FIl=QlsxxO}pImKow7=mKXgs*ec;-
#U8?u(l0UPN{nw1kT!soTAXvGlD)l6X!jgKBE`)zEwYu$$>=OH1ejL)F<KIuejh2W@L%2u~Y5+ww^*?2G;mWQQVKzaf>#(v86>XH9#|Z-
Jb<F8(^=N~rb3ieb)$`N($A4)kh|XVIiUPgx@x(+O7G-
qO&cWm(g*^P0al=fw{LioFhq{Iw6RC7U!fjr>_#km0e|{0agyZyFYMgI85oK~(uq7mW%sE1J1H`fVy^pGB{gYOw~b>&7lK(0E-
M>=1W$30ln;5Uc?!aYh~k3lObuG!G49e7g)NXW6?_*HbjN%ELM*<*d>{ZE^KNI_T-hP9Y4^ZO-
(gtK9(_v+*r$Ty`@Dl%O1MoF7L%;zj)(46E$7VN&a|D4jsdJ}XIVt;K}(aN3x%)8yA>%NJzYY2L;6<p6p4y7W2DTJhG<M>#@by;rg+2@{Y6x$agbgWh1lU93
tIMLw(Q{vX0@+A`=}<O;mo)2O9Wr`Olu1!LF&1JMAy9rGhf`O8qM2c*peq`y;_a^3kKam|TZvcg}(EyB;EwMAiW%e^G9vs>%I!iu=dV){)Xa1#0sh@oY5;|+
-9osO-)H=0XDIuj?=(!Yrw>;AM3@29{@MscD06RS!E`X4Fc)y~ttMq^ftRsMNc_IPp(Viar0jV3>sawYFjlj@)~!7RzVmqh0mzfU&!sL@mc-x)l-
W%iErMp$IryC<vyd8O}f`AwVQPaSW_W!-_i&_rgC5Oa49CMey8m4X16NWD8XVr_XJ@5S-lXPJp^qvU{g+oVT*y2dJ99l1SE&xB~q%_-
+}b(3cE)LgkvK6ZtRVy2>Zj*M`0WN9X%0b9f!ErZf{LhhV_#n$AQy6{PD@J-
HwWLW7zyxay`_fx*W&?{HhSm)wkRxc+|k502qytcf2Cvw!9W0;^qvIM(D=;G0>4%=sbkI#ZCn%3t{q;&ySIm9y~X>!lHAtndSPA{x!{avV*Ymgcbg9V9X#xH
&By!-2AU#{n2?TGMIy0e!Pyjv-7Qr8>6-OREC(H?MDxjMlu<Xyt{mzfTwUO6-DQhpiBm?t-Ba4`1!p|YYz4h+$$h_7DI`fWSB$%>w#C-
3h#x9E5`n3B*2g}B5XCv9B(PyNfN{$+cVsR{m*A4SUp;?L~KFuR0+5qA)H;AHvh${(!igMAYzhQ8>1nlt>i9NJHQZCp_A@~8W4g>PIeRP^kR6~OczrjsKx=@
U6r;Lm-GF3bsv#C*UJ@P>%|6Y5)Q^j-ii9NkV)4QLhKAE+NPu-deiIo+wYIm~qToi&gUR%tsi^J=p>xOO8vc8{O*8N2Pp#%tC&Ppj*(U4~}|ERLpd3JZL3no
7!q-
;S(*32s{&aSca4Z6@k={%UDaO1wAT67c|!P5}Np>4LVh_^Kb(twUVGbSq84NF2#93f78W6wQn9;s^`~<aU);tXnq=&E96|TYEAdbiv_U@wspk{R4TsEXsg_l
4`LVynBw38NBJZQvM&XLflPHDg!~q0sEkAB(|_;1v2MK9lB>rP*isghm0a|r{Tda9HWwSR8RGsGQf2-
{d4{Zgp?#CmdiS(sqDP#R~>}8Xhh6nId^V|cT{8y_4P?>S)!R|N~LloSQj=vC>yT_*@w|Z#jVV!KG<H0K+p>~yThHdK@Pfh(#Vy9i3CC_Q9B|Cs8+4Phkn$p
AkfWhnfh>0i;^P8-;-x7#iyH~MR`L|L-K9cUJkI;)=<8bBc0^{p@rH>7XVXx9<=;{;9yI8Cd?oxoBb6B$X|uop9NB-
^OnC`6ZlY<>({G;We&uPp@oRU*+I4dy>jhQ<CcwAVUL%TVKo@ljn_U>2Z>2B{vCqD#PP5exgP?~j!F$X3|KDbYzZ!cu@_3K#)-
mXnq?)=U}ftUB^XDT7G6<%XskLG;j4mX@%f9WVbZp};p)9Jj)`D!5dt2wm6+Di9~=yecz`4`iTTq851z1AL@Jc5W?weM-
5lAmR4UzFYa_CkoPnj%Az0HEEwGd8*88yQpCX*uSi|tqiFv)kBa(yQR_t4eu$<v8K*p>y)BcYQ(jW0Am#jCUQ}kV9%K3be&iLA}jvWVGlN&pWIvh{EIfi~ev
a#i<OB!A1DbT`<MTKnYqv5RJ`dD1oQ?Z~Z2yYlby?w9KGjrvNUKY&Q(ur_<3;aWGbc(<~X#LgOBaak7tn#z<Wi}oCh0(Ku<Uu5Odw;zOV|=<ka5W3_5cClX<
7OCrsQAzHnV94b&N-oG(b-9jD5}h?sMCS<@j%0vmBp1531D;0)+F9T%<PSuu2nNG$o|0(+BPz|$qTX;xyqz+05g5)Q)*+)@9J78ok-
<w?pD!|K363gx4|fpFmvhN;jP&&61#@RzIol@d>xJDi<a`z)D2K+-NHuf<Av;|OoqpgW6BHS-wXFEFOX*1!zk;M9$@gG{<G4Vf=Uz4Z>ld|qzl+>aJtk4ZsT
s_;D4|404ihMW`eh>y4UARlV9sxnsZi5`eQM)R6Z2`Ae>~!o3-{vjVPM9V$CW13ToT{64L^%%UC`--
SBRlk@XE<C6|W4a@X#~hK~=N8#$MK4Zrp!pMRm4IoG-
`Km}H5wpK%PzsMsp$IMy;G#;5k7z1g;v!~lLf?+_S4IKJNXBWZ3`8278$XJ6GWxj+qbp%655wD=snZSt;6al(5Bo}Lz{lq=w{!DNtc5rZ<Y)OvV%A7w?oAG@
mlmLu2N<3^}uB;&<+!x>_s@yoKrQt<d7vl9>*}*8XbN(VU;nXE-!%v))lv)Z|A7d&%UE%`=3QNc-
c>ZGrUyLH<0z;ML4Dp0)7K>c}w!?cA^1|*c`$=H+m|r`8x*2_+ODVos2q)q86Vn#q9H2umF{S9|0Po-;?c3q1@*~~iGkcEmb6gRIu-
n3<FQD<NM)6(ypg+Pn?L--sV0YBC0WRS^ayt($$a(0PIuW^Kg@iEXQ}BtsYpmxDv_PbvO{JpFG7A8rDSD2s`a1>e%%NoJHxP9lj#wu7$e}8bNqX;H=Xd2|Q!
;fFr**T(S{EvR;rYHE`#EOBu*n{9sA~j1_MEF0nk+<ZX#fVGLW<RrykF$W{6lf`*$OKp5(@8Lra}2gEqu1CV0omNpE>WU-Q&=6KO5Re_saghHtrn|-
|s(MB%0SB5)sv=(&_|paSa#>epP5vd|z^cmxu+<!X-
ca&b@z5<a1%=FeF@o5%(2OSodcRvDI(iXG6kA$@~yz!e5jy4q*rU<a1m!g(H{~Spl3PnVzvCmq}S93m&A-
N&D<U7Vn^B0o6GLIQfNNDFT}gTND&Y?|%c1Ao6w#0m$0pz9<9A9qEZ)`eT>0)xv$6jSD;9aCCIionW;w5ODW-
DAWAJ$?=R`SGU1@yw$=3IE0^@TwZjXwjVQzCLvZ?{O-*D;@)mwTf@B*w)0Qfq4rh5+uXJPYC-gOT3kb@+%mF`2=yYgWM23T`Q^Loh$P@CN7IZB)i#gkUny!3
j2YKt;}%w*_KOOyXe8Ri(cEzffJsM&r_W^h)a9E&ktzhPlZ5+lmYE;Fk&5kT;}8RpeH5v<kkpF<>kejC&|AiA``EOS?c4||&cEt}&!AmyZR`yL6ucbNTH$oR
)kOj(aA5&G0#+uxO^7*4F5yr>`Sm!7D3c4I+NTJh^237BUY+G(Abn(iF&Z#&AkhNb*ZcVTzx@VsrP}9M8W=+aESs(+N*RUSC*!IGY;cW<^Mg-BLI(cn-ri|H
r%Isnq)0tD0D++j7^SQ)y;Cye*3G&fb>vzyEHw^jOg^5u6KlY54jpBu({fBBKWHo5tUaa|UsET_C@N^H62RovV5t*@DrO>TGQ)&dauAUrvW*Vp9Sicq9Cn~<
>j@$<!Sy_GUmCAd{B_n#vS{!<Cc~%khU+r4_G^dRKFAXK>CJ>U-bPlvYGF6Wk;yx0W_y^-
_vC*ih~pnHkKd;2L)Vz^WSsajfCf0auddb6sHm$=yv`7l2x2HZDyF=|8eC2t#3dLR3X=Hwg1I?ZaOuSGn!=jv*akd`A_nb+s#5AzLA`MFb5)rNO)P_~<d*R1
Sy~K5o*x#3X=1gOYcw9;ShLo@8hYXU5Ec8%xdtZLheM?;CKKgmgQ#j9RTi!e92yN?GxA1P)~>IbQfQsTULk0Uho-
o}h4fW2@C2=jd;+8^8q`eCv8!v(b8U`q+b3-
!o98KbctTkRxw?RH8w$L~wWnWBD8j8hJWcwQZpo1)(&_baBe?9oL@{<VqY38US_cj5m3Ry0SlwS~<gp8Bk@L4yVr>l))0A#Fbm#1c(k^(9(1Fzy<Q^E_hHj<
?Rx5ZY7#{vZhcdM9@yTbGRv|regzc*vhyY*<nSPC%lTZJ-tj@$P|26PHwTNXo#<s>6+^s{T>kRD1sQY_JvTGz-dZbm?Q|Ac2L=NJY4<CSC{MM`Q?(PHz%nQp
cTWJNFOob+7zzH40y8(q-
WiMb7#S84)deJCgdn{q&*f<!saO?i;#pk=(TW?ZYD&*K&f@Y41jvI^^87*bB4bGZfz_4qyhLll@#E+!4eFz)^MKx@n?yaCrt^q;W=rlni-
G+Bfi3?vLLRdvC2T9OwduEyeNK#bUyK;=Rrgtd92y%Dk3If4$QR;I~J*K}!^<H-rN{v*tbMWMbfo=73jTPhq(XShnW=#y2nS@I+-
*aj~NX@giM|aMh!ez%EyAYOg_djgdFuf)>iW_q$7DVIuy`dKj0jkZyLO^x$;Yi7`@AU0z9%R<b4eH7^IP8=G>;KWyn;KGG-;76C-
+?O|)4u6`qhCI29Z#3XYKe&k$dGvZXAF<$?uY-BRE3J;+WCM)^@3=j{ZD%tfA{ua3?9VMsH<_9m{!dlT?y<UxGZS%>|m|$aJPfYpCpw&HrbJQXTo^pRz;K*V
C##Q+^Sb|z(40S?Wb1OS}E<;2U|9h2BF_hv@j}^IP;umVg0=bJtg1PQe&z5=WnSouH`qzB2K?)zU<yTDZ7HBsDuT{!+eQ$mF_R$W@lp@<uxDd^hl?{3H=H|m
VI6Lv0wPwI_%@3JLSr3(6tZDCQWe<{tmSe4kP(6U4QdAn=cptvMc4%nV_(jBWO6WK!PV20IYNtHhd5=;E8t8<k>_Dd_Gx0;;-
vGRjd!ZPR1Q=S)pe3ZNX+?L|a(rpkE8lxlHeth}WwkKc^UPR_<~nr9}U#p1W*Ewb?)+!<E@8SQkREk}B!}VgKj#?78AR4~dj?vF|0UI-
g3fsI+zexW4RqSzk#DjCKY=iUQn$ek4`V&!K<d1fvV&)xu6g5w1~aw$aMg@99EZxyY#Bgnr~~+e7%gi*dQR^2thf&aNp(G!<TlXkK>Vj}SuGke1E@Qdfn!u~
#zJxG3rD8@}9>Q-(+xF)oC&Ed3npLl9Uynf2*3QYmeUaz#N-
9OxE;|L20qNTt803n^9t7F1J?K$Rm%vFDO8lr5nu%s&R6DqHTMH3M6bx)Qe>Gn&9|S{3c4Wsx>>ud0w_oiX<)L#k&uuQaN*`zz~cC`3|S3UwJVPHn(>jAK*>
U}7^#!vHXRG0)6ikW7K$;*Ga2T5!Yr0;oTwU&a=up=_fVdU1*7`Im-
N<pzT)=Ds?I_;vHrOf5#O8o4z7A%!{zwT#d&<sL6JJl420fi<w6@`|};N=UE_=W6G1aytE_f`2I@HkOp0#Tgw82AzQ$&O(Ic_*r#kLj<zh`wG6d{vQMMg|J2
0WaH0t9?>~F)(~wfE}wYpMR^oFd+-
%8^?esN16YMy$R4dWipN{omJ>#oN|l6a&9u2(0xkflrUX)3Wx+Ml!{$s+e7kbY?5P2n7;<|npuu4#Su+P>Mq29YmzLpKligyqoguL%_-S#XS--
B$)+1KVJ~f7jv~XFYJPFMYpSSpL7d67VOS$W6sg=%#`PaHMzHP-3*VrOZne|BtSrS2W<<Hl^Kyeh@!m}TL=Ky$q`MK-iiC>i78Q<EQnmn@?f)ysW$iL4$GZX
uH80WF;bBX!p%}~<_$x#s{weQc8b)O$a0i63(D{NS($H+a~00j{jKQ+))btFd89n`6rD*01;wIS_pO)+w>e!wJ|+R*jE0My^<PA~2awARs<XwcAv3-
7F^EZMx>eQjdr7zB9Qcfo_QzXL7Ee-W?Ogu+4S=69k!!x!D+2_K=oy5m|2Wd4N}E7~-^v>nRLchMn&7yS!u=R@KE$b&hl{l#{x%H0EU3-vSh-
THZFT9UUii4**lCECbntVIFUe))q0F0(p0CqoGOK6ur=miCpVXnHa2yeduRuv_9QTXWM>)NuKjq497UbbKZ$oXp6kG(hYgkWY6lj-jQ-VW77P4@jYR3AZe^l
03Vx9#=5|3wu3UaSn9rZ41%}{Xl`&!zYcvb|8Ezgx9qscVw^<$k?Xv_aClctDdj4wxYVbDP@t$G0KP#()bKx0J$sz$1>gb&prLHXpnFWz+h|C1w2+6E8SjnY
6w#(#28aKb$WmortpmOC_guoQ_BPIEY0M!>n}Xi>h|d;ju3e{u?QpOOu2G}h1&bc^AW34F5Rcy<k8jSL^Vg}z$2M1m^5S%aXrCI<B^-
gq$p#2;AxMRrJ}z^n&8P_ewO0~?>zpZZ=9@tpVgDu_GfRivRY&_xPzSWD|R2T1YSW`*I+nOXeA87vv`C}F8$EjU-p4@ntbJ-
S?wAHejhMDk<gD65EHWvAr;{rr!n8_F?&$4(Tv8#a)^hJCotuyfr&>4eA~Cp637IqIusuG#EyVqwG(nWJ-
#xP_7Pw{lu(cEHq+FSdIZ9xWtf*u!N{4;^gfUry%!<mpXb(uPKCc#2+`18fNCz^o$!!;X76^wnnA09XWYF9iJQ(Xt|_+q=9M|UI6#dk`NTlP*coEhD_wtarV
?QH=}Im-0~|mzdSMB>_;2$U8;R*fCVWqk@$Uf(F&aS8-)=vW9lwJbV}s%LwE9b;>z1=`qN(6Y)2hwic#=9?wo_e37A#dbkTl3;ix()5GK_;R4-
8mlg70z@k!~JECY6v(YAL$({20-qm)dVr>w`dW-O|ONmKLnh8i3|xAArrH<Tp4xveHM1e2dyq#kH~hzM1iAWVr*xhkq}}=WtB6g4Je2DbJtb-
i~ngTor0FK9u9a+-!jbCS>*(7j;-k=h9SHZo%eQ!D>L93KX$V8W6$cRB%K<RJ-JZEpNCVTHYDDcqGhy3Kc~Md%_jL7T=(@zn4RC<6hzz1BuDO76V;VSv!27A
}IfXAMA37>|m~#7c5C!3CAJ(!BfkB6kX+op93<kOQKqthgpCU{Qdf`NqU<&ZdkGeQ{Ob86@JyQTTx+fE1gz><gl~TD;b*bXx%rth)PG!o6^?8#7q41RLJ(O4
2Gg`!q9xP7)B;mGDENlDbnf#csTx@78d_@Wy-2^IEMQNd95XiU36jsuWaPl3Y34Zq{6^X8h-
^iN7OqcM&<;R%LmgOMf;qZ&eN3aD>d^fsCs~X){6XI>D9xgyTGnlC<cjS^~g3nb7k%UmC4Mr3kY+%_~+ejAt6V6W-
+?@81q&7Bj%qp$}Ua1Ql|N0yHC1KQD*Z53|8ko{OVD5eLeO==U%e){t0ffsUrjkH#G41m~+RDHD}^6xDOXgFB0KXF9tU20X!kS^tGw8AzYsI`92Ncqd3FCzX
393b;EwgAawceU}IX&4sRW2J+eHotfK?oAs#{&TlcCdmA${6U42P?ODe`l7R6%XYF$;R$}nK(%Pph`N)8zS>+~ikhGO4QB=GY?<Ir6V+<vvGkGFM2eB!-
PR&BB3CG$bhyiqLg!o(9K^vW>08qJrHp{>}(ssc9-ofw*#x{XOAJvgH~&z9QrW|$_(@R0VwE|TY{?9O`EJNb)xNPo~vR(9i@C^6=cNSp&0sI)-
+9>FHktYCq=#@r!;{wHXSH<Zd8C%+Ucv3&@z-jU{rQU!QaeSBix*nP(zc@&XXM<{Pz7alif0r1N+_IXOr${Lp?M?M8aAAqLr-
@WlYOKts*0mgocvmhAKwm6qE_GZHQph6O@r4b`QaW?p;UkF;^dT(L;JEvqxKm%+rjq<sGAvkn{k`mpSQ*f@cAkO+@W>@l16diwy=cLIJlyK)VDHKLwGu-
S^MVBmm2!%FIKnQ<flb0*fQd4rtcUZYa>Il6I=DSvKNWB6t8p>M&Y~s?uO}PgILMTU1HY%bK#;v%LSgjUs)ib+azKC#<Y}Y^rAt@8h54%pJRL1J4n@voL?_F
K`Vk53GbNDue6_aXwLpFE`>GHb^Obm**FaEX2>9JVriPDT;(3L=}%;<*eh*jd^7}<w9d^s3{rmvQI4k05$U}2Nm@EH(;_(Pvo(eE!vI(E2chEV6yjxoq%nkn
gMcgggMwXcb?QBi$?cu3{EgI~tw%LCUf)q?`?lT~JMx=az8TPX)nBpvKw*yVlBI5%V-SEDMTBEmeN-iH$S%#Vl!Gi3hjcRA0*cbEBTr;cEt61F1_(oG*bwS-
YC$=j_v_Pa1h&_=ANiS1_FzxTg&#;#m=PJ@aeI&mjtb!~4!wk^eIQMBnSd3yjXS^HX0ky&L0q+U{pHigwIL5aPbXxK2>**=uE*#N2QH@Z~`CrP=2HO+RZE8!
?+s@ty`>x8y%AU0T5tZfi@H@$*kbb-
d93~ps#F*yF7JYIfJ7tLZ;5hY)<(I0E^4M{0NdQh4^{b$6BFFUC6I4{R_yEqCAu7es>&fVK?d6X%7KN_CG)U&W;ZIsS&C21|cKJD4vr(u@PyAt#BE}~k4D|e
F2(_HW)gnMt-Ig*PV`aKTuevd*(3z2%UXCy5nf6M^qL7P&Egkdn-
!(mgSBO`z=Wnj9tDkaoZ2LrT%b=%?Vptz8?682@B%tOL_2}QqYsK_6v66Hc<feKnsPR+y6h&H15JWZy&r&;LySAA)7JfRQy7!p|7_@p&9bTc`!&$$_!cR-sx
I+CkXyq}CpMFyZ%SKcD@e-JZpO~07?7MERBkd$-b=$63tp&LEHF(oldALQSl4Mc<Xem}bW5Q?wN2AW7jjSuXZzy^%0S12hl6>%qJDQ}G->-
2DIwrPr)(lS>!s*4Xds;0<d1W_)i{yp$JR2gYCQ{ml<i%hxPlww1KHrIdJk8=jxBoL=7!1|!~dNZL^!-llcyyRO?)MRu6pgoSbH0?2Hl$T6bIl~d4Uk?L5y-
5UPAuf(865+Yt6fngUV-hhLq4WimfeT()>nrzu&<5uq5p~#OO)~Q1+xLr>pVl+}3_9LgTUNIpF81~cAQ~0dXu5^!Q#r*I4A_F^Ny?{SrT0A}g!9Gvac1yEHt
EaSc9(%Bde4w4i^|?SFGH`II_V3#@JMt6DiI+VYwMLxC_l-TsIuI_th{IWv-7X`6V`V27#eFQjWP;MiTeQ<#ltSY((?}-pxm+rzB4t?U5v02LnKdL$Ky-
%!DhrRGP<uqeBJv1X(3c$0&X(|_??yFBY<xP2y(S=(s)l1Rhc2W@b*>P3jxt+P)@?>W>bC)ZtEa<*0L$)$-}G3n{#<~wO7|(gzJX3fFA2-
h*S;MZ0QUS_@noEv=W0wEP$U=&y8>UV_QQ%@uus@L~*k<TU}OE%=7e<&GpYwp9Ex+ar)V9sMeRviBvJ$NvD~UnFTQF{$x?+x;4|sXZ<zp_tbJ7)1R%_K668b
ce<zz178tyOfNQw1NWei#cw3v4<$zr8sCb#lsd^pW}blGt;dJy6J(wCxmAEUOlz42g@Dxm3+10oZxHI0_-
se|0Ejk$!dYdpFyAqCL!yt5FPkh+xC^S}4)thaICIr-VZw%A%qpxdCPN@KC2oM9Mn^n@>QeH)uh~?cqLOl22Oo<VbD18a{Cu>e)sW-PCK1_4+fv{1icAC49u
aUJ{UGInSgH^{CM?FOSHRx>Rn*aM7g}3KKeGcN)1z=w-
e(tU)TBNzQuy5(r29yc>3#2_iWIslT~Xu?foSiec&7_5`LbLTPX2M<e7w;xw~Ik*ms0?hHrhu<My%7}R4wuwx%a~9dS*jm!Z}QC3YmK-k>m-
kl7r*SdF`%ik$LlHZ-Opd;NMg?V>`OTf|&<_E8kxalcvswkYIfIG&QmUAdSnxee+Boef~vX>%!k;%JlB6M)dgBco3VRgL$8)!9WFgg!YF1pl4%KU4*ylBaj-
LV}bA^P<Pos;ui4feLd)YHMUKzx3ltzzzr-FNh^)iojqWi!gxe$5nD>3YFxsG%6*!DwX9w<m0fz8QMUeuD&f&o^<8U;JYa~=A^1E-
*ahL(@_U$~8}$l{K(I)>(5lK%h!PM>y<G@!Xj<8}xaP_<ilj^zWgh?FZm{M$DAKE2M(8q*)=@JJBvdyf&Y09GZgjWJyswrCpHDzr)(NQ@;=<YOH)b6>us*4=
297a>sTMv~9iuroGTp2Wn+^=>__ih1>bI-);Zp~p68a(a4X#6dpu*q4uwug%_%J<YGK-
C{=0QHoIs5<|II7H>>cqOKPq1%$^=BnUjCQBf1=&Zx0Wn#$10KR5Tku1ep96xFA}FGqtJQaD9Pc5mhLSQoiz;J#mewZ<ClFg^$sH*U!3wbcwti8>=bNFw;H+
Si05?jl5d4XB+CXb#oL!}DG}hRAek0~5mjPoj(N6`}X+*`16pA?q_Vi-`rfYRyB3EX9C-)j5$nBt#e5C-
b+HZCEeZ^iTizvU=#z%{V`r50<*L(cIAe9pz+YoSTi6rsBq6)6kMRmQBCMC<2!WRRM%({PN+UX?Y$jwmkUPG^;YYODh|I&b1RrtS2zo`ZkUu`8KCl59F<U7!
UIzvxYDX~Y&-
~M{7dgT&+jT0low~d;KCd3U#T^iZvD8&!qo9T7Y%(0E8j%WFzfBKDO9CTplQozvS8gU><^nhoW?bn>F?ij!lQIXk=lSyz5L({HEsifJ6+Rgdk5w>W%mqY57c
qJ3$-
{LAY{KBRZV5B<MqXCoy*JvcNkbvQ~A4bk}{>rktyja9NTlbM@#<KvpR4LiQF%IpCfH;38iY$mT|6!d}ve}VIr*EFQ*9NJ|#`MqfU|*{|L`YL4HYA}sgT%}~X
b@=v<F`jBJi-jt;_wl#7@HU}c(dd?iB|XkkO~7;HN)rGP=HQV)cDr;pC?+3CsMJ*`14G53zCEMvMYDvs!G?3mhLOMQ~NQlCtR#~UiH!JDoz=}p!>>#)nsAJo
RwE_q{PzZYFObhwNHI8V$yfm8RLvfXs1+Z@^r$3X2gZQ+~;z_e0oT)`1lqe4d!x>4H>I-
5$}Rw7^3(~XgCHJ;+H!YD6!}|`*v%{!~pZ+iGt!o&hPD(i@HwzBkRh>`%2t=MvC?My6kgN?vP3WsaTrY6d{%@;!IJJT2NGuH&(f*S7ZcnqoJ+Waumzdt@&@K
0tV{V_Fj<m{4#v$$?5r?w6I9;xGknAlz?|Kv|Vj1O2?*hSFGQHP9+o^$?LAY&yy4q<Gc+;EJp(1+q=<?3J4dH+C!iMer386#1m3fJE@T&M_{zEL&B#55i`4L
PCUgIqt|?TiN6fRC#tJB&%86&GE|Z)BPn=7gsUy-s_+#sxyJ`69+lLYzgzeoPS!dGn7ja-WEo&1x*%W=7|d{6mR7MlW+0}UsIXQn({0YDx|lGrjo&6`tki?G
wATj8XI&=ke^DE&MqhQd3}E_^nW}-
vSc>Mtr7~ly2O@Kt1Ov)i##{}CW=v|Ad<;N%s(qkfH;vQg$p4zjKrC2WKheUjA+Gg6N36%z<ie(4=_T9tLTeIee)i3r3GCRjN7C#wCX7!H|M>4o)4<<DLqnL
h!G?o<d=b@=iyO`=fz;41U0R>d2p_fg!2M{xs1-Uv?wc8~#9}R$?mur%mfkz9`~|~S!2be}gsIo-dblUnr*m=)iq-wo0r$>)qR-9p-
g(!^Do;bC+0sYt#}>JN^Zq4?M2Csnh;Aes#H~HaMJ7yEA&4Dhk5Xpgi;hN|q4uwjma8hOByOB@5KGQtA9&M=LBDMe#H(@vd`m6+|C9avZ$RZy7PpWHm)Fr%)
YkSb)nrTKk#d|4@WXtpeynqsL?eJ7%`VU~OjBpDtpZ6N5)D4n^MRhp1Cb&3HxLIgrI&Av7I>pTf2N%=-T<RomZZ#o0fTxRX7h#i4Y;JU`5lHTqqsf15|-
;gA3W;}FlC_B!uq|Yj-
;9M0axH^pT^Q_iwVJzpISVl$YY(R<I|w=&Rm*%X=;o|3i#*TDI#PesVzI<L^S1R3$6}=sZ5MGek`)cXy9Z@>>i`1?Pk5sX;wEE?~3OT5lAE+pAK}>l^mU)9&
tj+mh^(p@ENa@3xJ&+FX_Lxm&Yves}#zIh$I7=qA*viWn<4S(z?56Pwb;S7)HCQQ-U_6nC<T?3jcW@YKJF+9QiLvqLL(kD&W)r2u}F7Iq?4zuM~F<ar@Ff=Y
`%>>{k-dy}f$Ssnu_0WoE>zi<vJ{ji<Wrnk%O7YiaFD-d1k~x<Q11&LvEi9D@!l;AvONygc)9nv-
E}?S~2mb`xpsGw)x)ootKx0IwZK806CXiCM9M{&3s5$p&E1)ay@bE|TIu3yGq;r8TQ(8;dyHIs=`~IX2*RoOh(%7mc#;A<dt^Sj5)!+y)`)uQh~GqRsgjyZ*
IyO>2}_1cF$RR-GtEK|Gf4(5ad$^idK9kGugy;%-
&qdA_vC#2+iJt6#a(@Afs$+eT53D`A1(;%R>ZUDR$Xijgp~`v<TPI1lxrfMZrILQjX1L*km~@F<l5q8!;rR?5E?xEGa|oxi8_@r8v%7Tyu2Qkc+M#&TRv3tH
|Jc8z6odrhl@nQsANNq{La8=Y!W+;#65-y&S`(68<8T#Gn6aAB3>7Z|W=DGkXFkJcOP7~3s_GNOUp7XKxC?LglXUfbWyIIenE-;1^~D>e{u3}U-
y;4V(^8u)?el#m|G_cugvx=0ktHe@*%h|{E2uDfq#H@;tOj`fOJvv${c$n7@DbXWqf3wPE!9w%inhotchl_arJ4xhWpwmj;410<Ni*)K<WT^5TM%w6peS#%|
#sqH|!(Iu4e#-aB1zDNhXU`B+gE_ofy)m~xf>5uo`B@piE7LSY|Gp=$G)3{l1MUVL(SBmLL%4D!|1&Od>9=?Fc5-
QDRrf&{Xaz%YnD#GDm&F8wqI4FqS<zGT*;DUA0S8Zu%#({N(RoB3nS;b0Sk4jSH&hL>&q9OHC+7&iTgeo~9*FHt;$2yA8=!6itl#fkbB9tu7dHn5KF2>J|c|
YwK=4TjuA92UK#pme}0Hddn9bY-FUxkDON-v)GOJ$^PkJ9_-TJ`7;#abuEUzZYvqm0S2SDqEa=n=Jj?09r~aMNqR>JH8&5qyW>-
cfD=Uqo_@lwc>(i}G5L3^~7C3wmW3xC8khGwR7hk00zuH#0Cp*Qk6mA(j&kK~Gxd8%UzdMgZxBr&YRQE}4p-
0HI&_QqM<15K5=?sLE|?+hy;1Ao8)X?VHr{!#iok=W|tM<{0>mx>fT6kxv2d4w3Z!00000dd*XQ+>QcH00H5&1%R*^a&Ji5vBYQl0ssI200dcD
"""

In [ ]:
%%writefile umsf_fw_lab.py
"""UMSF Firmware Laboratory runtime (notebook release 1.2.0).

Written to disk by the notebook's setup section and imported by every analysis cell with
``from umsf_fw_lab import *``. Importing it is idempotent and cheap after the first time:

* installs only missing Python libraries and checks for gcc/g++/make;
* verifies the embedded payload (``umsf_fw_payload.py``, an xz-compressed tar) and all original source hashes, and extracts
  the project once into ``umsf_firmware_runs/project`` (reused while it verifies);
* defines plotting conventions and output helpers;
* exposes lazily computed, cached native states: host tests, compiled C library, experiment CSVs,
  vector and corruption checks, and the verification ledger.

Because each shared state is built on first use, any analysis cell can run on its own, in any order, and
after a kernel restart. Native builds are incremental: unchanged binaries are reused. The original
firmware 1.0.0 sources are never modified.
"""
from __future__ import annotations

import base64, ctypes as ct, functools, hashlib, importlib.util, io, json, os, platform, re, shutil
import subprocess, sys, tarfile, time, zipfile
from pathlib import Path
from types import SimpleNamespace

NOTEBOOK_RELEASE = '1.2.0'
FIRMWARE_RELEASE = '1.0.0'
assert sys.version_info >= (3, 10), 'Python 3.10 or later is required.'

# ---------------------------------------------------------------- 1. dependencies
REQUIREMENTS = {'numpy': 'numpy>=1.24,<3', 'pandas': 'pandas>=2.0,<4', 'matplotlib': 'matplotlib>=3.7,<4'}
MISSING_INSTALLED = [req for mod, req in REQUIREMENTS.items() if importlib.util.find_spec(mod) is None]
if MISSING_INSTALLED:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *MISSING_INSTALLED])
    importlib.invalidate_caches()
for _tool in ('gcc', 'g++', 'make'):
    if shutil.which(_tool) is None:
        raise RuntimeError(f'{_tool} missing: use a Linux CPU runtime or install build-essential.')

# ---------------------------------------------------------------- 2. verified project
RUNS = Path.cwd() / 'umsf_firmware_runs'
RUN = RUNS / 'project'
_MARKER = RUN / '.payload_sha256'


def _verify_sources(root: Path) -> dict:
    expected = json.loads((root / 'provenance/source_sha256.json').read_text())
    for name, digest in expected.items():
        if hashlib.sha256((root / 'firmware' / name).read_bytes()).hexdigest() != digest:
            raise AssertionError(f'Original firmware file changed: {name}')
    return expected


def _prepare_project():
    from umsf_fw_payload import PAYLOAD, PAYLOAD_SHA256
    if _MARKER.exists() and _MARKER.read_text().strip() == PAYLOAD_SHA256:
        try:
            return PAYLOAD_SHA256, _verify_sources(RUN), 'reused'
        except (AssertionError, OSError):
            pass  # damaged or edited copy: extract again below
    blob = base64.b85decode(''.join(PAYLOAD.split()))
    assert hashlib.sha256(blob).hexdigest() == PAYLOAD_SHA256, 'Embedded payload checksum mismatch.'
    staging = RUNS / '.staging'
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:xz') as archive:
        members = archive.getmembers()
        for m in members:
            assert m.isfile() and (staging / m.name).resolve().is_relative_to(staging.resolve()), 'Unsafe archive entry.'
        archive.extractall(staging, members=members, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
    expected = _verify_sources(staging)
    shutil.rmtree(RUN, ignore_errors=True)
    staging.rename(RUN)
    _MARKER.write_text(PAYLOAD_SHA256)
    return PAYLOAD_SHA256, expected, 'extracted'


PAYLOAD_SHA256, EXPECTED_SOURCES, PROJECT_STATUS = _prepare_project()
FW = RUN / 'firmware'
OUT = RUN / 'results'
FIG = OUT / 'figures'
BUILD = RUN / 'build'  # notebook-owned binaries; the original tree keeps only its own test/host/build
for _d in (OUT, FIG, BUILD):
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- 3. libraries and conventions
import numpy as np, pandas as pd, matplotlib
try:
    _ip = get_ipython()  # noqa: F821 - defined inside IPython/Colab
except NameError:
    try:
        from IPython import get_ipython as _get_ipython
        _ip = _get_ipython()
    except ImportError:
        _ip = None
if _ip is not None and 'inline' not in matplotlib.get_backend().lower():
    _ip.run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, HTML, Markdown

plt.rcParams.update({'figure.figsize': (10, 4.6), 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.18, 'font.size': 11,
                     'axes.titleweight': 'bold', 'svg.fonttype': 'none', 'savefig.facecolor': 'white',
                     'svg.hashsalt': 'umsf-fw'})
COLORS = ['#096B72', '#E98032', '#5665A6', '#B84C65', '#6C9457', '#7E6A9E']
plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=COLORS)
EXPECTED_FIGURES = 30
JOBS = str(max(1, os.cpu_count() or 1))


def newfig(*args, **kwargs):
    return plt.subplots(*args, layout='constrained', **kwargs)


def figure_index() -> list:
    path = OUT / 'figure_index.json'
    return json.loads(path.read_text()) if path.exists() else []


def savefig(name, fig=None):
    """Save the current figure as PNG + SVG, register it in the persistent index and show it."""
    fig = fig or plt.gcf()
    fig.savefig(FIG / f'{name}.png', dpi=180, bbox_inches='tight')
    fig.savefig(FIG / f'{name}.svg', bbox_inches='tight', metadata={'Date': None})
    (OUT / 'figure_index.json').write_text(json.dumps(sorted(set(figure_index()) | {name}), indent=2) + '\n')
    plt.show()
    plt.close(fig)


def run_checked(cmd, cwd=None):
    r = subprocess.run([str(x) for x in cmd], cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode:
        print(r.stdout)
        r.check_returncode()
    return r.stdout


def _stale(target: Path, sources) -> bool:
    """True when ``target`` is missing or older than any source (make-style incremental build)."""
    return not target.exists() or any(Path(s).stat().st_mtime > target.stat().st_mtime for s in sources)


def _timed(label):
    """Cache a zero-argument state builder and report its first computation."""
    def wrap(fn):
        cached = functools.cache(fn)

        @functools.wraps(fn)
        def inner():
            first = cached.cache_info().currsize == 0
            started = time.perf_counter()
            value = cached()
            if first:
                print(f'[umsf_fw_lab] {label} ready in {time.perf_counter() - started:.1f} s')
            return value
        inner.cache_clear = cached.cache_clear
        return inner
    return wrap


INCS = [f'-I{FW / "lib" / lib / "src"}' for lib in ('umsf_proto', 'umsf_features', 'umsf_buffer', 'umsf_gwcore')]
PROTO_C = sorted((FW / 'lib/umsf_proto/src').glob('*.c'))
HEADERS = sorted(p for lib in (FW / 'lib').iterdir() for p in (lib / 'src').glob('*.h'))


@functools.cache
def versions() -> dict:
    v = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
         'matplotlib': matplotlib.__version__, 'platform': platform.platform(),
         'gcc': subprocess.check_output(['gcc', '--version'], text=True).splitlines()[0],
         'notebook_release': NOTEBOOK_RELEASE, 'firmware_release': FIRMWARE_RELEASE,
         'payload_sha256': PAYLOAD_SHA256}
    (OUT / 'versions.json').write_text(json.dumps(v, indent=2) + '\n')
    return v


# ---------------------------------------------------------------- 4. lazily built native states
def _start_shared_library():
    """Compile the original C protocol sources into a shared library (background process)."""
    shared = BUILD / 'libumsf.so'
    if not _stale(shared, PROTO_C + HEADERS):
        return shared, None
    cmd = ['gcc', '-std=c99', '-shared', '-fPIC', '-O2', *INCS, *map(str, PROTO_C), '-o', str(shared)]
    return shared, subprocess.Popen(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)


_SHARED = None


@_timed('Original host tests (make)')
def native_tests() -> SimpleNamespace:
    """Build (parallel, incremental) and run the original 43-case host test suite."""
    global _SHARED
    _SHARED = _SHARED or _start_shared_library()  # independent build overlaps with make
    build = run_checked(['make', '--no-print-directory', '-C', 'test/host', f'-j{JOBS}', 'build/umsf_tests'], FW)
    log = build + run_checked(['make', '--no-print-directory', '-C', 'test/host', 'test'], FW)
    (OUT / 'native_tests.log').write_text(log)
    match = re.search(r'(\d+) test cases, (\d+) passed, (\d+) failed; (\d+) checks, (\d+) failed checks', log)
    assert match and match.group(1) == match.group(2) and match.group(3) == '0' and match.group(5) == '0'
    counts = dict(zip(['cases', 'passed', 'failed', 'checks', 'failed_checks'], map(int, match.groups())))
    return SimpleNamespace(log=log, counts=counts, names=re.findall(r'\[PASS\] (\w+)', log))


@_timed('Compiled original C library (ctypes)')
def native_lib() -> SimpleNamespace:
    """ctypes binding to the original CRC and BLE fragmentation functions."""
    global _SHARED
    shared, proc = _SHARED or _start_shared_library()
    _SHARED = (shared, None)
    if proc is not None:
        output, _ = proc.communicate()
        if proc.returncode:
            print(output)
            raise subprocess.CalledProcessError(proc.returncode, proc.args, output)
    native = ct.CDLL(str(shared))
    bytep = ct.POINTER(ct.c_uint8)
    native.umsf_crc16.argtypes = [bytep, ct.c_size_t]; native.umsf_crc16.restype = ct.c_uint16
    native.umsf_frag_count.argtypes = [ct.c_size_t, ct.c_size_t]; native.umsf_frag_count.restype = ct.c_uint
    native.umsf_frag_build.argtypes = [bytep, ct.c_size_t, ct.c_uint16, ct.c_uint, ct.c_size_t, bytep, ct.c_size_t]
    native.umsf_frag_build.restype = ct.c_size_t
    arr = lambda b: (ct.c_uint8 * len(b)).from_buffer_copy(b)  # noqa: E731
    crc = lambda b: native.umsf_crc16(arr(b), len(b))  # noqa: E731
    assert crc(b'123456789') == 0x29B1
    return SimpleNamespace(native=native, arr=arr, crc=crc, ct=ct)


@_timed('Native experiment harness')
def experiments() -> SimpleNamespace:
    """Compile lab/native_experiments.cpp against the original libraries, run it, load its CSVs."""
    tests = native_tests()  # provides the original umsf_proto object files
    del tests
    objects = sorted((FW / 'test/host/build/lib/umsf_proto/src').glob('*.o'))
    sources = [RUN / 'lab/native_experiments.cpp', FW / 'lib/umsf_features/src/umsf_features.cpp']
    exe = BUILD / 'native_experiments'
    if _stale(exe, sources + objects + HEADERS):
        run_checked(['g++', '-std=c++17', '-O2', '-Wall', '-Wextra', '-Werror', *INCS, *sources, *objects, '-o', exe])
    run_checked([exe, OUT])
    st = pd.read_csv(OUT / 'station.csv')
    st['baseline_mv'] = st.baseline_mv.replace(65535, np.nan)
    return SimpleNamespace(phys=pd.read_csv(OUT / 'physio.csv'), cases=pd.read_csv(OUT / 'physio_cases.csv'),
                           st=st, inp=pd.read_csv(OUT / 'station_input.csv'), events=pd.read_csv(OUT / 'events.csv'),
                           button=pd.read_csv(OUT / 'button.csv'), link=pd.read_csv(OUT / 'link.csv'),
                           buf=pd.read_csv(OUT / 'buffer.csv'), edge=pd.read_csv(OUT / 'buffer_edge.csv'))


@_timed('Protocol vectors')
def vector_check() -> SimpleNamespace:
    """Independent Python verifier plus the native CRC on every supplied vector."""
    log = run_checked([sys.executable, FW / 'tools/verify_vectors.py', FW / 'test_vectors.json'])
    (OUT / 'vector_verification.log').write_text(log)
    vectors = json.loads((FW / 'test_vectors.json').read_text())
    if isinstance(vectors, dict):
        vectors = vectors['vectors']
    crc = native_lib().crc
    rows = []
    for v in vectors:
        b = bytes.fromhex(v['hex'])
        rows.append({'name': v['name'], 'bytes': len(b), 'crc_valid': crc(b[:-2]) == int.from_bytes(b[-2:], 'little')})
    vdf = pd.DataFrame(rows)
    assert len(vdf) == 19 and vdf.crc_valid.sum() == 18
    return SimpleNamespace(log=log, vectors=vectors, vdf=vdf, first=bytes.fromhex(vectors[0]['hex']))


@_timed('Single-bit corruption experiment')
def mutation_check() -> list:
    first, crc = vector_check().first, native_lib().crc
    assert crc(first[:-2]) == int.from_bytes(first[-2:], 'little')
    rejected = []
    for bit in range(len(first) * 8):
        altered = bytearray(first)
        altered[bit // 8] ^= 1 << (bit % 8)
        rejected.append(crc(bytes(altered[:-2])) != int.from_bytes(altered[-2:], 'little'))
    assert all(rejected)
    return rejected


@_timed('Verification checks')
def _verified_checks() -> dict:
    t, x, v, rejected = native_tests(), experiments(), vector_check(), mutation_check()
    st = x.st
    assert len(x.phys) == 1155 and len(st) == 260
    assert 'possible_impact_local' in set(x.events.name)
    assert not ((st.query('time_s < 180')['flags'].to_numpy() & 16) > 0).any()
    assert st.query('time_s < 180').baseline_mv.isna().all()
    assert ((st.query('time_s > 95')['flags'].to_numpy() & 128) > 0).all()
    assert _verify_sources(RUN) == EXPECTED_SOURCES  # sources still unchanged after all builds
    ledger = {'status': 'PASS', 'original_native_tests': t.counts, 'supplied_vectors': len(v.vdf),
              'expected_bad_crc_vectors': int((~v.vdf.crc_valid).sum()),
              'single_bit_mutations_rejected': sum(rejected), 'physio_sweep_rows': len(x.phys),
              'station_windows': len(st), 'native_events': len(x.events),
              'source_files_unchanged': len(EXPECTED_SOURCES), 'esp32_images_compiled': False,
              'hardware_tested': False, 'hosted_colab_session_tested': False, 'synthetic_experiments_only': True,
              'versions': {k: versions()[k] for k in ('python', 'numpy', 'pandas', 'matplotlib', 'platform', 'gcc')}}
    return ledger


def verification() -> dict:
    """Verification ledger: cached native checks plus the current figure count, written to disk."""
    checks = _verified_checks()
    ledger = {**{k: checks[k] for k in list(checks)[:8]}, 'figures': len(figure_index()),
              **{k: checks[k] for k in list(checks)[8:]}}
    (OUT / 'verification.json').write_text(json.dumps(ledger, indent=2) + '\n')
    return ledger


def status() -> None:
    """Print a one-screen summary of the runtime and outputs produced so far."""
    v = versions()
    print(f'UMSF Firmware Laboratory {NOTEBOOK_RELEASE} | firmware {FIRMWARE_RELEASE} | Python {v["python"]}')
    print(f'Payload {PAYLOAD_SHA256[:16]}… {PROJECT_STATUS}; original source files verified: {len(EXPECTED_SOURCES)}')
    print('Installed now:', ', '.join(MISSING_INSTALLED) or 'nothing (all libraries present)', '|', v['gcc'])
    print(f'Working project: {os.path.relpath(RUN)} | figures saved: {len(figure_index())}/{EXPECTED_FIGURES}')

In [ ]:
import importlib, sys
if 'umsf_fw_lab' in sys.modules:       # re-running setup: reload a possibly edited runtime, clear caches
    importlib.reload(sys.modules['umsf_fw_lab'])
from umsf_fw_lab import *
status()

## 1. Runtime and visual conventions

The setup reports actual versions and uses deterministic synthetic inputs. Python draws the plots; compiled original C/C++ code computes feature windows, detectors, buffer actions and link states. No random sensor data are passed off as observations.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
print(json.dumps(versions(), indent=2))

## 2. Architecture and evidence flow

Personnel and stationary nodes create distinct evidence streams. The gateway preserves source identity. MQTT PUBACK acknowledges broker receipt; only an application ACK following durable site commit authorizes release of held records. The diagram describes the intended hardware architecture, while this notebook exercises its portable software components.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
fig,ax=newfig(figsize=(12,5));ax.set(xlim=(0,12),ylim=(0,5));ax.axis('off')
boxes=[(.2,3,'Personnel node\nPPG · temperature · motion'),(.2,.6,'Stationary node\nIMU · pressure · gas · sound'),(4.3,1.8,'BLE gateway\nidentity · RAM · JSON'),(8.6,1.8,'Site agent\ndurable storage · ACK')]
for x,y,label in boxes:
    ax.add_patch(FancyBboxPatch((x,y),3,1.1,boxstyle='round,pad=.12',facecolor='#E6F2F1',edgecolor=COLORS[0]))
    ax.text(x+1.5,y+.55,label,ha='center',va='center',fontsize=11)
for a,b in [((3.3,3.4),(4.2,2.6)),((3.3,1.2),(4.2,2.1)),((7.4,2.5),(8.5,2.5)),((8.5,1.8),(7.4,1.8))]:
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=16,color=COLORS[0]))
ax.text(7.95,2.85,'MQTT',ha='center');ax.text(7.95,1.4,'App ACK',ha='center')
ax.text(6,.25,'Native laboratory: codec, features, rules, queues and freshness — no hardware claims',ha='center',fontsize=11)
ax.set_title('From local evidence to acknowledged storage');savefig('01_architecture')

## 3. Build and execute the original host tests

The original test suite compiles the portable libraries with warnings treated as errors. A failed command stops execution. These are software tests, not measurements of physical accuracy. The complete compiler/test log is preserved.

*Optimization:* the Makefile is driven with parallel jobs (`make -j`), the shared protocol library used later is compiled at the same time, and an up-to-date build is reused after a kernel restart.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
T = native_tests()
print(T.log[T.log.find('[PASS]'):])
groups=pd.Series([n.split('_')[0] for n in T.names]).value_counts().sort_values()
fig,ax=newfig(figsize=(9,5));groups.plot.barh(ax=ax,color=COLORS[0]);ax.set(xlabel='Passing original test cases',title='Native test coverage by test-name family');savefig('02_tests')

## 4. Compile the experiment harness

The added harness calls `PhysioWindow`, `StationDetector`, `SosButton`, `LinkMonitor` and `RecordBuffer` directly. It exports CSV traces from controlled inputs. The shared library exposes the original C CRC and BLE fragment functions to Python through `ctypes`; it is not a reimplementation.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
X, L = experiments(), native_lib()
print('CSV traces:', ', '.join(sorted(p.name for p in OUT.glob('*.csv'))))
print('Native CRC-16 of "123456789":', hex(L.crc(b'123456789')))
print('Compiled original core and generated experiment CSVs.')

## 5. Protocol vectors and integrity

Nineteen supplied vectors include an intentionally corrupted packet. “Expected rejection” is a passing test outcome. The independent Python verifier checks decoded values against the supplied expected fields; the original native CRC is also checked against every vector.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
V = vector_check(); vdf = V.vdf
print(V.log); display(vdf)
fig,ax=newfig(figsize=(11,7));ax.barh(vdf.name,vdf.bytes,color=[COLORS[0] if x else COLORS[3] for x in vdf.crc_valid])
ax.invert_yaxis();ax.set(xlabel='Serialized bytes',title='Supplied protocol vectors (red = intentional CRC failure)');savefig('03_vectors')

## 6. Where the bytes go

Full frames carry an 18-byte common header and a 2-byte CRC. COMPACT is a separate 20-byte layout. Header identity includes device, boot and sequence. CRC detects accidental corruption; it provides no authentication or confidentiality.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
first = vector_check().first
layout=pd.DataFrame({'type':['PHYSIO','STATION','EVENT','STATUS','ACK','CONFIG'],
 'header':[18]*6,'payload':[30,32,12,16,8,8],'CRC':[2]*6}).set_index('type')
fig,ax=newfig();layout.plot.barh(stacked=True,ax=ax,color=COLORS[:3]);ax.set(xlabel='Bytes',title='Full-frame byte budget');ax.legend(loc='lower right');savefig('04_frame_budget')
fig,ax=newfig(figsize=(12,2.7));data=np.array(list(first))[None,:]
im=ax.imshow(data,cmap='viridis',aspect='auto',vmin=0,vmax=255)
for i,x in enumerate(first): ax.text(i,0,f'{x:02x}',ha='center',va='center',fontsize=7,color='white' if x<140 else 'black')
ax.axvline(17.5,color='red');ax.axvline(len(first)-2.5,color='red');ax.set(yticks=[],xlabel='Zero-based wire byte offset',title='PHYSIO example: hexadecimal bytes; red lines delimit header and CRC');savefig('05_wire_bytes')

## 7. Corruption experiment

Flip each individual bit in one valid frame and recompute its CRC using the compiled library. This finite experiment checks single-bit corruption of this packet only; it does not establish cryptographic security. An attacker can recompute an unkeyed CRC.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
rejected = mutation_check()
fig,ax=newfig(figsize=(12,3));ax.imshow(np.array(rejected).reshape(-1,8).T,cmap='Greens',vmin=0,vmax=1,aspect='auto')
ax.set(xlabel='Byte offset',ylabel='Bit position',title=f'All {len(rejected)} single-bit mutations fail the CRC check');savefig('06_crc_mutations')

## 8. BLE fragmentation cost

For an ATT MTU M, the notification value budget is M−3 bytes. This implementation adds a 4-byte fragment header, leaving M−7 bytes per chunk. Native `umsf_frag_count` determines the counts. COMPACT uses its dedicated unfragmented path and is excluded here. The plots count notification value bytes, not radio airtime or total BLE overhead.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
native = native_lib().native
mtus=np.arange(23,248)
lengths={'ACK':28,'EVENT':32,'PHYSIO':50,'STATION':52,'Maximum reassembly':512}
frag=pd.DataFrame({name:[native.umsf_frag_count(n,int(m)-3) for m in mtus] for name,n in lengths.items()},index=mtus)
fig,ax=newfig();frag.plot(ax=ax,drawstyle='steps-post');ax.set(xlabel='ATT MTU (bytes)',ylabel='Notifications per message',title='Native fragmentation counts');savefig('07_fragment_counts')
fig,ax=newfig();
for name in ['PHYSIO','STATION']:
    n=lengths[name];ax.plot(mtus,n/(n+4*frag[name]),label=name)
ax.set(xlabel='ATT MTU (bytes)',ylabel='Frame bytes / notification value bytes',ylim=(.7,1.02),title='Fragment-header efficiency');ax.legend();savefig('08_fragment_efficiency')

## 9. Inspect actual native fragments

At the default MTU 23, a 50-byte PHYSIO message becomes four notifications. Fragment order is intentionally reversed, then restored using the encoded fragment index. This transparent Python inspection is separate from the original native reassembly tests, which also exercise out-of-order reception, limits and timeout behavior.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
L, first = native_lib(), vector_check().first
packets=[];att=20;count=L.native.umsf_frag_count(len(first),att)
for i in range(count):
    out=(L.ct.c_uint8*att)();n=L.native.umsf_frag_build(L.arr(first),len(first),7,i,att,out,att)
    packets.append(bytes(out[:n]))
shuffled=packets[::-1]
reconstructed=b''.join(p[4:] for p in sorted(shuffled,key=lambda p:p[2]))
assert reconstructed==first
fd=pd.DataFrame([{'arrival':i,'message_id':int.from_bytes(p[:2],'little'),'index':p[2],
 'count':p[3],'chunk_bytes':len(p)-4,'hex':p.hex()} for i,p in enumerate(shuffled)])
display(fd)
fig,ax=newfig();ax.bar(fd.arrival,fd.chunk_bytes,label='Frame chunk');ax.bar(fd.arrival,[4]*count,bottom=fd.chunk_bytes,label='Fragment header')
ax.set(xticks=fd.arrival,xlabel='Arrival order (reversed)',ylabel='Notification value bytes',title='Actual C-generated fragments at MTU 23');ax.legend();savefig('09_actual_fragments')

## 10. Personnel quality: two independent axes

A 10-second window expects 1,000 PPG samples and 250 IMU samples under default configuration. The harness sweeps received sample count and valid PPG fraction. Valid means contact present and neither channel saturated. Quality is a heuristic, not a calibrated probability of correctness. When no PPG samples exist and no missing-channel error is reported, the remaining complete channels can still produce quality 100: channel presence must be read alongside quality.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
phys = experiments().phys
q=phys.query('motion_mg == 0').pivot(index='valid_fraction',columns='completeness',values='quality')
fig,ax=newfig(figsize=(11,5));im=ax.imshow(q,origin='lower',aspect='auto',extent=(-.025,1.025,-.05,1.05),vmin=0,vmax=100,cmap='viridis')
ax.set(xlabel='Received / expected PPG samples',ylabel='Valid / received PPG samples',title='Native heuristic quality (%) — motion-free windows');fig.colorbar(im,ax=ax,label='Quality (%)');savefig('10_quality_surface')

## 11. Motion penalty and threshold

Motion is computed as RMS of acceleration-magnitude deviation from 1g: sqrt(mean((|a|−1000 mg)²)). The motion-artifact condition is strictly greater than 150 mg, not greater-or-equal. When set, the internal quality score is halved. The sweep uses a constant acceleration deviation; it is not a human movement dataset.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
phys = experiments().phys
fig,ax=newfig()
for motion,g in phys.query('valid_fraction == 1').groupby('motion_mg'):
    ax.plot(g.completeness,g.quality,label=f'{motion} mg',marker='.' if motion==200 else None)
ax.set(xlabel='PPG completeness',ylabel='Quality (%)',title='Native quality penalty across motion inputs');ax.legend(ncol=3);savefig('11_motion_quality')
fig,ax=newfig();m=phys.query('completeness == 1 and valid_fraction == 1')
ax.plot(m.motion_mg,m.motion_rms,'o-',label='Native motion RMS');ax.axvline(150,color=COLORS[3],ls='--',label='Artifact threshold (>150 mg)')
ax.set(xlabel='Injected magnitude deviation (mg)',ylabel='Reported RMS (mg)',title='Controlled acceleration input and output');ax.legend();savefig('12_motion_rms')

## 12. Artifact flags and missing values

Flags identify failure modes independently of numeric values. A null sentinel must become missing data, never zero or a normal reading. The fully missing case has quality 255 on the wire, meaning null. SpO₂ remains missing in every case. Synthetic HR is deliberately enabled only in its named case and carries its channel flag.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
cases = experiments().cases
flag_names=['Motion','No contact','Saturation','Sample gap','Temp error','IMU error']
bits=((cases.artifacts.to_numpy()[:,None] & (1<<np.arange(6)))>0).astype(int)
fig,ax=newfig(figsize=(10,5));ax.imshow(bits,cmap='Oranges',vmin=0,vmax=1,aspect='auto')
ax.set(xticks=range(6),xticklabels=flag_names,yticks=range(len(cases)),yticklabels=cases['case'],title='Artifact flags from native feature windows');savefig('13_artifact_flags')
shown=cases.copy();shown['quality']=shown.quality.replace(255,np.nan)
shown['hr_bpm']=shown.hr.replace(65535,np.nan)/10;shown['spo2_pct']=shown.spo2.replace(65535,np.nan)/10
display(shown[['case','quality','hr_bpm','spo2_pct','channels','artifacts']])
fig,ax=newfig();ax.bar(shown['case'],shown.quality,color=COLORS[0]);ax.tick_params(axis='x',rotation=25)
ax.set(ylabel='Quality (%)',ylim=(0,110),title='Missing quality is a gap, not a zero score');savefig('14_quality_cases')

## 13. Synthetic HR is a test pattern

The documented deterministic helper generates 70–80 bpm in a repeated 11-step pattern. This chart visualizes that formula for illustration; it is not an estimator or a recording. The native experiment above separately confirms the synthetic channel flag. Default HR and every SpO₂ field stay null.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
cases = experiments().cases
seq=np.arange(33);synthetic=(700+(seq%11)*10)/10
assert cases.loc[cases['case']=='synthetic_hr','hr'].iloc[0]==770
assert cases.loc[cases['case']!='synthetic_hr','hr'].eq(65535).all()
assert cases.spo2.eq(65535).all()
fig,ax=newfig();ax.step(seq,synthetic,where='mid',color=COLORS[3]);ax.set(xlabel='Synthetic sequence',ylabel='Test field (bpm)',title='Explicitly synthetic HR pattern — not physiological evidence');savefig('15_synthetic_hr')

## 14. A controlled stationary-node scenario

A 260-second experiment runs the original detector with 200 Hz acceleration. It injects a brief magnitude excursion, coincident sound/pressure changes, temporary tilt, PIR input, an SOS flag and an uncalibrated gas-voltage rise. Pressure is sampled at 20 Hz, gas at 10 Hz and microphone at 200 Hz in this laboratory. These are test stimulus rates, not a claim about real acquisition fidelity. Saved input traces are downsampled to 40 Hz for compact plotting; detector outputs are one-second windows.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
inp = experiments().inp
fig,axs=newfig(4,1,figsize=(12,9),sharex=True)
for ax,col,label in zip(axs,['accel_z_mg','angle_deg','pressure_pa','gas_mv'],['Z acceleration (mg)','Mount angle (degrees)','Pressure (Pa)','Gas channel (mV)']):
    ax.plot(inp.time_s,inp[col]);ax.set_ylabel(label)
axs[0].set_title('Injected synthetic inputs');axs[-1].set_xlabel('Time (s)');savefig('16_station_inputs')

## 15. Impact and pressure response

The default magnitude-excursion threshold is 150 cg = 1.5g relative to gravity magnitude. The impact event closes after 50 ms without another above-threshold sample. The pressure rule compares recent values within a one-second window. This scenario tests event logic; it does not identify the cause of the disturbance.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
st = experiments().st
fig,axs=newfig(2,1,sharex=True,figsize=(11,6))
axs[0].plot(st.time_s,st.peak_cg/100,'o-',label='Peak excursion');axs[0].axhline(1.5,ls='--',color=COLORS[3],label='Threshold')
axs[0].set(ylabel='Excursion (g)',title='Native stationary-window response');axs[0].legend()
axs[1].plot(st.time_s,st.pressure_delta_pa,'o-');axs[1].axhline(100,ls='--',color=COLORS[3])
axs[1].set(xlim=(27,35),xlabel='Time (s)',ylabel='Pressure change (Pa)');savefig('17_impact_pressure')

## 16. Tilt and attachment uncertainty

A baseline at (0,0,1g) makes tilt change interpretable relative to the initial mounting. A 12° rotation triggers the 5° default threshold. The tilt alarm clears below 80% of that threshold, but attachment uncertainty remains latched until recommissioning. Returning the numerical angle to zero does not establish that the mounting is verified.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
st = experiments().st
fig,axs=newfig(2,1,sharex=True,figsize=(11,6))
axs[0].plot(st.time_s,st.tilt_delta_ddeg/10,label='Tilt change');axs[0].axhline(5,ls='--',color=COLORS[3]);axs[0].set(ylabel='Degrees',title='Tilt return does not clear attachment uncertainty')
for mask,label in [(32,'Tilt alarm'),(128,'Attachment uncertain')]:axs[1].step(st.time_s,(st['flags'].to_numpy()&mask)>0,label=label,where='post')
axs[1].set(xlim=(50,110),xlabel='Time (s)',ylabel='Flag state');axs[1].legend();savefig('18_tilt_attachment')

## 17. Gas warmup, baseline and hysteresis

The MQ channel is a voltage, not ppm and not a gas identity. Before 180 seconds, the baseline is null and gas alarm logic is locked out. After warmup, a rise of at least 300 mV sets the alarm; a rise below 150 mV clears it. The baseline stops adapting while the alarm is active. This synthetic sequence makes all three phases visible.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
st = experiments().st
fig,axs=newfig(2,1,sharex=True,figsize=(12,7))
axs[0].plot(st.time_s,st.gas_mv,label='Window voltage');axs[0].plot(st.time_s,st.baseline_mv,label='Native baseline')
axs[0].fill_between(st.time_s,0,1700,where=st.time_s<=180,color='#CCCCCC',alpha=.3,label='Warmup interval')
axs[0].set(ylim=(800,1700),ylabel='mV',title='Uncalibrated gas channel: warmup and hysteresis');axs[0].legend(ncol=3)
axs[1].step(st.time_s,(st['flags'].to_numpy()&16)>0,label='Gas alarm',where='post');axs[1].step(st.time_s,(st.sensor_flags.to_numpy()&4)>0,label='Gas warm',where='post')
axs[1].set(xlabel='Time (s)',ylabel='Flag state');axs[1].legend();savefig('19_gas_hysteresis')

## 18. Presence hold and manual SOS

PIR presence is held for 30 seconds after the falling edge. It indicates channel activity, not a count or identity of people. The SOS trace here is a direct test input to the detector; debouncing and latching are examined separately below. `StationDetector.set_sos` sets a state flag; the node application owns SOS event emission.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
X = experiments(); st, inp = X.st, X.inp
fig,axs=newfig(2,1,sharex=True,figsize=(11,6))
axs[0].plot(inp.time_s,inp.pir,label='Raw PIR input',alpha=.7);axs[0].step(st.time_s,(st['flags'].to_numpy()&1)>0,label='Held presence',where='post')
axs[0].set(ylabel='State',title='Native presence hold and SOS state');axs[0].legend()
axs[1].step(st.time_s,(st['flags'].to_numpy()&2)>0,where='post',color=COLORS[3]);axs[1].set(xlabel='Time (s)',ylabel='SOS state',xlim=(0,140));savefig('20_presence_sos')

## 19. Event chronology

The chart uses event timestamps returned by the original detector. A composite can be emitted later while retaining the initiating impact timestamp. The rule requires an impact and nearby sound or pressure evidence within two seconds; it does not establish what caused them. Events have different physical units, so raw values must not be compared on a single magnitude axis.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
events = experiments().events
display(events)
order=list(dict.fromkeys(events.name));idx={name:i for i,name in enumerate(order)}
fig,ax=newfig(figsize=(12,6));scatter=ax.scatter(events.time_s,events.name.map(idx),c=events.level,cmap='plasma',vmin=0,vmax=3,s=80,edgecolors='white')
ax.set(yticks=range(len(order)),yticklabels=order,xlabel='Event timestamp (s)',title='Native event chronology');fig.colorbar(scatter,ax=ax,ticks=range(4),label='Severity code (0–3)');savefig('21_events')

## 20. Stationary flags and quality

The bitfield view separates simultaneous conditions. Quality is an availability heuristic averaged over five channel contributions; gas warmup alone changes this scenario from 80% to 100%. It does not measure the probability of an incident, nor guarantee that a mounted sensor retains structural meaning.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
st = experiments().st
labels=['PIR presence','SOS','Impact','Loud sound','Gas alarm','Tilt alarm','Pressure transient','Attachment uncertain']
mat=((st['flags'].to_numpy()[None,:]&(1<<np.arange(8)[:,None]))>0)
fig,ax=newfig(figsize=(12,5));ax.imshow(mat,origin='upper',aspect='auto',cmap='Blues',extent=(.5,260.5,7.5,-.5),vmin=0,vmax=1)
ax.set(yticks=range(8),yticklabels=labels,xlabel='Window end (s)',title='Independent stationary state flags');savefig('22_station_flags')
fig,ax=newfig();ax.plot(st.time_s,st.quality);ax.axvline(180,ls='--',color=COLORS[3],label='Gas warmup ends')
ax.set(xlabel='Time (s)',ylabel='Quality (%)',ylim=(0,105),title='Channel availability is not incident confidence');ax.legend();savefig('23_station_quality')

## 21. Button debounce, latch and reset

The input bounces at 100–115 ms and becomes stable at 120 ms. A 30 ms stable press raises the latch at 150 ms. A second press begins at 1,000 ms; after debounce and a further three-second hold, reset occurs at 4,030 ms. The first press cannot reset the latch it just raised.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
button = experiments().button
assert button.loc[button.action==1,'time_ms'].tolist()==[150]
assert button.loc[button.action==2,'time_ms'].tolist()==[4030]
fig,axs=newfig(2,1,figsize=(12,6))
for ax in axs:
    ax.step(button.time_ms,button.raw,label='Raw pressed',alpha=.6,where='post')
    ax.step(button.time_ms,button.active,label='SOS latched',where='post');ax.set_ylabel('State')
axs[0].set(title='Native SOS button state machine',xlim=(0,5000));axs[0].legend()
axs[1].set(xlim=(80,180),xlabel='Time (ms)',title='Debounce detail: stable input + 30 ms');savefig('24_button')

## 22. Lost links mean missing evidence

The native monitor marks stationary data lost after age >2,000 ms, and personnel data after age >25,000 ms. This test samples supervision every 100 ms, so loss is observed at the next supervision tick. A new frame restores the state. Link loss says nothing by itself about casualties or structural collapse.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
link = experiments().link
fig,axs=newfig(2,1,sharex=True,figsize=(12,6))
for ax,col,label in zip(axs,['station_state','personnel_state'],['Stationary (2 s threshold)','Personnel (25 s threshold)']):
    ax.step(link.time_ms/1000,link[col],where='post');ax.set(yticks=[0,1,2],yticklabels=['Unknown','Up','Lost'],title=label)
axs[-1].set_xlabel('Time (s)');savefig('25_link_states')
fig,ax=newfig();ax.plot(link.time_ms/1000,link.station_age_ms/1000,label='Stationary age');ax.plot(link.time_ms/1000,link.personnel_age_ms/1000,label='Personnel age')
ax.axhline(2,ls=':',color=COLORS[0]);ax.axhline(25,ls=':',color=COLORS[1]);ax.set(xlabel='Time (s)',ylabel='Age of last received frame (s)',title='Freshness is separate from connectivity');ax.legend();savefig('26_link_age')

## 23. RAM retention and acknowledgements

A small eight-record buffer makes eviction easy to see. Sending does not free capacity. MQTT PUBACK is modeled as a no-op because release is an application-layer operation. Requeueing makes unacknowledged records eligible again. ACKs for the wrong source or boot do nothing. A matching cumulative ACK frees sequences at or below its watermark, including any not sent: the caller must only issue a safe watermark after durable commit and gap accounting.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
buf = experiments().buf; display(buf)
fig,ax=newfig(figsize=(12,5))
for col in ['occupancy','evicted','acked']:ax.step(buf.step,buf[col],where='post',label=col)
for s in buf.loc[buf.operation.str.contains('ACK'),'step']:
    ax.axvline(s,color='gray',alpha=.2)
ax.set(xlabel='Operation index (see table)',ylabel='Records',title='Only matching application ACKs release held records');ax.legend();savefig('27_buffer_occupancy')

## 24. Priority and retry order

The urgent sequence 3 is sent first, followed by retained telemetry in age order. Retry sends the same held records again: downstream processing needs idempotency by (source, boot, sequence). The trace distinguishes transmissions from unique retained records.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
buf = experiments().buf
sent=buf[buf.selected_seq>=0].copy();sent['attempt']=np.arange(len(sent))
fig,ax=newfig();ax.plot(sent.attempt,sent.selected_seq,'o-',color=COLORS[0]);ax.axvline(7.5,ls='--',color=COLORS[3],label='Retry batch')
ax.set(xlabel='Transmission attempt',ylabel='Selected sequence',title='Native priority selection followed by retransmission');ax.legend();savefig('28_send_order')
assert buf.loc[buf.operation=='MQTT_PUBACK_no_release','occupancy'].iloc[0]==8
assert buf.loc[buf.operation=='wrong_source_ACK','occupancy'].iloc[0]==8
assert buf.loc[buf.operation=='wrong_boot_ACK','occupancy'].iloc[0]==8
assert buf.iloc[-1].occupancy==0

## 25. An important priority boundary

Priority chooses the victim among existing records. It does not reject a lower-priority arrival. When all three slots contain urgent records, inserting telemetry evicts the oldest urgent record. The original firmware is preserved; this experiment documents its behavior instead of silently changing it. RAM contents also disappear on reset or power loss.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
edge = experiments().edge; display(edge)
assert edge.retained.tolist()==[0,1,1,1]
fig,ax=newfig();ax.bar(edge.seq,edge.retained,color=[COLORS[3] if p==2 else COLORS[0] for p in edge.priority])
ax.set(xticks=edge.seq,yticks=[0,1],yticklabels=['Evicted','Retained'],xlabel='Sequence (1–3 urgent; 4 telemetry)',title='Priority is victim selection, not guaranteed urgent retention');savefig('29_eviction_boundary')

## 26. Capacity planning is a model

The following is an analytic illustration N/r, not a native stress test or a promise of loss-free buffering. It assumes a constant arrival rate and no ACKs. Event bursts, retransmission scheduling and shared-source traffic complicate real operation. A 32-record personnel buffer at one record per 10 seconds spans 320 seconds of arrivals; a 48-record gateway shared by three such streams spans 160 seconds.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
rates=np.geomspace(.1,20,120)
fig,ax=newfig()
for n in [32,48,64]:ax.loglog(rates,n/rates,label=f'{n} records')
ax.scatter([.1,.3],[320,160],color=COLORS[3],s=70,zorder=5)
ax.set(xlabel='Aggregate arrival rate (records/s)',ylabel='Capacity / arrival rate (s)',title='Idealized buffer horizon without application ACKs');ax.legend();savefig('30_capacity_model')

## 27. Verification ledger and reproducible exports

This ledger records what this run actually exercised. The original 43-test suite contains more checks than the plotted scenarios. The added assertions focus on the notebook’s critical claims. The site-ACK parser is a limited field extractor, not a strict general-purpose JSON validator; passing its existing tests is not a security audit. Native verification does not validate the ESP32 builds, radio path or hardware. The ledger computes any check that has not run yet, so this cell is self-sufficient as well.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
ledger = verification()
print(json.dumps(ledger, indent=2))
if ledger['figures'] < EXPECTED_FIGURES:
    print(f"Note: {EXPECTED_FIGURES-ledger['figures']} figure(s) not produced yet - run the remaining sections, then this cell again.")

## 28. Download the results

The export contains CSV traces, 30 PNG/SVG figures, build/test logs, version information and verification results. It is written to `umsf_firmware_runs/latest/UMSF_Firmware_Lab_results.zip`; in Colab, download it from the Files pane. This cell is self-sufficient: it refreshes the verification ledger itself and reports figure sections not yet executed. The full Zenodo release additionally includes original source, clean and executed notebooks, documentation and provenance.

In [ ]:
from umsf_fw_lab import *  # shared runtime from Section 0; every cell is self-sufficient after setup
ledger = verification()   # cached native checks + current figure count
files=[p for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='SHA256SUMS.txt']
(OUT/'SHA256SUMS.txt').write_text(''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(OUT).as_posix()}\n' for p in files))
latest=RUNS/'latest'; latest.mkdir(exist_ok=True)
result_zip=latest/'UMSF_Firmware_Lab_results.zip'
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files+[OUT/'SHA256SUMS.txt']: z.write(p,p.relative_to(RUN).as_posix())
print(f"Figures: {ledger['figures']}/{EXPECTED_FIGURES} | original tests: {ledger['original_native_tests']['passed']}/{ledger['original_native_tests']['cases']} passed")
print(f'Results archive: {os.path.relpath(result_zip)} ({result_zip.stat().st_size:,} bytes)')
# Optional: in Colab, uncomment these two lines to open a browser download.
# from google.colab import files
# files.download(str(result_zip))

## Interpretation and next experiments

- Change one stimulus at a time in `lab/native_experiments.cpp` (in `umsf_firmware_runs/project/`), then re-run any experiment cell: the runtime rebuilds the harness only when a source is newer than its binary. Run Section 0 again to clear cached results. Compare event timing and flags before interpreting numeric magnitudes.
- Replace a synthetic trace with real sensor data only after specifying units, sampling gaps, calibration, consent and provenance. Never relabel these generated traces as measurements.
- Use `firmware/docs/TEST_PLAN.md` and the hardware commissioning checklist for future device validation. PlatformIO environments are preserved in `firmware/platformio.ini`; their build and flash commands belong to a separate hardware workflow.
- Preserve nulls and identity fields in downstream storage. Do not equate absent data with a safe state, quality with confidence, broker receipt with durable commit, or a local composite with causal attribution.

### Source map and reuse
`firmware/lib/umsf_proto` defines the byte protocol and fragmentation. `umsf_features` implements feature windows and local detectors. `umsf_buffer` implements volatile retention; `umsf_gwcore` implements freshness and ACK field extraction. The original tests and vectors remain in `firmware/test/host` and `firmware/test_vectors.json`. New experiments are in `lab/native_experiments.cpp`; the notebook runtime is `notebook/umsf_fw_lab.py`. The supplied MIT license and author metadata are preserved in the release. Cite `CITATION.cff`; no DOI has been invented. This notebook is an educational and reproducibility companion, not a certified monitoring system.